# r/AskScience ICAI pipeline — 10,000 pairs / 5,000 candidates

This notebook scales up the earlier AskScience run. It starts with 10,000 usable preference pairs, generates candidate principles, clusters them into 5,000 representatives, and tests those candidates in stages.

To keep the cost manageable, embeddings are saved and reused, LLM calls are checkpointed, and weaker principles are dropped before the later testing rounds.


In [ ]:
!pip -q install openai sentence-transformers scikit-learn pandas numpy scipy python-docx tqdm matplotlib


In [ ]:
import getpass, json, math, random, re, time
from pathlib import Path

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from docx import Document
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT
from docx.shared import Inches, Pt
from openai import AzureOpenAI
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = Path('/content/askscience part 1 preference pairs.csv')

# this folder keeps older checkpoints from getting mixed into this run.
OUTPUT_DIR = Path('/content/askscience_icai_outputs_10k_5k_cost_safe')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AZURE_ENDPOINT = 'https://tpicc-mj8qs589-eastus2.cognitiveservices.azure.com/'
AZURE_API_VERSION = '2024-12-01-preview'
AZURE_DEPLOYMENT = 'gpt-5.1'
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# this keeps paid llm calls off until i turn them on myself.
RUN_PAID_LLM = True

INITIAL_PAIR_TARGET = 10000
N_CLUSTERS = 5000
PRINCIPLES_PER_PAIR = 2

# larger batches keep the request count lower.
GENERATION_PAIRS_PER_CALL = 100

MAX_POST_CHARS = 1800
MIN_APPLICABILITY = .10
FINAL_N_PRINCIPLES = 20
MAX_LLM_TEST_PRINCIPLES = 5000
TOP_RELEVANT_PAIRS_PER_PRINCIPLE = 25
LLM_JUDGMENTS_PER_CALL = 100

# every candidate starts small, then only the stronger ones keep going.
INCREMENTAL_TEST_SIZES = [3, 10, 25]
ROUND_SURVIVOR_LIMITS = {3: 500, 10: 100}
MIN_APPLICABLE_BY_ROUND = {3: 1, 10: 3, 25: 3}
MIN_ROUND_ACCURACY = .52
MAX_RETRIES = 2

# this estimates the worst-case call count before anything paid runs.
PROJECTED_GENERATION_CALLS = math.ceil(
    INITIAL_PAIR_TARGET / GENERATION_PAIRS_PER_CALL
)

PROJECTED_TEST_CALLS = (
    math.ceil(N_CLUSTERS * 3 / LLM_JUDGMENTS_PER_CALL)
    + math.ceil(
        ROUND_SURVIVOR_LIMITS[3] * (10 - 3) / LLM_JUDGMENTS_PER_CALL
    )
    + math.ceil(
        ROUND_SURVIVOR_LIMITS[10] * (25 - 10) / LLM_JUDGMENTS_PER_CALL
    )
)

PROJECTED_MAX_CALLS = PROJECTED_GENERATION_CALLS + PROJECTED_TEST_CALLS
MAX_TOTAL_LLM_CALLS = 305  # this leaves 5 calls of retry/headroom.
FORCE_REGENERATE = False
FORCE_REEMBED = False
FORCE_RETEST = False

if not DATA_PATH.exists():
    from google.colab import files

    up = files.upload()

    if not up:
        raise FileNotFoundError('No CSV uploaded.')

    DATA_PATH = Path('/content') / next(iter(up))

client = None

if RUN_PAID_LLM:
    subscription_key = getpass.getpass('Azure OpenAI API key: ')

    client = AzureOpenAI(
        api_version=AZURE_API_VERSION,
        azure_endpoint=AZURE_ENDPOINT,
        api_key=subscription_key
    )

print('Using:', DATA_PATH)
print(
    f'Target experiment: {INITIAL_PAIR_TARGET:,} usable pairs -> '
    f'{N_CLUSTERS:,} candidate principles.'
)
print('\nPAID-CALL ESTIMATE (worst case, before checkpoints):')
print(f'  Principle generation: {PROJECTED_GENERATION_CALLS} calls')
print(f'  5,000-candidate staged testing: {PROJECTED_TEST_CALLS} calls')
print(f'  Total planned maximum: {PROJECTED_MAX_CALLS} calls')
print(f'  Emergency hard cap: {MAX_TOTAL_LLM_CALLS} calls')
print(
    '  Note: Azure billing is token-based, so call count is a '
    'safety proxy, not a dollar estimate.'
)
print(
    'PAID LLM STATUS:',
    'ENABLED' if RUN_PAID_LLM else 'BLOCKED (dry-run safe)'
)

if PROJECTED_MAX_CALLS > MAX_TOTAL_LLM_CALLS:
    raise RuntimeError(
        'Configuration error: projected call plan exceeds the hard cap.'
    )


## setup helpers, checkpoints, and llm call tracking


In [ ]:
CALL_LOG = OUTPUT_DIR / 'llm_call_log.jsonl'

def clean_field(v):
    return '' if pd.isna(v) else re.sub(r'\s+', ' ', str(v)).strip()

def build_post_text(t, b):
    t, b = clean_field(t), clean_field(b)
    parts = []

    if t:
        parts.append('Title: ' + t)

    if b:
        parts.append('Body: ' + b)

    return '\n'.join(parts)[:MAX_POST_CHARS]

def read_jsonl(p):
    p = Path(p)

    if not p.exists():
        return []

    with p.open(encoding='utf-8') as f:
        return [json.loads(x) for x in f if x.strip()]

def append_jsonl(p, rows):
    with Path(p).open('a', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def current_call_count():
    return len(read_jsonl(CALL_LOG))

def estimate_calls(n, b):
    return int(math.ceil(n / max(1, b)))

def budget_check(stage, n):
    used = current_call_count()
    projected = used + n

    print(
        f'{stage}: about {n} new calls; '
        f'projected {projected}/{MAX_TOTAL_LLM_CALLS}'
    )

    if projected > MAX_TOTAL_LLM_CALLS:
        raise RuntimeError(
            'Projected LLM calls exceed the hard cap; no calls sent.'
        )

    if n and not RUN_PAID_LLM:
        print(
            '  DRY RUN: paid calls remain blocked. '
            'Set RUN_PAID_LLM=True only after reviewing this estimate.'
        )

def extract_json(text):
    text = (text or '').strip()
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.I)
    text = re.sub(r'\s*```$', '', text)

    try:
        return json.loads(text)

    except Exception:
        starts = [
            i for i in [text.find('{'), text.find('[')]
            if i >= 0
        ]

        if not starts:
            raise

        return json.loads(
            text[
                min(starts):
                max(text.rfind('}'), text.rfind(']')) + 1
            ]
        )

def chat_json(
    system,
    user,
    stage,
    max_tokens=12000,
    retries=MAX_RETRIES
):
    if not RUN_PAID_LLM:
        raise RuntimeError(
            'Paid LLM calls are BLOCKED. Review the printed estimate, '
            'then set RUN_PAID_LLM=True and rerun from the configuration cell.'
        )

    if client is None:
        raise RuntimeError(
            'RUN_PAID_LLM=True but Azure client is not initialized. '
            'Rerun the configuration cell.'
        )

    last = None

    for a in range(retries):
        if current_call_count() >= MAX_TOTAL_LLM_CALLS:
            raise RuntimeError(
                'LLM call hard cap reached. '
                'No additional paid call was sent.'
            )

        try:
            r = client.chat.completions.create(
                model=AZURE_DEPLOYMENT,
                messages=[
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': user}
                ],
                max_completion_tokens=max_tokens,
                response_format={'type': 'json_object'}
            )

            u = getattr(r, 'usage', None)

            append_jsonl(
                CALL_LOG,
                [{
                    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
                    'stage': stage,
                    'attempt': a + 1,
                    'prompt_tokens': getattr(u, 'prompt_tokens', None),
                    'completion_tokens': getattr(u, 'completion_tokens', None),
                    'total_tokens': getattr(u, 'total_tokens', None)
                }]
            )

            return extract_json(r.choices[0].message.content)

        except Exception as e:
            last = e
            print(stage, 'attempt', a + 1, 'failed:', e)

            if a < retries - 1:
                time.sleep(min(20, 3 * 2 ** a))

    raise RuntimeError(stage) from last

def chunks(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]

def safe_corr(x, y, kind='pearson'):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)

    if m.sum() < 3 or np.std(x[m]) == 0 or np.std(y[m]) == 0:
        return np.nan, np.nan

    return (
        pearsonr(x[m], y[m])
        if kind == 'pearson'
        else spearmanr(x[m], y[m])
    )


## load the 10,000 preference pairs

This checks that there are enough usable pairs and keeps the run fixed at 10,000 so the later LLM cost doesn't grow unexpectedly.


In [ ]:
required = {
    'pair_id',
    'preferred_post_title',
    'preferred_post_body',
    'preferred_upvote_ratio',
    'nonpreferred_post_title',
    'nonpreferred_post_body',
    'nonpreferred_upvote_ratio'
}

df = pd.read_csv(
    DATA_PATH,
    keep_default_na=False
)

missing = required - set(df.columns)

if missing:
    raise ValueError(f'Missing columns: {sorted(missing)}')

# build llm-ready post text.
df['preferred_text'] = [
    build_post_text(t, b)
    for t, b in zip(
        df.preferred_post_title,
        df.preferred_post_body
    )
]

df['nonpreferred_text'] = [
    build_post_text(t, b)
    for t, b in zip(
        df.nonpreferred_post_title,
        df.nonpreferred_post_body
    )
]

before = len(df)

# only remove genuinely empty post text.
empty_mask = (
    (df['preferred_text'].str.strip() == '')
    |
    (df['nonpreferred_text'].str.strip() == '')
)

if empty_mask.any():
    print(
        f'Warning: {empty_mask.sum():,} genuinely empty pairs '
        'were found after loading.'
    )

df = df.loc[~empty_mask].copy().reset_index(drop=True)

# pair ids are strings because they are also used for checkpoints.
df['pair_id'] = df['pair_id'].astype(str)

# ensure ratios are numeric.
df['preferred_upvote_ratio'] = pd.to_numeric(
    df['preferred_upvote_ratio'],
    errors='coerce'
)

df['nonpreferred_upvote_ratio'] = pd.to_numeric(
    df['nonpreferred_upvote_ratio'],
    errors='coerce'
)

if 'upvote_ratio_gap' in df.columns:
    df['upvote_ratio_gap'] = pd.to_numeric(
        df['upvote_ratio_gap'],
        errors='coerce'
    )
else:
    df['upvote_ratio_gap'] = (
        df['preferred_upvote_ratio']
        - df['nonpreferred_upvote_ratio']
    )

# the sql output should contain the full 10,000 pairs.
if len(df) < INITIAL_PAIR_TARGET:
    raise ValueError(
        f'CSV contained {before:,} rows, but only {len(df):,} '
        f'contain usable text. Expected {INITIAL_PAIR_TARGET:,}.'
    )

if len(df) > INITIAL_PAIR_TARGET:
    df = (
        df.sample(
            n=INITIAL_PAIR_TARGET,
            random_state=SEED
        )
        .sort_values('pair_id')
        .reset_index(drop=True)
    )

if df['pair_id'].duplicated().any():
    raise ValueError(
        'pair_id must be unique for checkpointing. '
        'Duplicate pair_id values were found.'
    )

df.to_csv(
    OUTPUT_DIR / 'initial_10000_pairs_used.csv',
    index=False
)

print(
    f'Loaded {before:,}; '
    f'using exactly {len(df):,} preference pairs.'
)


## step 1 — generate candidate principles in batches


In [ ]:

# step 1 — robust batched llm principle generation

# cost behavior:
# - normal plan: ~100 generation calls.
# - maximum generation repair headroom: 5 calls.
# - valid results are checkpointed immediately.
# - a single omitted pair no longer destroys a whole batch.
# - only missing pairs are eligible for a repair pass.

GEN = OUTPUT_DIR / 'generation_checkpoint.jsonl'

if FORCE_REGENERATE and GEN.exists():
    GEN.unlink()

def generation_call_count():
    """Count only calls belonging to Step 1 generation."""
    return sum(
        1
        for x in read_jsonl(CALL_LOG)
        if str(x.get('stage', '')).startswith('generation')
    )

# the normal plan is 100 calls.
# the overall notebook already reserved 5 calls of headroom,
# so step 1 may use at most 105 generation calls total.
GENERATION_NORMAL_CALLS = PROJECTED_GENERATION_CALLS
GENERATION_MAX_CALLS = PROJECTED_GENERATION_CALLS + 5

def completed_generation_ids():
    return {
        str(x['pair_id'])
        for x in read_jsonl(GEN)
        if x.get('pair_id') is not None
    }

def make_generation_payload(rows):
    return [
        {
            'pair_id': str(r.pair_id),
            'preferred_post': r.preferred_text,
            'nonpreferred_post': r.nonpreferred_text
        }
        for r in rows
    ]

def save_valid_generation_results(payload, out):
    """
    Save every complete result immediately.

    Returns:
        saved_count
        missing_pair_ids
    """

    expected_ids = {
        str(x['pair_id'])
        for x in payload
    }

    got = {}

    for item in out.get('results', []):
        pid = str(item.get('pair_id', ''))

        if pid in expected_ids:
            got[pid] = item

    already_done = completed_generation_ids()

    rows = []
    missing = []

    for x in payload:
        pid = str(x['pair_id'])

        if pid in already_done:
            continue

        principles = got.get(pid, {}).get('principles', [])

        if not isinstance(principles, list):
            principles = []

        principles = [
            str(p).strip()
            for p in principles
            if str(p).strip()
        ]

        if len(principles) >= PRINCIPLES_PER_PAIR:

            rows.append({
                'pair_id': pid,
                'principles': principles[:PRINCIPLES_PER_PAIR]
            })

        else:
            missing.append(pid)

    # critical difference from old code:
    # save all good results even when some items were omitted.
    if rows:
        append_jsonl(GEN, rows)

    return len(rows), missing

# primary generation pass

done = completed_generation_ids()

pending = [
    r
    for _, r in df.iterrows()
    if str(r.pair_id) not in done
]

needed_calls = estimate_calls(
    len(pending),
    GENERATION_PAIRS_PER_CALL
)

used_generation_calls = generation_call_count()

remaining_generation_budget = (
    GENERATION_MAX_CALLS
    - used_generation_calls
)

print(
    f'Generation checkpoint already contains '
    f'{len(done):,}/{len(df):,} pairs.'
)

print(
    f'Generation calls already paid/logged: '
    f'{used_generation_calls}/{GENERATION_MAX_CALLS}'
)

print(
    f'Primary pass currently needs about '
    f'{needed_calls} calls.'
)

if needed_calls > remaining_generation_budget:
    raise RuntimeError(
        f'Need approximately {needed_calls} generation calls, '
        f'but only {remaining_generation_budget} remain inside '
        f'the protected Step 1 budget.'
    )

budget_check(
    'Generation',
    needed_calls
)

omitted_this_pass = []

for batch in tqdm(
    list(chunks(pending, GENERATION_PAIRS_PER_CALL)),
    desc='Generating'
):

    payload = make_generation_payload(batch)

    prompt = f'''
For EVERY preference pair below, generate exactly
{PRINCIPLES_PER_PAIR} concise principles explaining why the
community preferred the preferred post.

REQUIREMENTS:

1. Return exactly one result object for EVERY supplied pair_id.
2. Copy each pair_id EXACTLY as supplied.
3. Do not skip, merge, reorder identifiers, or invent pair IDs.
4. Each result must contain exactly {PRINCIPLES_PER_PAIR}
   different principles.
5. Each principle must begin with:
   "Prefer posts that"
   OR
   "Avoid posts that"
6. Describe observable differences in post content.
7. Do not mention scores, votes, upvote ratios, IDs, or popularity.
8. Each principle must use at most 18 words.

Return ONLY valid JSON in this exact general format:

{{
  "results": [
    {{
      "pair_id": "...",
      "principles": [
        "Prefer posts that ...",
        "Avoid posts that ..."
      ]
    }}
  ]
}}

There must be one results entry for every pair_id.

PAIRS:
{json.dumps(payload, ensure_ascii=False)}
'''

    out = chat_json(
        'Analyze pairwise community preferences. '
        'Return complete valid JSON only.',
        prompt,
        'generation'
    )

    saved_count, missing_ids = (
        save_valid_generation_results(
            payload,
            out
        )
    )

    omitted_this_pass.extend(missing_ids)

    if missing_ids:
        print(
            f'\nBatch returned {saved_count}/{len(payload)} '
            f'complete pair results. '
            f'{len(missing_ids)} will remain pending.'
        )

# optional repair pass

# only genuinely missing pairs are retried.
# never exceed the five-call repair allowance.

done = completed_generation_ids()

remaining = [
    r
    for _, r in df.iterrows()
    if str(r.pair_id) not in done
]

if remaining:

    used_generation_calls = generation_call_count()

    repair_calls_available = max(
        0,
        GENERATION_MAX_CALLS
        - used_generation_calls
    )

    repair_calls_needed = estimate_calls(
        len(remaining),
        GENERATION_PAIRS_PER_CALL
    )

    repair_calls_to_use = min(
        repair_calls_needed,
        repair_calls_available
    )

    print()
    print(
        f'Primary generation pass left '
        f'{len(remaining):,} pairs incomplete.'
    )

    print(
        f'Repair calls available: '
        f'{repair_calls_available}'
    )

    if repair_calls_to_use > 0:

        print(
            f'Using at most {repair_calls_to_use} '
            f'repair call(s), only for missing pairs.'
        )

        repair_batches = list(
            chunks(
                remaining,
                GENERATION_PAIRS_PER_CALL
            )
        )[:repair_calls_to_use]

        for batch in tqdm(
            repair_batches,
            desc='Repair missing'
        ):

            payload = make_generation_payload(batch)

            prompt = f'''
Some pair IDs were omitted from a previous generation response.

For EVERY pair below, return exactly
{PRINCIPLES_PER_PAIR} concise principles.

You MUST return one object for every supplied pair_id.

Copy every pair_id EXACTLY.

Principles must:
- begin with "Prefer posts that" or "Avoid posts that"
- describe observable content differences
- avoid scores, votes, popularity, and IDs
- use at most 18 words

Return JSON only:

{{
  "results": [
    {{
      "pair_id": "...",
      "principles": [
        "...",
        "..."
      ]
    }}
  ]
}}

PAIRS:
{json.dumps(payload, ensure_ascii=False)}
'''

            out = chat_json(
                'Repair omitted principle-generation results. '
                'Return complete valid JSON only.',
                prompt,
                'generation_repair'
            )

            saved_count, missing_ids = (
                save_valid_generation_results(
                    payload,
                    out
                )
            )

            print(
                f'Repair saved '
                f'{saved_count}/{len(payload)}.'
            )

# final generation checkpoint summary

done = completed_generation_ids()

remaining_ids = (
    set(df['pair_id'].astype(str))
    - done
)

coverage = len(done) / len(df)

raw = []

for r in read_jsonl(GEN):

    principles = r.get('principles', [])

    for i, p in enumerate(principles, 1):

        raw.append({
            'pair_id': str(r['pair_id']),
            'principle_index': i,
            'raw_principle': p
        })

raw_df = pd.DataFrame(raw)

raw_df.to_csv(
    OUTPUT_DIR / 'raw_principles.csv',
    index=False
)

# save missing ids for complete transparency.
pd.DataFrame({
    'missing_pair_id': sorted(remaining_ids)
}).to_csv(
    OUTPUT_DIR / 'generation_missing_pair_ids.csv',
    index=False
)

print()
print('================ GENERATION SUMMARY ================')

print(
    f'Pairs with generated principles: '
    f'{len(done):,}/{len(df):,} '
    f'({coverage:.2%})'
)

print(
    f'Pairs still missing: '
    f'{len(remaining_ids):,}'
)

print(
    f'Raw principles: '
    f'{len(raw_df):,}'
)

print(
    f'Total paid calls logged so far: '
    f'{current_call_count()}'
)

print(
    f'Generation-stage calls: '
    f'{generation_call_count()}/'
    f'{GENERATION_MAX_CALLS}'
)

print('====================================================')


In [ ]:

# zero-cost checkpoint audit
# run this immediately after stopping step 1.

RUN_PAID_LLM = False   # blocks any accidental paid llm call

GEN = OUTPUT_DIR / 'generation_checkpoint.jsonl'

generation_rows = read_jsonl(GEN)
call_rows = read_jsonl(CALL_LOG)

completed_pair_ids = {
    str(x['pair_id'])
    for x in generation_rows
    if x.get('pair_id') is not None
}

generation_calls = [
    x for x in call_rows
    if str(x.get('stage', '')).startswith('generation')
]

raw_principle_count = sum(
    len(x.get('principles', []))
    for x in generation_rows
)

prompt_tokens = sum(
    x.get('prompt_tokens') or 0
    for x in generation_calls
)

completion_tokens = sum(
    x.get('completion_tokens') or 0
    for x in generation_calls
)

total_tokens = sum(
    x.get('total_tokens') or 0
    for x in generation_calls
)

print("========== SAVED WORK ==========")
print(f"Pairs checkpointed: {len(completed_pair_ids):,} / 10,000")
print(f"Principles checkpointed: {raw_principle_count:,}")
print(f"Generation calls logged: {len(generation_calls):,}")
print()
print("========== LOGGED TOKEN USE ==========")
print(f"Prompt tokens:     {prompt_tokens:,}")
print(f"Completion tokens: {completion_tokens:,}")
print(f"Total tokens:      {total_tokens:,}")
print()
print(f"Pairs still unprocessed: {10_000 - len(completed_pair_ids):,}")
print("Paid LLM calls are now BLOCKED.")


In [ ]:

# zero-cost principle yield audit

# this makes no llm/api calls.
# it tells us how many generated principles survive cleaning
# and estimates how many more representative pairs we actually
# need to send to the llm.

import math
import re
import pandas as pd

RUN_PAID_LLM = False

GEN = OUTPUT_DIR / 'generation_checkpoint.jsonl'

generation_rows = read_jsonl(GEN)

raw = []

for r in generation_rows:
    for i, p in enumerate(r.get('principles', []), 1):
        raw.append({
            'pair_id': str(r['pair_id']),
            'principle_index': i,
            'raw_principle': p
        })

raw_df_audit = pd.DataFrame(raw)

def audit_clean_principle(text):

    if pd.isna(text):
        return None

    text = re.sub(r'\s+', ' ', str(text)).strip()

    text = re.sub(
        r'^[-*\s]+',
        '',
        text
    )

    text = re.sub(
        r'^\d+[.)]\s*',
        '',
        text
    )

    text = text.strip(' "\'`')

    if not text:
        return None

    low = text.lower()

    if not (
        low.startswith('prefer posts that')
        or
        low.startswith('avoid posts that')
    ):
        text = (
            'Prefer posts that '
            + text[0].lower()
            + text[1:]
        )

    text = text.rstrip(' .') + '.'

    if not 4 <= len(text.split()) <= 24:
        return None

    return text

audit = raw_df_audit.copy()

audit['cleaned_principle'] = (
    audit['raw_principle']
    .map(audit_clean_principle)
)

valid = (
    audit
    .dropna(subset=['cleaned_principle'])
    .copy()
)

valid['dedupe_key'] = (
    valid['cleaned_principle']
    .str.lower()
    .str.replace(
        r'[^a-z0-9 ]',
        '',
        regex=True
    )
)

unique_cleaned = (
    valid
    .drop_duplicates('dedupe_key')
    .copy()
)

raw_n = len(raw_df_audit)
valid_n = len(valid)
unique_n = len(unique_cleaned)

retention_rate = (
    unique_n / raw_n
    if raw_n
    else 0
)

# give ourselves some margin above 5,000 because the clustering
# stage needs at least 5,000 unique cleaned principles.
SAFE_UNIQUE_TARGET = 5500

if retention_rate > 0:

    estimated_raw_needed = math.ceil(
        SAFE_UNIQUE_TARGET
        / retention_rate
    )

    estimated_total_pairs_needed = math.ceil(
        estimated_raw_needed
        / PRINCIPLES_PER_PAIR
    )

    already_completed_pairs = len({
        str(r['pair_id'])
        for r in generation_rows
    })

    estimated_more_pairs = max(
        0,
        estimated_total_pairs_needed
        - already_completed_pairs
    )

    estimated_more_calls = math.ceil(
        estimated_more_pairs
        / GENERATION_PAIRS_PER_CALL
    )

else:

    estimated_raw_needed = None
    estimated_total_pairs_needed = None
    estimated_more_pairs = None
    estimated_more_calls = None

print("========== PRINCIPLE YIELD ==========")

print(
    f"Raw principles:             "
    f"{raw_n:,}"
)

print(
    f"Valid after cleaning:       "
    f"{valid_n:,}"
)

print(
    f"Unique cleaned principles:  "
    f"{unique_n:,}"
)

print(
    f"Unique retention rate:      "
    f"{retention_rate:.1%}"
)

print()

print("========== COST-SAVING ESTIMATE ==========")

print(
    f"Safe unique target:         "
    f"{SAFE_UNIQUE_TARGET:,}"
)

print(
    f"Estimated raw principles needed: "
    f"{estimated_raw_needed:,}"
)

print(
    f"Estimated total generation pairs needed: "
    f"{estimated_total_pairs_needed:,}"
)

print(
    f"Already checkpointed pairs: "
    f"{already_completed_pairs:,}"
)

print(
    f"Estimated ADDITIONAL pairs needed: "
    f"{estimated_more_pairs:,}"
)

print(
    f"Estimated additional LLM calls "
    f"at current batch size: "
    f"{estimated_more_calls:,}"
)

print()

print(
    "NO paid LLM calls were made by this cell."
)


In [ ]:

# zero-cost representative-pair selection

# goal:
# keep the 799 pairs already paid for.
# select only enough additional diverse pairs from the full
# 10,000-pair dataset to support ~5,500 raw principles.

# no azure/openai llm calls are made here.
# sentence-transformer embeddings run locally in colab.

import math
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans

# safety: paid llm calls stay blocked.

RUN_PAID_LLM = False

# targets

# we ultimately need 5,000 unique principles for 5,000
# principle clusters.

# 5,500 gives us a 10% buffer.

# at 2 principles per pair:
# 5,500 / 2 = 2,750 total generation pairs.

TARGET_RAW_PRINCIPLES = 5500

TARGET_GENERATION_PAIRS = math.ceil(
    TARGET_RAW_PRINCIPLES / PRINCIPLES_PER_PAIR
)

GEN = OUTPUT_DIR / 'generation_checkpoint.jsonl'

completed_ids = {
    str(x['pair_id'])
    for x in read_jsonl(GEN)
    if x.get('pair_id') is not None
}

already_completed = len(completed_ids)

additional_needed = max(
    0,
    TARGET_GENERATION_PAIRS - already_completed
)

print("========== GENERATION TARGET ==========")

print(
    f"Already-paid completed pairs: "
    f"{already_completed:,}"
)

print(
    f"Target total generation pairs: "
    f"{TARGET_GENERATION_PAIRS:,}"
)

print(
    f"Additional representative pairs needed: "
    f"{additional_needed:,}"
)

print(
    f"Expected raw principles at target: "
    f"{TARGET_GENERATION_PAIRS * PRINCIPLES_PER_PAIR:,}"
)

print()

# identify all unprocessed pairs

df = df.copy()

df['pair_id'] = df['pair_id'].astype(str)

pending_mask = ~df['pair_id'].isin(completed_ids)

pending_df = (
    df.loc[pending_mask]
    .copy()
    .reset_index()
    .rename(columns={'index': 'original_df_index'})
)

print(
    f"Unprocessed pairs available: "
    f"{len(pending_df):,}"
)

if additional_needed > len(pending_df):

    raise ValueError(
        f"Need {additional_needed:,} additional pairs, "
        f"but only {len(pending_df):,} are available."
    )

# embed all 10,000 pairs locally

# this is the same pair-post embedding file step 2 uses later.

# therefore this work should not need to be repeated in step 2.

PAIR_EMBED_FILE = (
    OUTPUT_DIR / 'pair_post_embeddings.npz'
)

model = SentenceTransformer(EMBED_MODEL)

use_saved = False

if PAIR_EMBED_FILE.exists():

    try:

        z = np.load(
            PAIR_EMBED_FILE,
            allow_pickle=True
        )

        saved_pref = z['preferred']
        saved_non = z['nonpreferred']

        if (
            len(saved_pref) == len(df)
            and
            len(saved_non) == len(df)
        ):

            pref_emb = saved_pref
            non_emb = saved_non

            use_saved = True

            print(
                "Using existing saved pair embeddings."
            )

        else:

            print(
                "Existing pair embeddings do not match "
                "the current 10,000-pair dataset. "
                "Recomputing locally."
            )

    except Exception as exc:

        print(
            "Could not safely reuse existing embeddings. "
            "Recomputing locally."
        )

        print("Reason:", exc)

if not use_saved:

    print()
    print(
        "Embedding the 10,000 preferred and "
        "10,000 nonpreferred posts locally..."
    )

    all_embeddings = model.encode(

        df['preferred_text'].tolist()
        +
        df['nonpreferred_text'].tolist(),

        batch_size=64,

        show_progress_bar=True,

        normalize_embeddings=True
    )

    pref_emb = all_embeddings[:len(df)]

    non_emb = all_embeddings[len(df):]

    np.savez_compressed(

        PAIR_EMBED_FILE,

        preferred=pref_emb,

        nonpreferred=non_emb,

        pair_ids=df['pair_id']
        .astype(str)
        .to_numpy(dtype='U')
    )

    print(
        "Saved reusable pair embeddings to:",
        PAIR_EMBED_FILE
    )

# build a representation of each preference pair

# we want diversity in both:

# 1. what topic/content the two posts discuss
# 2. what distinguishes preferred from nonpreferred

# topic_vector:
# midpoint of preferred + nonpreferred embeddings

# contrast_vector:
# direction from nonpreferred -> preferred

# concatenating them makes pair selection sensitive to both
# topic and preference contrast.

def normalize_rows(x):

    norms = np.linalg.norm(
        x,
        axis=1,
        keepdims=True
    )

    norms[norms == 0] = 1

    return x / norms

topic_vectors = normalize_rows(
    pref_emb + non_emb
)

contrast_vectors = normalize_rows(
    pref_emb - non_emb
)

pair_vectors = np.concatenate(
    [
        topic_vectors,
        contrast_vectors
    ],
    axis=1
)

pair_vectors = normalize_rows(
    pair_vectors
)

# pull vectors only for the still-unprocessed pairs.

pending_indices = (
    pending_df['original_df_index']
    .to_numpy()
)

pending_vectors = pair_vectors[
    pending_indices
]

# cluster the 9,201 unprocessed pairs into exactly the number
# of new representatives we need.

# this uses local sklearn only.
# no gpt / azure calls.

print()
print(
    f"Clustering {len(pending_df):,} unprocessed pairs "
    f"into {additional_needed:,} diversity groups..."
)

clusterer = MiniBatchKMeans(

    n_clusters=additional_needed,

    random_state=SEED,

    batch_size=2048,

    n_init='auto'
)

labels = clusterer.fit_predict(
    pending_vectors
)

centers = clusterer.cluster_centers_

# pick the pair closest to each cluster center

# this gives one representative comparison per diversity group.

assigned_centers = centers[labels]

distance_to_center = np.sum(
    (
        pending_vectors
        -
        assigned_centers
    ) ** 2,
    axis=1
)

selection_work = pending_df.copy()

selection_work[
    'representative_cluster'
] = labels

selection_work[
    'distance_to_cluster_center'
] = distance_to_center

selected_indices = (

    selection_work

    .groupby(
        'representative_cluster'
    )[
        'distance_to_cluster_center'
    ]

    .idxmin()

    .tolist()
)

selected_new = (
    selection_work
    .loc[selected_indices]
    .copy()
)

# extremely unlikely fallback:

# minibatchkmeans can theoretically leave an unused cluster.
# if that happens, deterministically add more unused pairs so
# the exact requested count is still obtained.

if len(selected_new) < additional_needed:

    shortfall = (
        additional_needed
        -
        len(selected_new)
    )

    already_selected_rows = set(
        selected_new.index
    )

    leftovers = selection_work.loc[
        ~selection_work.index.isin(
            already_selected_rows
        )
    ].copy()

    # favor unusual examples:
    # farther from their assigned cluster centers.
    filler = (

        leftovers

        .sort_values(
            'distance_to_cluster_center',
            ascending=False
        )

        .head(shortfall)
    )

    selected_new = pd.concat(
        [
            selected_new,
            filler
        ],
        ignore_index=False
    )

# exact-count safety check.

if len(selected_new) != additional_needed:

    raise RuntimeError(
        f"Representative selection produced "
        f"{len(selected_new):,} pairs; "
        f"expected {additional_needed:,}."
    )

if selected_new[
    'pair_id'
].duplicated().any():

    raise RuntimeError(
        "Duplicate pair IDs found in "
        "representative selection."
    )

# save the new pairs that gpt will actually see

REPRESENTATIVE_FILE = (
    OUTPUT_DIR
    /
    'step1_additional_representative_pairs.csv'
)

save_columns = [

    'pair_id',

    'preferred_post_title',
    'preferred_post_body',
    'preferred_upvote_ratio',

    'nonpreferred_post_title',
    'nonpreferred_post_body',
    'nonpreferred_upvote_ratio',

    'preferred_text',
    'nonpreferred_text',

    'representative_cluster',
    'distance_to_cluster_center'
]

save_columns = [
    c
    for c in save_columns
    if c in selected_new.columns
]

selected_new[
    save_columns
].to_csv(
    REPRESENTATIVE_FILE,
    index=False
)

# also save the complete generation target:

# 799 already paid
# + 1,951 newly selected
# = 2,750 generation pairs

selected_new_ids = set(
    selected_new['pair_id'].astype(str)
)

target_ids = (
    completed_ids
    |
    selected_new_ids
)

if len(target_ids) != TARGET_GENERATION_PAIRS:

    raise RuntimeError(
        f"Expected {TARGET_GENERATION_PAIRS:,} "
        f"total target pair IDs, "
        f"but got {len(target_ids):,}."
    )

generation_target = (

    df[
        df['pair_id'].isin(target_ids)
    ]

    .copy()

    .sort_values('pair_id')
)

generation_target.to_csv(

    OUTPUT_DIR
    /
    'step1_generation_target_2750_pairs.csv',

    index=False
)

# summary

print()
print(
    "========== REPRESENTATIVE SELECTION COMPLETE =========="
)

print(
    f"Full preference dataset:          "
    f"{len(df):,}"
)

print(
    f"Already-paid pairs retained:      "
    f"{already_completed:,}"
)

print(
    f"New representative pairs chosen: "
    f"{len(selected_new):,}"
)

print(
    f"Total generation target:          "
    f"{len(generation_target):,}"
)

print(
    f"Expected raw principles:          "
    f"{len(generation_target) * PRINCIPLES_PER_PAIR:,}"
)

print()

print(
    "PAID LLM CALLS MADE BY THIS CELL: 0"
)

print(
    "RUN_PAID_LLM remains:",
    RUN_PAID_LLM
)

print(
    "========================================================"
)


In [ ]:

# step 1 — cost-controlled representative principle generation

# processes only the selected 2,750-pair generation target.

# already completed: 799 pairs
# expected new pairs: 1,951
# expected new calls: 20 at 100 pairs/call

# important:
# - saves valid results even if gpt omits some pair ids.
# - does not automatically repair omissions.
# - maximum 20 new successful llm calls in this cell.

import getpass
import json
import math
import re
import pandas as pd

from openai import AzureOpenAI
from tqdm.auto import tqdm

GEN = OUTPUT_DIR / 'generation_checkpoint.jsonl'

TARGET_FILE = (
    OUTPUT_DIR /
    'step1_generation_target_2750_pairs.csv'
)

# load the preselected generation target

if not TARGET_FILE.exists():
    raise FileNotFoundError(
        f'Missing representative-pair target file:\n{TARGET_FILE}'
    )

generation_target = pd.read_csv(
    TARGET_FILE,
    keep_default_na=False
)

generation_target['pair_id'] = (
    generation_target['pair_id']
    .astype(str)
)

# find what has already been paid for / checkpointed

def completed_generation_ids():

    return {
        str(x['pair_id'])
        for x in read_jsonl(GEN)
        if x.get('pair_id') is not None
    }

done_before = completed_generation_ids()

pending_target = (
    generation_target[
        ~generation_target['pair_id'].isin(done_before)
    ]
    .copy()
    .reset_index(drop=True)
)

expected_calls = math.ceil(
    len(pending_target)
    / GENERATION_PAIRS_PER_CALL
)

print("========== BEFORE PAID GENERATION ==========")

print(
    f"Total representative generation target: "
    f"{len(generation_target):,}"
)

print(
    f"Already checkpointed: "
    f"{len(done_before):,}"
)

print(
    f"Still requiring generation: "
    f"{len(pending_target):,}"
)

print(
    f"Expected NEW LLM calls: "
    f"{expected_calls:,}"
)

print(
    f"Batch size: "
    f"{GENERATION_PAIRS_PER_CALL}"
)

print("============================================")

# strict local cost limit

MAX_NEW_CALLS_THIS_CELL = 20

if expected_calls > MAX_NEW_CALLS_THIS_CELL:

    raise RuntimeError(
        f'This cell would require approximately '
        f'{expected_calls} calls, which exceeds its '
        f'local cap of {MAX_NEW_CALLS_THIS_CELL}. '
        f'No paid calls were made.'
    )

# explicitly enable paid calls

# we do this here rather than rerunning the whole
# configuration cell.

RUN_PAID_LLM = True

subscription_key = getpass.getpass(
    'Azure OpenAI API key: '
)

client = AzureOpenAI(
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=subscription_key
)

print()
print(
    f'Paid generation enabled for this cell only. '
    f'Local maximum: {MAX_NEW_CALLS_THIS_CELL} new calls.'
)

# helper: save every valid result

# unlike the original step 1, one omitted pair does not
# discard the rest of the paid batch.

def save_valid_generation_results(
    payload,
    model_output
):

    expected_ids = {
        str(x['pair_id'])
        for x in payload
    }

    returned = {}

    for item in model_output.get(
        'results',
        []
    ):

        pid = str(
            item.get(
                'pair_id',
                ''
            )
        )

        if pid in expected_ids:

            returned[pid] = item

    already_done = (
        completed_generation_ids()
    )

    valid_rows = []

    missing_ids = []

    for item in payload:

        pid = str(
            item['pair_id']
        )

        if pid in already_done:

            continue

        principles = (
            returned
            .get(
                pid,
                {}
            )
            .get(
                'principles',
                []
            )
        )

        if not isinstance(
            principles,
            list
        ):

            principles = []

        principles = [

            str(p).strip()

            for p in principles

            if str(p).strip()

        ]

        if len(principles) >= PRINCIPLES_PER_PAIR:

            valid_rows.append({

                'pair_id':
                    pid,

                'principles':
                    principles[
                        :PRINCIPLES_PER_PAIR
                    ]

            })

        else:

            missing_ids.append(
                pid
            )

    # critical:
    # save all successful results immediately.

    if valid_rows:

        append_jsonl(
            GEN,
            valid_rows
        )

    return (
        len(valid_rows),
        missing_ids
    )

# generate principles

new_calls_sent = 0

all_omitted_ids = []

batches = list(
    chunks(
        [
            r
            for _, r
            in pending_target.iterrows()
        ],
        GENERATION_PAIRS_PER_CALL
    )
)

for batch in tqdm(
    batches,
    desc='Representative generation'
):

    # local hard stop.

    if new_calls_sent >= MAX_NEW_CALLS_THIS_CELL:

        print()
        print(
            'LOCAL GENERATION CALL LIMIT REACHED.'
        )

        break

    payload = [

        {
            'pair_id':
                str(r.pair_id),

            'preferred_post':
                r.preferred_text,

            'nonpreferred_post':
                r.nonpreferred_text
        }

        for r in batch
    ]

    prompt = f'''
For EVERY preference pair below, generate exactly
{PRINCIPLES_PER_PAIR} concise principles explaining why the
community preferred the preferred post.

REQUIREMENTS:

1. Return one result object for EVERY supplied pair_id.

2. Copy each pair_id EXACTLY as supplied.

3. Do not skip, combine, alter, or invent pair IDs.

4. Each result must contain exactly
   {PRINCIPLES_PER_PAIR} different principles.

5. Each principle must begin with either:
   "Prefer posts that"
   OR
   "Avoid posts that"

6. Describe observable differences in post content.

7. Do NOT mention:
   - scores
   - votes
   - upvote ratios
   - popularity
   - pair IDs

8. Each principle must contain at most 18 words.

Return ONLY valid JSON:

{{
  "results": [
    {{
      "pair_id": "...",
      "principles": [
        "Prefer posts that ...",
        "Avoid posts that ..."
      ]
    }}
  ]
}}

There must be one result for every supplied pair_id.

PAIRS:

{json.dumps(payload, ensure_ascii=False)}
'''

    # retries=1:
    # no automatic surprise retry call.

    out = chat_json(

        'Analyze pairwise community preferences. '
        'Return complete valid JSON only.',

        prompt,

        'generation_representative',

        retries=1

    )

    new_calls_sent += 1

    saved_count, missing_ids = (
        save_valid_generation_results(
            payload,
            out
        )
    )

    all_omitted_ids.extend(
        missing_ids
    )

    if missing_ids:

        print(
            f'\nBatch {new_calls_sent}: '
            f'saved {saved_count}/{len(payload)}; '
            f'{len(missing_ids)} omitted.'
        )

# immediately block further paid calls

RUN_PAID_LLM = False

client = None

# rebuild raw_principles.csv from the checkpoint

raw = []

for record in read_jsonl(GEN):

    principles = record.get(
        'principles',
        []
    )

    for i, principle in enumerate(
        principles,
        1
    ):

        raw.append({

            'pair_id':
                str(
                    record['pair_id']
                ),

            'principle_index':
                i,

            'raw_principle':
                principle

        })

raw_df = pd.DataFrame(raw)

raw_df.to_csv(
    OUTPUT_DIR /
    'raw_principles.csv',
    index=False
)

# zero-cost cleaning / unique count

def generation_clean_principle(text):

    if pd.isna(text):
        return None

    text = re.sub(
        r'\s+',
        ' ',
        str(text)
    ).strip()

    text = re.sub(
        r'^[-*\s]+',
        '',
        text
    )

    text = re.sub(
        r'^\d+[.)]\s*',
        '',
        text
    )

    text = text.strip(
        ' "\'`'
    )

    if not text:
        return None

    low = text.lower()

    if not (
        low.startswith(
            'prefer posts that'
        )
        or
        low.startswith(
            'avoid posts that'
        )
    ):

        text = (
            'Prefer posts that '
            + text[0].lower()
            + text[1:]
        )

    text = (
        text.rstrip(' .')
        + '.'
    )

    if not (
        4
        <= len(text.split())
        <= 24
    ):

        return None

    return text

audit = raw_df.copy()

audit[
    'cleaned_principle'
] = audit[
    'raw_principle'
].map(
    generation_clean_principle
)

audit = (
    audit
    .dropna(
        subset=[
            'cleaned_principle'
        ]
    )
    .copy()
)

audit[
    'dedupe_key'
] = (
    audit[
        'cleaned_principle'
    ]
    .str.lower()
    .str.replace(
        r'[^a-z0-9 ]',
        '',
        regex=True
    )
)

unique_cleaned = (
    audit
    .drop_duplicates(
        'dedupe_key'
    )
    .copy()
)

# final summary

done_after = (
    completed_generation_ids()
)

target_completed = (
    set(
        generation_target[
            'pair_id'
        ].astype(str)
    )
    &
    done_after
)

still_missing = (
    set(
        generation_target[
            'pair_id'
        ].astype(str)
    )
    -
    done_after
)

print()
print(
    "========== STEP 1 GENERATION SUMMARY =========="
)

print(
    f"New paid calls made by this cell: "
    f"{new_calls_sent:,}"
)

print(
    f"Representative pairs completed: "
    f"{len(target_completed):,} / "
    f"{len(generation_target):,}"
)

print(
    f"Representative pairs still omitted: "
    f"{len(still_missing):,}"
)

print(
    f"Raw principles checkpointed: "
    f"{len(raw_df):,}"
)

print(
    f"Unique cleaned principles: "
    f"{len(unique_cleaned):,}"
)

print()

if len(unique_cleaned) >= N_CLUSTERS:

    print(
        f"SUCCESS: we have enough unique principles "
        f"to create {N_CLUSTERS:,} candidate clusters."
    )

    print(
        "Do NOT spend repair calls."
    )

else:

    print(
        f"NOT YET ENOUGH: need "
        f"{N_CLUSTERS - len(unique_cleaned):,} "
        f"more unique principles."
    )

    print(
        "Do NOT automatically rerun anything. "
        "We will repair only the minimum number needed."
    )

print()
print(
    "PAID LLM CALLS ARE BLOCKED AGAIN:",
    not RUN_PAID_LLM
)

print(
    "==============================================="
)


In [ ]:

# recover valid results from the failed step 1 call
# zero additional llm calls

if 'out' not in globals() or 'payload' not in globals():
    print(
        "The failed response is no longer in memory. "
        "Skip this recovery cell and use the replacement Step 1 cell below."
    )

else:
    got_recovery = {
        str(x.get('pair_id')): x
        for x in out.get('results', [])
        if x.get('pair_id') is not None
    }

    recovered_rows = []
    missing_ids = []

    for x in payload:
        pid = str(x['pair_id'])

        ps = got_recovery.get(pid, {}).get('principles', [])

        if not isinstance(ps, list):
            ps = []

        ps = [
            str(p).strip()
            for p in ps
            if str(p).strip()
        ]

        if len(ps) >= PRINCIPLES_PER_PAIR:
            recovered_rows.append({
                'pair_id': pid,
                'principles': ps[:PRINCIPLES_PER_PAIR]
            })
        else:
            missing_ids.append(pid)

    # avoid duplicates if you happen to run this recovery twice.
    already_done = {
        str(x['pair_id'])
        for x in read_jsonl(GEN)
    }

    recovered_rows = [
        r for r in recovered_rows
        if r['pair_id'] not in already_done
    ]

    if recovered_rows:
        append_jsonl(GEN, recovered_rows)

    print(f"Recovered without another API call: {len(recovered_rows):,}")
    print(f"Missing from failed batch: {len(missing_ids):,}")

    if missing_ids:
        print("Missing pair IDs:", missing_ids[:20])


## step 2 — clean the principles, save embeddings, and cluster them


In [ ]:
def clean_principle(text):
    text = clean_field(text)
    text = re.sub(r'^[-*\s]+', '', text)
    text = re.sub(r'^\d+[.)]\s*', '', text)
    text = text.strip(' "\'`')
    text = re.sub(r'\s+', ' ', text)

    if not text:
        return None

    low = text.lower()

    if not (
        low.startswith('prefer posts that')
        or low.startswith('avoid posts that')
    ):
        text = 'Prefer posts that ' + text[0].lower() + text[1:]

    text = text.rstrip(' .') + '.'

    return text if 4 <= len(text.split()) <= 24 else None

cleaned = raw_df.copy()
cleaned['cleaned_principle'] = cleaned.raw_principle.map(clean_principle)
cleaned = cleaned.dropna(subset=['cleaned_principle']).copy()

cleaned['k'] = (
    cleaned.cleaned_principle
    .str.lower()
    .str.replace(r'[^a-z0-9 ]', '', regex=True)
)

cleaned = (
    cleaned
    .drop_duplicates('k')
    .drop(columns='k')
    .reset_index(drop=True)
)

cleaned.insert(
    0,
    'cleaned_principle_id',
    [f'P{i:04d}' for i in range(1, len(cleaned) + 1)]
)

model = SentenceTransformer(EMBED_MODEL)
pair_file = OUTPUT_DIR / 'pair_post_embeddings.npz'
princ_file = OUTPUT_DIR / 'principle_embeddings.npy'

if pair_file.exists() and not FORCE_REEMBED:
    z = np.load(pair_file)
    pref_emb = z['preferred']
    non_emb = z['nonpreferred']

else:
    e = model.encode(
        df.preferred_text.tolist() + df.nonpreferred_text.tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    pref_emb = e[:len(df)]
    non_emb = e[len(df):]

    np.savez_compressed(
        pair_file,
        preferred=pref_emb,
        nonpreferred=non_emb,
        pair_ids=df.pair_id.to_numpy()
    )

if princ_file.exists() and not FORCE_REEMBED:
    princ_emb = np.load(princ_file)

    if len(princ_emb) != len(cleaned):
        princ_emb = model.encode(
            cleaned.cleaned_principle.tolist(),
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        np.save(princ_file, princ_emb)

else:
    princ_emb = model.encode(
        cleaned.cleaned_principle.tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    np.save(princ_file, princ_emb)

# this checks whether semantic distance is related to the upvote-ratio gap.
pair_distance = 1 - np.sum(pref_emb * non_emb, axis=1)
df['embedding_distance'] = pair_distance

pr, pp = safe_corr(
    pair_distance,
    df.upvote_ratio_gap,
    'pearson'
)

sr, sp = safe_corr(
    pair_distance,
    df.upvote_ratio_gap,
    'spearman'
)

pd.DataFrame([{
    'pearson_r': pr,
    'pearson_p': pp,
    'spearman_rho': sr,
    'spearman_p': sp,
    'n': len(df)
}]).to_csv(
    OUTPUT_DIR / 'pair_embedding_gap_correlation.csv',
    index=False
)

plt.figure(figsize=(7, 5))
plt.scatter(
    pair_distance,
    df.upvote_ratio_gap,
    alpha=.35,
    s=18
)
plt.xlabel('Cosine distance between pair posts')
plt.ylabel('Upvote-ratio gap')
plt.title(f'Semantic distance vs preference gap (rho={sr:.3f})')
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / 'pair_embedding_distance_vs_upvote_gap.png',
    dpi=180
)
plt.close()

# this makes sure there are enough unique principles before clustering.
if len(cleaned) < N_CLUSTERS:
    raise ValueError(
        f'Only {len(cleaned):,} unique cleaned principles were generated; '
        f'need at least {N_CLUSTERS:,} to create 5,000 candidate clusters.'
    )

k = N_CLUSTERS

clusterer = MiniBatchKMeans(
    n_clusters=k,
    random_state=SEED,
    batch_size=4096,
    n_init='auto'
)

cleaned['cluster_id'] = clusterer.fit_predict(princ_emb)
cleaned.to_csv(
    OUTPUT_DIR / 'cleaned_principles.csv',
    index=False
)

print(
    'Embeddings saved. Pair-distance/upvote-gap Spearman rho=',
    sr
)


## step 3 — pick one representative principle from each cluster


In [ ]:
rows = []

for cid, idxs0 in cleaned.groupby('cluster_id').groups.items():
    idxs = np.array(list(idxs0), int)
    center = princ_emb[idxs].mean(0)
    center /= np.linalg.norm(center) + 1e-12
    chosen = idxs[np.argmax(princ_emb[idxs] @ center)]
    r = cleaned.iloc[chosen]

    rows.append({
        'representative_id': f'C{int(cid):04d}',
        'cluster_id': int(cid),
        'cluster_size': len(idxs),
        'cleaned_principle_id': r.cleaned_principle_id,
        'cleaned_principle': r.cleaned_principle,
        'pair_id': r.pair_id,
        'raw_principle': r.raw_principle,
        'principle_row_index': int(chosen)
    })

representatives = (
    pd.DataFrame(rows)
    .sort_values('cluster_id')
    .reset_index(drop=True)
)

# this compares each representative with the preferred and nonpreferred post embeddings.
rep_emb = princ_emb[
    representatives.principle_row_index.to_numpy()
]

sim_p = rep_emb @ pref_emb.T
sim_n = rep_emb @ non_emb.T

pol = np.where(
    representatives.cleaned_principle
    .str.lower()
    .str.startswith('avoid'),
    -1.,
    1.
)[:, None]

margin = pol * (sim_p - sim_n)

representatives['embedding_accuracy'] = (margin > 0).mean(1)
representatives['embedding_mean_margin'] = margin.mean(1)
representatives['embedding_gap_spearman'] = [
    safe_corr(
        margin[i],
        df.upvote_ratio_gap,
        'spearman'
    )[0]
    for i in range(len(representatives))
]

representatives.to_csv(
    OUTPUT_DIR / 'cluster_representatives.csv',
    index=False
)

print('Representatives:', len(representatives))

if len(representatives) != N_CLUSTERS:
    raise RuntimeError(
        f'Expected {N_CLUSTERS:,} representatives, '
        f'got {len(representatives):,}.'
    )


In [ ]:

# step 2b / step 3 fix
# exact 5,000 non-empty embedding clusters

# why:
# minibatchkmeans requested 5,000 clusters but produced only
# 4,173 occupied clusters.

# we have 5,493 unique principles and need 5,000 candidates.
# therefore exactly 493 merges are required:

# 5,493 - 493 = 5,000

# we merge the most semantically similar non-overlapping pairs
# using the already-computed sentencetransformer embeddings.

# zero llm/api calls.

import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# safety

RUN_PAID_LLM = False

# basic checks

n_principles = len(cleaned)

if len(princ_emb) != n_principles:
    raise RuntimeError(
        f'Embedding count ({len(princ_emb):,}) does not match '
        f'cleaned principle count ({n_principles:,}).'
    )

if n_principles < N_CLUSTERS:
    raise RuntimeError(
        f'Only {n_principles:,} unique principles exist, '
        f'but {N_CLUSTERS:,} candidate clusters are required.'
    )

merges_needed = n_principles - N_CLUSTERS

print("========== EXACT CLUSTER PLAN ==========")
print(f"Unique cleaned principles: {n_principles:,}")
print(f"Target clusters:           {N_CLUSTERS:,}")
print(f"Semantic merges required:  {merges_needed:,}")
print(f"Singletons after merging:  {n_principles - 2 * merges_needed:,}")
print("Paid LLM calls:            0")
print("========================================")
print()

# find local semantic neighbors

# princ_emb is already normalized by sentencetransformer.
# cosine distance:
# 0   = extremely similar
# 1   = unrelated

# 50 neighbors gives us many candidate edges while remaining
# lightweight for only ~5,500 principles.
N_NEIGHBORS = min(50, n_principles)

nn = NearestNeighbors(
    n_neighbors=N_NEIGHBORS,
    metric='cosine',
    algorithm='brute'
)

nn.fit(princ_emb)

distances, neighbors = nn.kneighbors(
    princ_emb,
    return_distance=True
)

# build unique candidate pairs

edges = {}

for i in range(n_principles):

    for pos in range(1, N_NEIGHBORS):
        # position 0 is normally the point itself

        j = int(neighbors[i, pos])

        if i == j:
            continue

        a, b = sorted((i, j))

        d = float(distances[i, pos])

        key = (a, b)

        # keep the smallest distance if the same edge appears
        # from both directions.
        if key not in edges or d < edges[key]:
            edges[key] = d

candidate_edges = [
    (distance, i, j)
    for (i, j), distance in edges.items()
]

candidate_edges.sort(
    key=lambda x: (x[0], x[1], x[2])
)

print(
    f'Candidate semantic neighbor edges: '
    f'{len(candidate_edges):,}'
)

# greedily select the closest non-overlapping pairs

# a principle can participate in at most one merge.
# this guarantees every final cluster is non-empty.

used = set()

semantic_pairs = []

for distance, i, j in candidate_edges:

    if i in used or j in used:
        continue

    semantic_pairs.append(
        (i, j, distance)
    )

    used.add(i)
    used.add(j)

    if len(semantic_pairs) == merges_needed:
        break

# safety fallback.

# with 50 nearest neighbors this should not normally be needed,
# but if we somehow cannot find enough disjoint semantic pairs,
# pair remaining principles deterministically.

if len(semantic_pairs) < merges_needed:

    shortfall = (
        merges_needed
        - len(semantic_pairs)
    )

    print(
        f'Nearest-neighbor matching was short by '
        f'{shortfall:,} merges.'
    )

    remaining = [
        i
        for i in range(n_principles)
        if i not in used
    ]

    # pair remaining items in deterministic order.
    for k in range(0, len(remaining) - 1, 2):

        if len(semantic_pairs) == merges_needed:
            break

        i = remaining[k]
        j = remaining[k + 1]

        # cosine distance because embeddings are normalized
        distance = float(
            1.0 - np.dot(
                princ_emb[i],
                princ_emb[j]
            )
        )

        semantic_pairs.append(
            (i, j, distance)
        )

        used.add(i)
        used.add(j)

if len(semantic_pairs) != merges_needed:

    raise RuntimeError(
        f'Could only form {len(semantic_pairs):,} merges; '
        f'needed exactly {merges_needed:,}.'
    )

# assign exact cluster ids

# first:
# each semantic pair becomes one cluster

# then:
# every remaining principle gets its own singleton cluster

cluster_id = np.full(
    n_principles,
    -1,
    dtype=int
)

cluster_counter = 0

# paired semantic clusters

for i, j, distance in semantic_pairs:

    cluster_id[i] = cluster_counter
    cluster_id[j] = cluster_counter

    cluster_counter += 1

# singleton clusters

for i in range(n_principles):

    if cluster_id[i] == -1:

        cluster_id[i] = cluster_counter

        cluster_counter += 1

# exact-count validation

unique_cluster_ids = np.unique(
    cluster_id
)

print()
print(
    f'Actual non-empty clusters created: '
    f'{len(unique_cluster_ids):,}'
)

if len(unique_cluster_ids) != N_CLUSTERS:

    raise RuntimeError(
        f'Expected exactly {N_CLUSTERS:,} clusters, '
        f'but created {len(unique_cluster_ids):,}.'
    )

# put corrected labels back into cleaned.

cleaned = cleaned.copy()

cleaned['cluster_id'] = cluster_id

cleaned.to_csv(
    OUTPUT_DIR / 'cleaned_principles.csv',
    index=False
)

# save cluster-diagnostic information

pair_cluster_rows = []

for cid, (i, j, distance) in enumerate(
    semantic_pairs
):

    pair_cluster_rows.append({

        'cluster_id': cid,

        'principle_row_1': i,

        'principle_row_2': j,

        'principle_id_1':
            cleaned.iloc[i].cleaned_principle_id,

        'principle_id_2':
            cleaned.iloc[j].cleaned_principle_id,

        'cosine_distance':
            distance,

        'cosine_similarity':
            1.0 - distance
    })

semantic_merge_df = pd.DataFrame(
    pair_cluster_rows
)

semantic_merge_df.to_csv(
    OUTPUT_DIR /
    'exact_5000_semantic_merges.csv',
    index=False
)

# step 3 — choose one representative per cluster

# for singleton clusters:
# that principle is automatically representative.

# for two-principle semantic clusters:
# both are approximately equally central, so choose the one
# with slightly greater similarity to the normalized cluster
# centroid. ties resolve deterministically.

rows = []

for cid, idxs0 in cleaned.groupby(
    'cluster_id'
).groups.items():

    idxs = np.array(
        list(idxs0),
        dtype=int
    )

    if len(idxs) == 1:

        chosen = idxs[0]

    else:

        center = princ_emb[idxs].mean(
            axis=0
        )

        center = (
            center
            /
            (
                np.linalg.norm(center)
                + 1e-12
            )
        )

        similarity_to_center = (
            princ_emb[idxs]
            @ center
        )

        chosen = idxs[
            np.argmax(
                similarity_to_center
            )
        ]

    r = cleaned.iloc[
        chosen
    ]

    rows.append({

        'representative_id':
            f'C{int(cid):04d}',

        'cluster_id':
            int(cid),

        'cluster_size':
            len(idxs),

        'cleaned_principle_id':
            r.cleaned_principle_id,

        'cleaned_principle':
            r.cleaned_principle,

        'pair_id':
            r.pair_id,

        'raw_principle':
            r.raw_principle,

        'principle_row_index':
            int(chosen)
    })

representatives = (
    pd.DataFrame(rows)
    .sort_values('cluster_id')
    .reset_index(drop=True)
)

# recreate the same embedding diagnostics used by the notebook

rep_emb = princ_emb[
    representatives[
        'principle_row_index'
    ].to_numpy()
]

sim_p = (
    rep_emb
    @ pref_emb.T
)

sim_n = (
    rep_emb
    @ non_emb.T
)

pol = np.where(

    representatives[
        'cleaned_principle'
    ]
    .str.lower()
    .str.startswith('avoid'),

    -1.0,

    1.0

)[:, None]

margin = (
    pol
    *
    (
        sim_p
        -
        sim_n
    )
)

representatives[
    'embedding_accuracy'
] = (
    margin > 0
).mean(axis=1)

representatives[
    'embedding_mean_margin'
] = margin.mean(
    axis=1
)

representatives[
    'embedding_gap_spearman'
] = [

    safe_corr(
        margin[i],
        df.upvote_ratio_gap,
        'spearman'
    )[0]

    for i in range(
        len(representatives)
    )
]

representatives.to_csv(
    OUTPUT_DIR /
    'cluster_representatives.csv',
    index=False
)

# final validation

cluster_sizes = (
    cleaned
    .groupby('cluster_id')
    .size()
)

print()
print(
    "========== EXACT CLUSTERING COMPLETE =========="
)

print(
    f'Unique cleaned principles: '
    f'{len(cleaned):,}'
)

print(
    f'Non-empty clusters:        '
    f'{cleaned.cluster_id.nunique():,}'
)

print(
    f'Representatives:           '
    f'{len(representatives):,}'
)

print(
    f'Two-principle clusters:    '
    f'{(cluster_sizes == 2).sum():,}'
)

print(
    f'Singleton clusters:        '
    f'{(cluster_sizes == 1).sum():,}'
)

print(
    f'Largest cluster:           '
    f'{cluster_sizes.max():,}'
)

print()
print(
    'PAID LLM CALLS MADE: 0'
)

print(
    'RUN_PAID_LLM remains:',
    RUN_PAID_LLM
)

print(
    "==============================================="
)

if len(representatives) != N_CLUSTERS:

    raise RuntimeError(
        f'Expected {N_CLUSTERS:,} representatives, '
        f'got {len(representatives):,}.'
    )


In [ ]:

# step 4 preparation / cost audit
# zero paid llm calls

# purpose:
# 1. keep all 5,000 candidate principles.
# 2. build a diverse pool of 250 preference pairs from
# the full 10,000-pair training set.
# 3. give every principle its 3 most embedding-relevant
# pairs from that shared pool.
# 4. group many principles around the same post pair so
# reddit text is not repeatedly sent to the llm.
# 5. construct an exact round-1 call plan before enabling
# azure.
# 6. estimate the worst-case call count through rounds
# 3 -> 10 -> 25.

# no azure/openai calls are made by this cell.

import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans

# hard safety

RUN_PAID_LLM = False

# new step-4 cost settings

# shared diverse pair pool from the complete 10,000-pair set.
STEP4_PAIR_POOL_SIZE = 250

# round 1:
# every one of the 5,000 principles receives 3 llm judgments.
ROUND1_PAIRS_PER_PRINCIPLE = 3

# pack many judgments into one request.

# this does not mean 350 reddit pairs.
# it means up to 350 principle votes, with post text grouped
# and transmitted only once per distinct pair inside the call.
STEP4_JUDGMENTS_PER_CALL = 350

# existing staged-testing logic.
ROUND2_TOTAL_PAIRS = 10
ROUND3_TOTAL_PAIRS = 25

ROUND1_SURVIVOR_CAP = 500
ROUND2_SURVIVOR_CAP = 100

# validate the 5,000 principles

REP_FILE = OUTPUT_DIR / 'cluster_representatives.csv'

if (
    'representatives' not in globals()
    or len(representatives) != N_CLUSTERS
):

    if not REP_FILE.exists():
        raise FileNotFoundError(
            f'Missing {REP_FILE}'
        )

    representatives = pd.read_csv(
        REP_FILE,
        keep_default_na=False
    )

if len(representatives) != 5000:
    raise RuntimeError(
        f'Expected exactly 5,000 representatives; '
        f'found {len(representatives):,}.'
    )

representatives[
    'representative_id'
] = representatives[
    'representative_id'
].astype(str)

print(
    f'Validated representatives: '
    f'{len(representatives):,}'
)

# validate / reload the full 10,000 pairs

if (
    'df' not in globals()
    or len(df) != 10000
):

    DF_FILE = (
        OUTPUT_DIR /
        'initial_10000_pairs_used.csv'
    )

    if not DF_FILE.exists():
        raise FileNotFoundError(
            f'Missing {DF_FILE}'
        )

    df = pd.read_csv(
        DF_FILE,
        keep_default_na=False
    )

df = df.copy()

df['pair_id'] = (
    df['pair_id']
    .astype(str)
)

if len(df) != 10000:
    raise RuntimeError(
        f'Expected 10,000 preference pairs; '
        f'found {len(df):,}.'
    )

if df['pair_id'].duplicated().any():
    raise RuntimeError(
        'pair_id is not unique.'
    )

# rebuild text only if necessary.
if 'preferred_text' not in df.columns:

    df['preferred_text'] = [
        build_post_text(t, b)
        for t, b in zip(
            df['preferred_post_title'],
            df['preferred_post_body']
        )
    ]

if 'nonpreferred_text' not in df.columns:

    df['nonpreferred_text'] = [
        build_post_text(t, b)
        for t, b in zip(
            df['nonpreferred_post_title'],
            df['nonpreferred_post_body']
        )
    ]

print(
    f'Validated training pairs: '
    f'{len(df):,}'
)

# load the saved local embeddings

# no model download/api call should be needed.

PAIR_EMBED_FILE = (
    OUTPUT_DIR /
    'pair_post_embeddings.npz'
)

PRINCIPLE_EMBED_FILE = (
    OUTPUT_DIR /
    'principle_embeddings.npy'
)

if not PAIR_EMBED_FILE.exists():
    raise FileNotFoundError(
        f'Missing {PAIR_EMBED_FILE}'
    )

z = np.load(
    PAIR_EMBED_FILE,
    allow_pickle=True
)

pref_emb = z['preferred']
non_emb = z['nonpreferred']

if (
    len(pref_emb) != 10000
    or len(non_emb) != 10000
):
    raise RuntimeError(
        'Saved post embeddings do not match '
        'the 10,000-pair dataset.'
    )

if (
    'princ_emb' not in globals()
    or len(princ_emb) != len(cleaned)
):

    if not PRINCIPLE_EMBED_FILE.exists():
        raise FileNotFoundError(
            f'Missing {PRINCIPLE_EMBED_FILE}'
        )

    princ_emb = np.load(
        PRINCIPLE_EMBED_FILE
    )

# exact-clustering step 3 saved the source principle row
# used by each representative.
if 'principle_row_index' not in representatives.columns:
    raise RuntimeError(
        'cluster_representatives.csv is missing '
        'principle_row_index. Run the corrected exact '
        'clustering Step 3 first.'
    )

rep_indices = (
    representatives[
        'principle_row_index'
    ]
    .astype(int)
    .to_numpy()
)

rep_emb = princ_emb[
    rep_indices
]

if len(rep_emb) != 5000:
    raise RuntimeError(
        'Representative embedding count mismatch.'
    )

print(
    'Loaded saved post/principle embeddings.'
)

# helper: row normalization

def normalize_rows_step4(x):

    x = np.asarray(
        x,
        dtype=np.float32
    )

    norm = np.linalg.norm(
        x,
        axis=1,
        keepdims=True
    )

    norm[norm == 0] = 1.0

    return x / norm

# part a
# build a diverse representative pool of 250 pairs

# each pair is represented by:

# topic vector:
# preferred + nonpreferred

# contrast vector:
# preferred - nonpreferred

# this captures both what the posts discuss and how the two
# posts differ.

topic_vectors = normalize_rows_step4(
    pref_emb + non_emb
)

contrast_vectors = normalize_rows_step4(
    pref_emb - non_emb
)

pair_vectors = np.concatenate(
    [
        topic_vectors,
        contrast_vectors
    ],
    axis=1
)

pair_vectors = normalize_rows_step4(
    pair_vectors
)

print()
print(
    f'Clustering all {len(df):,} training pairs '
    f'into {STEP4_PAIR_POOL_SIZE} diversity groups...'
)

pair_clusterer = MiniBatchKMeans(

    n_clusters=STEP4_PAIR_POOL_SIZE,

    random_state=SEED,

    batch_size=2048,

    n_init='auto'
)

pair_labels = pair_clusterer.fit_predict(
    pair_vectors
)

pair_centers = (
    pair_clusterer.cluster_centers_
)

assigned_centers = (
    pair_centers[pair_labels]
)

distance_to_center = np.sum(

    (
        pair_vectors
        -
        assigned_centers
    ) ** 2,

    axis=1
)

pair_selection = pd.DataFrame({

    'df_index':
        np.arange(len(df)),

    'pair_cluster':
        pair_labels,

    'distance_to_center':
        distance_to_center

})

# one medoid-like pair per occupied cluster.
chosen_pair_indices = (

    pair_selection

    .groupby(
        'pair_cluster'
    )[
        'distance_to_center'
    ]

    .idxmin()

    .tolist()
)

chosen_df_indices = (

    pair_selection
    .loc[chosen_pair_indices, 'df_index']
    .astype(int)
    .tolist()
)

# minibatchkmeans can theoretically leave an empty cluster.

# guarantee exactly 250 pool pairs by adding diverse
# unselected outliers if needed.

if len(chosen_df_indices) < STEP4_PAIR_POOL_SIZE:

    shortfall = (
        STEP4_PAIR_POOL_SIZE
        -
        len(chosen_df_indices)
    )

    used = set(
        chosen_df_indices
    )

    fallback = (

        pair_selection[
            ~pair_selection[
                'df_index'
            ].isin(used)
        ]

        .sort_values(
            'distance_to_center',
            ascending=False
        )

        .head(shortfall)

        ['df_index']
        .astype(int)
        .tolist()
    )

    chosen_df_indices.extend(
        fallback
    )

chosen_df_indices = (
    chosen_df_indices[
        :STEP4_PAIR_POOL_SIZE
    ]
)

if len(set(chosen_df_indices)) != STEP4_PAIR_POOL_SIZE:
    raise RuntimeError(
        'Could not construct exactly '
        '250 unique Step-4 pool pairs.'
    )

pool = (
    df.iloc[chosen_df_indices]
    .copy()
    .reset_index(drop=False)
    .rename(
        columns={
            'index': 'original_df_index'
        }
    )
)

pool[
    'step4_pool_index'
] = np.arange(
    len(pool)
)

pool.to_csv(

    OUTPUT_DIR /
    'step4_representative_pair_pool.csv',

    index=False
)

print(
    f'Diverse Step-4 pair pool: '
    f'{len(pool):,}'
)

# part b
# randomize a/b positions once

# same idea as the original notebook:
# the preferred post must not always appear as a.

rng = random.Random(
    SEED + 404
)

preferred_is_a = [
    rng.choice(
        [True, False]
    )
    for _ in range(len(df))
]

pair_order = df[
    [
        'pair_id',
        'preferred_text',
        'nonpreferred_text'
    ]
].copy()

pair_order[
    'preferred_is_a'
] = preferred_is_a

pair_order[
    'post_a'
] = np.where(

    pair_order[
        'preferred_is_a'
    ],

    pair_order[
        'preferred_text'
    ],

    pair_order[
        'nonpreferred_text'
    ]
)

pair_order[
    'post_b'
] = np.where(

    pair_order[
        'preferred_is_a'
    ],

    pair_order[
        'nonpreferred_text'
    ],

    pair_order[
        'preferred_text'
    ]
)

pair_order[
    'correct_vote'
] = np.where(

    pair_order[
        'preferred_is_a'
    ],

    'A',

    'B'
)

pair_order.to_csv(

    OUTPUT_DIR /
    'step4_pair_order.csv',

    index=False
)

# fast pair-id lookup.
pair_order_lookup = (
    pair_order
    .set_index('pair_id')
)

# part c
# find each principle's 3 most relevant pairs
# within the shared 250-pair pool

# this mirrors the original embedding relevance idea:

# abs(
# similarity(principle, preferred)

# similarity(principle, nonpreferred)
# )

# higher magnitude means the principle strongly distinguishes
# the two posts.

pool_original_indices = (
    pool[
        'original_df_index'
    ]
    .astype(int)
    .to_numpy()
)

pool_pref_emb = pref_emb[
    pool_original_indices
]

pool_non_emb = non_emb[
    pool_original_indices
]

# 5000 x 250 matrix -- small enough to calculate locally.
raw_margin_pool = (

    rep_emb
    @ pool_pref_emb.T

    -

    rep_emb
    @ pool_non_emb.T
)

# polarity only affects expected direction, not relevance
# magnitude, but save signed margin as a diagnostic.
polarity = np.where(

    representatives[
        'cleaned_principle'
    ]
    .str.lower()
    .str.startswith('avoid'),

    -1.0,

    1.0

)[:, None]

signed_margin_pool = (
    polarity
    *
    raw_margin_pool
)

relevance_pool = np.abs(
    signed_margin_pool
)

# top 3 relevant pool pairs per principle

k = ROUND1_PAIRS_PER_PRINCIPLE

top_unsorted = np.argpartition(

    relevance_pool,

    -k,

    axis=1

)[:, -k:]

top_scores_unsorted = np.take_along_axis(

    relevance_pool,

    top_unsorted,

    axis=1
)

top_order = np.argsort(

    -top_scores_unsorted,

    axis=1
)

top_pool_positions = np.take_along_axis(

    top_unsorted,

    top_order,

    axis=1
)

# part d
# build the 15,000 round-1 judgments

# 5,000 principles x 3 preference pairs = 15,000.

assignment_rows = []

for principle_i, rep in (
    representatives
    .reset_index(drop=True)
    .iterrows()
):

    rid = str(
        rep['representative_id']
    )

    principle = str(
        rep['cleaned_principle']
    )

    for relevance_rank, pool_pos in enumerate(

        top_pool_positions[
            principle_i
        ],

        start=1
    ):

        pool_pos = int(
            pool_pos
        )

        pair_id = str(
            pool.iloc[
                pool_pos
            ]['pair_id']
        )

        q = pair_order_lookup.loc[
            pair_id
        ]

        assignment_rows.append({

            'representative_id':
                rid,

            'principle':
                principle,

            'pair_id':
                pair_id,

            'relevance_rank':
                relevance_rank,

            'embedding_relevance':
                float(
                    relevance_pool[
                        principle_i,
                        pool_pos
                    ]
                ),

            'signed_embedding_margin':
                float(
                    signed_margin_pool[
                        principle_i,
                        pool_pos
                    ]
                ),

            'post_a':
                q['post_a'],

            'post_b':
                q['post_b'],

            'correct_vote':
                q['correct_vote']

        })

round1_assignments = pd.DataFrame(
    assignment_rows
)

expected_judgments = (
    N_CLUSTERS
    *
    ROUND1_PAIRS_PER_PRINCIPLE
)

if len(round1_assignments) != expected_judgments:

    raise RuntimeError(
        f'Expected {expected_judgments:,} '
        f'Round-1 judgments; '
        f'created {len(round1_assignments):,}.'
    )

duplicate_tasks = (
    round1_assignments
    .duplicated(
        [
            'representative_id',
            'pair_id'
        ]
    )
    .sum()
)

if duplicate_tasks:
    raise RuntimeError(
        f'Found {duplicate_tasks:,} '
        'duplicate principle/pair tasks.'
    )

round1_assignments.to_csv(

    OUTPUT_DIR /
    'step4_round1_assignments.csv',

    index=False
)

# part e
# pack round 1 into shared calls

# sort by pair first.

# therefore many principles referring to the same post pair
# remain together, and post a/b text only needs to be included
# once for that pair inside the eventual prompt.

round1_plan = (

    round1_assignments

    .sort_values(
        [
            'pair_id',
            'embedding_relevance',
            'representative_id'
        ],

        ascending=[
            True,
            False,
            True
        ]
    )

    .reset_index(drop=True)
)

round1_plan[
    'planned_call_id'
] = (

    np.arange(
        len(round1_plan)
    )

    //
    STEP4_JUDGMENTS_PER_CALL

    +
    1
)

round1_plan.to_csv(

    OUTPUT_DIR /
    'step4_round1_call_plan.csv',

    index=False
)

round1_calls = int(
    round1_plan[
        'planned_call_id'
    ].max()
)

# part f
# worst-case later-round call plan

# round 1:
# 5,000 x 3

# round 2:
# max 500 survivors
# receive 7 additional judgments
# so each reaches 10 total.

# round 3:
# max 100 survivors
# receive 15 additional judgments
# so each reaches 25 total.

round1_judgments = (
    N_CLUSTERS
    *
    ROUND1_PAIRS_PER_PRINCIPLE
)

round2_additional_judgments = (

    ROUND1_SURVIVOR_CAP

    *
    (
        ROUND2_TOTAL_PAIRS
        -
        ROUND1_PAIRS_PER_PRINCIPLE
    )
)

round3_additional_judgments = (

    ROUND2_SURVIVOR_CAP

    *
    (
        ROUND3_TOTAL_PAIRS
        -
        ROUND2_TOTAL_PAIRS
    )
)

round2_calls_worst = math.ceil(

    round2_additional_judgments

    /
    STEP4_JUDGMENTS_PER_CALL
)

round3_calls_worst = math.ceil(

    round3_additional_judgments

    /
    STEP4_JUDGMENTS_PER_CALL
)

total_test_calls_worst = (

    round1_calls
    +
    round2_calls_worst
    +
    round3_calls_worst
)

# part g
# audit how much post text is reused

call_audit_rows = []

for call_id, g in (
    round1_plan
    .groupby('planned_call_id')
):

    distinct_pairs = (
        g['pair_id']
        .nunique()
    )

    judgments = len(g)

    # approximate prompt characters.

    # pair text counted once for each distinct pair.
    # principle text counted once for each judgment.
    pair_chars = 0

    for pair_id in (
        g['pair_id']
        .drop_duplicates()
    ):

        q = pair_order_lookup.loc[
            str(pair_id)
        ]

        pair_chars += (
            len(str(q['post_a']))
            +
            len(str(q['post_b']))
        )

    principle_chars = (
        g['principle']
        .astype(str)
        .str.len()
        .sum()
    )

    approximate_prompt_chars = (
        pair_chars
        +
        principle_chars
    )

    # very rough tokenizer-independent diagnostic only.
    rough_tokens = (
        approximate_prompt_chars
        / 4.0
    )

    call_audit_rows.append({

        'planned_call_id':
            int(call_id),

        'judgments':
            int(judgments),

        'distinct_pairs':
            int(distinct_pairs),

        'pair_text_chars_once':
            int(pair_chars),

        'principle_chars':
            int(principle_chars),

        'approx_prompt_chars':
            int(
                approximate_prompt_chars
            ),

        'very_rough_prompt_tokens':
            float(rough_tokens)

    })

call_audit = pd.DataFrame(
    call_audit_rows
)

call_audit.to_csv(

    OUTPUT_DIR /
    'step4_round1_call_audit.csv',

    index=False
)

# part h
# save cost plan

cost_plan = pd.DataFrame([{

    'candidate_principles':
        N_CLUSTERS,

    'shared_pair_pool':
        STEP4_PAIR_POOL_SIZE,

    'round1_pairs_per_principle':
        ROUND1_PAIRS_PER_PRINCIPLE,

    'round1_judgments':
        round1_judgments,

    'judgments_per_call_cap':
        STEP4_JUDGMENTS_PER_CALL,

    'round1_calls':
        round1_calls,

    'round1_survivor_cap':
        ROUND1_SURVIVOR_CAP,

    'round2_total_pairs_per_survivor':
        ROUND2_TOTAL_PAIRS,

    'round2_additional_judgments':
        round2_additional_judgments,

    'round2_worst_case_calls':
        round2_calls_worst,

    'round2_survivor_cap':
        ROUND2_SURVIVOR_CAP,

    'round3_total_pairs_per_survivor':
        ROUND3_TOTAL_PAIRS,

    'round3_additional_judgments':
        round3_additional_judgments,

    'round3_worst_case_calls':
        round3_calls_worst,

    'worst_case_step4_calls':
        total_test_calls_worst

}])

cost_plan.to_csv(

    OUTPUT_DIR /
    'step4_cost_plan.csv',

    index=False
)

# final zero-cost summary

pair_assignment_counts = (

    round1_assignments

    .groupby('pair_id')

    .size()
)

principle_assignment_counts = (

    round1_assignments

    .groupby('representative_id')

    .size()
)

print()
print(
    "========== STEP 4 ZERO-COST AUDIT =========="
)

print(
    f'Candidate principles:             '
    f'{N_CLUSTERS:,}'
)

print(
    f'Diverse shared pair pool:         '
    f'{len(pool):,}'
)

print(
    f'Round-1 pairs per principle:      '
    f'{ROUND1_PAIRS_PER_PRINCIPLE}'
)

print(
    f'Round-1 LLM judgments:            '
    f'{round1_judgments:,}'
)

print(
    f'Judgments allowed per call:       '
    f'{STEP4_JUDGMENTS_PER_CALL:,}'
)

print()
print(
    f'ROUND 1 exact planned calls:      '
    f'{round1_calls:,}'
)

print(
    f'ROUND 2 worst-case calls:         '
    f'{round2_calls_worst:,}'
)

print(
    f'ROUND 3 worst-case calls:         '
    f'{round3_calls_worst:,}'
)

print(
    f'WORST-CASE STEP-4 CALLS:          '
    f'{total_test_calls_worst:,}'
)

print()
print(
    'Pair reuse within Round 1:'
)

print(
    f'  unique pool pairs actually used: '
    f'{round1_assignments.pair_id.nunique():,}'
)

print(
    f'  mean principles per used pair:   '
    f'{pair_assignment_counts.mean():.1f}'
)

print(
    f'  maximum principles on one pair:  '
    f'{pair_assignment_counts.max():,}'
)

print()
print(
    'Principle coverage check:'
)

print(
    f'  minimum judgments/principle:     '
    f'{principle_assignment_counts.min():,}'
)

print(
    f'  maximum judgments/principle:     '
    f'{principle_assignment_counts.max():,}'
)

print()
print(
    'Approximate Round-1 prompt-size diagnostic:'
)

print(
    f'  mean rough tokens/call:          '
    f'{call_audit.very_rough_prompt_tokens.mean():,.0f}'
)

print(
    f'  max rough tokens/call:           '
    f'{call_audit.very_rough_prompt_tokens.max():,.0f}'
)

print()
print(
    'Files saved:'
)

print(
    '  step4_representative_pair_pool.csv'
)

print(
    '  step4_pair_order.csv'
)

print(
    '  step4_round1_assignments.csv'
)

print(
    '  step4_round1_call_plan.csv'
)

print(
    '  step4_round1_call_audit.csv'
)

print(
    '  step4_cost_plan.csv'
)

print()
print(
    'PAID LLM CALLS MADE BY THIS CELL: 0'
)

print(
    'RUN_PAID_LLM remains:',
    RUN_PAID_LLM
)

print(
    "============================================="
)


In [ ]:

# step 4 final cost replan
# zero paid llm calls

# increase packing from 350 -> 500 very compact votes/call.

# nothing is sent to azure here.

import math
import numpy as np
import pandas as pd

RUN_PAID_LLM = False

STEP4_JUDGMENTS_PER_CALL = 500

ROUND1_PAIRS_PER_PRINCIPLE = 3
ROUND2_TOTAL_PAIRS = 10
ROUND3_TOTAL_PAIRS = 25

ROUND1_SURVIVOR_CAP = 500
ROUND2_SURVIVOR_CAP = 100

# reload the already-created round-1 assignments if needed.

ROUND1_ASSIGNMENT_FILE = (
    OUTPUT_DIR /
    'step4_round1_assignments.csv'
)

if 'round1_assignments' not in globals():

    if not ROUND1_ASSIGNMENT_FILE.exists():
        raise FileNotFoundError(
            f'Missing {ROUND1_ASSIGNMENT_FILE}'
        )

    round1_assignments = pd.read_csv(
        ROUND1_ASSIGNMENT_FILE,
        keep_default_na=False
    )

# sort by pair so identical reddit text stays together.

round1_plan_500 = (

    round1_assignments

    .sort_values(
        [
            'pair_id',
            'embedding_relevance',
            'representative_id'
        ],
        ascending=[
            True,
            False,
            True
        ]
    )

    .reset_index(drop=True)
)

round1_plan_500[
    'planned_call_id'
] = (

    np.arange(
        len(round1_plan_500)
    )

    //
    STEP4_JUDGMENTS_PER_CALL

    +
    1
)

round1_calls = int(
    round1_plan_500[
        'planned_call_id'
    ].max()
)

round2_additional_judgments = (

    ROUND1_SURVIVOR_CAP
    *
    (
        ROUND2_TOTAL_PAIRS
        -
        ROUND1_PAIRS_PER_PRINCIPLE
    )
)

round3_additional_judgments = (

    ROUND2_SURVIVOR_CAP
    *
    (
        ROUND3_TOTAL_PAIRS
        -
        ROUND2_TOTAL_PAIRS
    )
)

round2_calls = math.ceil(
    round2_additional_judgments
    /
    STEP4_JUDGMENTS_PER_CALL
)

round3_calls = math.ceil(
    round3_additional_judgments
    /
    STEP4_JUDGMENTS_PER_CALL
)

worst_case_calls = (
    round1_calls
    +
    round2_calls
    +
    round3_calls
)

# prompt-size audit with 500-judgment packing

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order[
    'pair_id'
] = pair_order[
    'pair_id'
].astype(str)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

audit_rows = []

for call_id, g in (
    round1_plan_500
    .groupby('planned_call_id')
):

    pair_chars = 0

    for pair_id in (
        g['pair_id']
        .astype(str)
        .drop_duplicates()
    ):

        q = pair_lookup.loc[
            pair_id
        ]

        pair_chars += (
            len(str(q['post_a']))
            +
            len(str(q['post_b']))
        )

    principle_chars = (
        g['principle']
        .astype(str)
        .str.len()
        .sum()
    )

    total_chars = (
        pair_chars
        +
        principle_chars
    )

    audit_rows.append({

        'planned_call_id':
            int(call_id),

        'judgments':
            len(g),

        'distinct_pairs':
            g['pair_id'].nunique(),

        'approx_prompt_chars':
            total_chars,

        'very_rough_prompt_tokens':
            total_chars / 4.0

    })

audit_500 = pd.DataFrame(
    audit_rows
)

round1_plan_500.to_csv(

    OUTPUT_DIR /
    'step4_round1_call_plan_500.csv',

    index=False
)

audit_500.to_csv(

    OUTPUT_DIR /
    'step4_round1_call_audit_500.csv',

    index=False
)

print(
    "========== STEP 4 FINAL COST PLAN =========="
)

print(
    f'Round-1 judgments:             '
    f'{len(round1_plan_500):,}'
)

print(
    f'Judgments/call:                '
    f'{STEP4_JUDGMENTS_PER_CALL:,}'
)

print()

print(
    f'ROUND 1 planned calls:         '
    f'{round1_calls:,}'
)

print(
    f'ROUND 2 worst-case calls:      '
    f'{round2_calls:,}'
)

print(
    f'ROUND 3 worst-case calls:      '
    f'{round3_calls:,}'
)

print(
    f'WORST-CASE STEP-4 CALLS:       '
    f'{worst_case_calls:,}'
)

print()

print(
    f'Mean rough prompt tokens/call: '
    f'{audit_500.very_rough_prompt_tokens.mean():,.0f}'
)

print(
    f'Max rough prompt tokens/call:  '
    f'{audit_500.very_rough_prompt_tokens.max():,.0f}'
)

print()

print(
    'PAID LLM CALLS MADE: 0'
)

print(
    'RUN_PAID_LLM remains:',
    RUN_PAID_LLM
)

print(
    "============================================"
)


In [ ]:

# step 4 round 1 — one-call paid smoke test

# purpose:
# test the new 500-judgment shared-pair format on exactly
# one planned call before committing to all 30 calls.

# safety:
# - maximum: 1 paid call
# - retries=1
# - saves every valid returned vote
# - does not treat omitted votes as "none"
# - refuses to pay for the same planned call twice
# - paid calls are disabled again in a finally block

import getpass
import json
from collections import defaultdict

import pandas as pd
from openai import AzureOpenAI

# files

PLAN_FILE = (
    OUTPUT_DIR /
    'step4_round1_call_plan_500.csv'
)

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

ROUND1_CKPT = (
    OUTPUT_DIR /
    'step4_round1_shared_checkpoint.jsonl'
)

# start blocked

RUN_PAID_LLM = False

# load the exact 30-call plan

plan = pd.read_csv(
    PLAN_FILE,
    keep_default_na=False
)

plan['pair_id'] = (
    plan['pair_id']
    .astype(str)
)

plan['representative_id'] = (
    plan['representative_id']
    .astype(str)
)

if len(plan) != 15000:
    raise RuntimeError(
        f'Expected 15,000 planned judgments; '
        f'found {len(plan):,}.'
    )

if plan['planned_call_id'].nunique() != 30:
    raise RuntimeError(
        f'Expected 30 planned calls; found '
        f'{plan["planned_call_id"].nunique():,}.'
    )

# load a/b order

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order['pair_id'] = (
    pair_order['pair_id']
    .astype(str)
)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

# choose exactly call 1

SMOKE_CALL_ID = 1

stage_name = (
    f'step4_round1_call_'
    f'{SMOKE_CALL_ID:03d}'
)

call_df = (
    plan[
        plan['planned_call_id']
        == SMOKE_CALL_ID
    ]
    .copy()
)

if len(call_df) != 500:
    raise RuntimeError(
        f'Smoke-test call should contain 500 judgments; '
        f'found {len(call_df):,}.'
    )

# duplicate-spend protection

# chat_json writes the stage to call_log after a successful
# api response. therefore, if this stage already appears there,
# we will not pay for it again automatically.

call_log_before = read_jsonl(
    CALL_LOG
)

already_logged_stages = {
    str(x.get('stage', ''))
    for x in call_log_before
}

saved_call_ids = {
    int(x.get('planned_call_id'))
    for x in read_jsonl(ROUND1_CKPT)
    if x.get('planned_call_id') is not None
}

if SMOKE_CALL_ID in saved_call_ids:

    raise RuntimeError(
        'Smoke-test call 1 is already safely checkpointed. '
        'No paid call was made.'
    )

if stage_name in already_logged_stages:

    raise RuntimeError(
        f'{stage_name} already appears in the paid-call log '
        'but is not safely checkpointed. '
        'It will NOT be called again automatically.'
    )

# build shared-pair payload

# the reddit post text appears once per distinct pair.
# each pair then carries the principles that should be judged
# against that pair.

pair_payload = []

for pair_id, g in (
    call_df.groupby(
        'pair_id',
        sort=False
    )
):

    pair_id = str(
        pair_id
    )

    q = pair_lookup.loc[
        pair_id
    ]

    principles_for_pair = [

        {
            'id':
                str(r['representative_id']),

            'principle':
                str(r['principle'])
        }

        for _, r
        in g.iterrows()

    ]

    pair_payload.append({

        'pair_id':
            pair_id,

        'post_a':
            str(q['post_a']),

        'post_b':
            str(q['post_b']),

        'principles':
            principles_for_pair

    })

print(
    "========== SMOKE TEST PREVIEW =========="
)

print(
    f'Planned call ID:        '
    f'{SMOKE_CALL_ID}'
)

print(
    f'Judgments:              '
    f'{len(call_df):,}'
)

print(
    f'Distinct Reddit pairs:  '
    f'{call_df.pair_id.nunique():,}'
)

print(
    f'Paid calls to be made:  1'
)

print(
    'Automatic retries:     0'
)

print(
    "========================================"
)

# prompt

system_prompt = """
Apply each explicit principle independently to its assigned
pair of Reddit posts.

For every principle:
- Vote "A" if Post A better satisfies that principle.
- Vote "B" if Post B better satisfies that principle.
- Vote "None" if the principle is not applicable, does not
  meaningfully distinguish the posts, or neither post is
  clearly favored by that principle.

Judge ONLY by the supplied principle.
Do not judge overall post quality.
Do not use popularity, scores, votes, or outside information.
Treat every principle independently.

Return valid JSON only.
""".strip()

user_prompt = f"""
Evaluate EVERY supplied principle.

You must return every supplied pair_id and every supplied
principle ID exactly once.

Return this compact JSON structure:

{{
  "results": [
    {{
      "pair_id": "PAIR_ID",
      "votes": {{
        "C0000": "A",
        "C0001": "B",
        "C0002": "None"
      }}
    }}
  ]
}}

The only permitted vote strings are:
"A", "B", or "None".

DATA:

{json.dumps(pair_payload, ensure_ascii=False)}
""".strip()

# enable paid call for this one request only

subscription_key = getpass.getpass(
    'Azure OpenAI API key: '
)

client = AzureOpenAI(
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=subscription_key
)

RUN_PAID_LLM = True

try:

    # one and only one paid request

    result = chat_json(

        system_prompt,

        user_prompt,

        stage_name,

        max_tokens=12000,

        retries=1

    )

    # normalize output

    # important:
    # missing results remain missing.
    # we do not silently convert missing output into "none".

    expected_by_pair = defaultdict(
        set
    )

    for _, r in call_df.iterrows():

        expected_by_pair[
            str(r['pair_id'])
        ].add(
            str(r['representative_id'])
        )

    returned_votes = defaultdict(
        dict
    )

    for item in result.get(
        'results',
        []
    ):

        pid = str(
            item.get(
                'pair_id',
                ''
            )
        )

        if pid not in expected_by_pair:
            continue

        votes = item.get(
            'votes',
            {}
        )

        if not isinstance(
            votes,
            dict
        ):
            continue

        for rid, raw_vote in votes.items():

            rid = str(
                rid
            )

            if rid not in expected_by_pair[
                pid
            ]:
                continue

            # do not interpret json null as "none".
            if not isinstance(
                raw_vote,
                str
            ):
                continue

            v = (
                raw_vote
                .strip()
                .upper()
            )

            if v == 'A':

                returned_votes[
                    pid
                ][rid] = 'A'

            elif v == 'B':

                returned_votes[
                    pid
                ][rid] = 'B'

            elif v == 'NONE':

                returned_votes[
                    pid
                ][rid] = 'None'

    # build valid vote rows

    valid_rows = []

    missing_tasks = []

    for _, r in call_df.iterrows():

        pid = str(
            r['pair_id']
        )

        rid = str(
            r['representative_id']
        )

        vote = (
            returned_votes
            .get(
                pid,
                {}
            )
            .get(
                rid
            )
        )

        if vote is None:

            missing_tasks.append(
                f'{rid}|{pid}'
            )

            continue

        correct_vote = str(
            pair_lookup.loc[
                pid,
                'correct_vote'
            ]
        )

        outcome = (

            'not_applicable'

            if vote == 'None'

            else (

                'correct'

                if vote
                == correct_vote

                else 'incorrect'

            )
        )

        valid_rows.append({

            'planned_call_id':
                SMOKE_CALL_ID,

            'representative_id':
                rid,

            'pair_id':
                pid,

            'vote':
                vote,

            'correct_vote':
                correct_vote,

            'outcome':
                outcome

        })

    # save the paid response immediately

    checkpoint_record = {

        'planned_call_id':
            SMOKE_CALL_ID,

        'stage':
            stage_name,

        'expected_judgments':
            int(
                len(call_df)
            ),

        'received_judgments':
            int(
                len(valid_rows)
            ),

        'missing_judgments':
            int(
                len(missing_tasks)
            ),

        'votes':
            valid_rows,

        'missing_task_ids':
            missing_tasks

    }

    append_jsonl(
        ROUND1_CKPT,
        [
            checkpoint_record
        ]
    )

finally:

    # critical safety:
    # block paid calls again even if something fails

    RUN_PAID_LLM = False

    client = None

# read actual token usage from the notebook's call log

stage_log = [

    x

    for x in read_jsonl(
        CALL_LOG
    )

    if str(
        x.get(
            'stage',
            ''
        )
    ) == stage_name

]

if not stage_log:

    raise RuntimeError(
        'The smoke-test call completed but no '
        'token-usage log entry was found.'
    )

usage = stage_log[-1]

prompt_tokens = (
    usage.get(
        'prompt_tokens'
    )
    or 0
)

completion_tokens = (
    usage.get(
        'completion_tokens'
    )
    or 0
)

total_tokens = (
    usage.get(
        'total_tokens'
    )
    or 0
)

# actual 30-call token projection

# this is only a rough extrapolation from call 1, but it is
# much better than the prior character-count approximation.

projected_round1_prompt = (
    prompt_tokens
    *
    30
)

projected_round1_completion = (
    completion_tokens
    *
    30
)

projected_round1_total = (
    total_tokens
    *
    30
)

# smoke test quality / coverage

valid_df = pd.DataFrame(
    valid_rows
)

if len(valid_df):

    vote_counts = (
        valid_df[
            'vote'
        ]
        .value_counts()
        .to_dict()
    )

else:

    vote_counts = {}

coverage = (
    len(valid_rows)
    /
    len(call_df)
)

print()
print(
    "========== STEP 4 SMOKE TEST RESULT =========="
)

print(
    f'Expected judgments:       '
    f'{len(call_df):,}'
)

print(
    f'Valid judgments returned: '
    f'{len(valid_rows):,}'
)

print(
    f'Missing judgments:        '
    f'{len(missing_tasks):,}'
)

print(
    f'Coverage:                 '
    f'{coverage:.1%}'
)

print()

print(
    'Vote counts:'
)

print(
    f'  A:    '
    f'{vote_counts.get("A", 0):,}'
)

print(
    f'  B:    '
    f'{vote_counts.get("B", 0):,}'
)

print(
    f'  None: '
    f'{vote_counts.get("None", 0):,}'
)

print()

print(
    'ACTUAL Azure token usage for this call:'
)

print(
    f'  Prompt tokens:     '
    f'{prompt_tokens:,}'
)

print(
    f'  Completion tokens: '
    f'{completion_tokens:,}'
)

print(
    f'  Total tokens:      '
    f'{total_tokens:,}'
)

print()

print(
    'Very rough extrapolation if all 30 calls '
    'were similar:'
)

print(
    f'  Round-1 prompt tokens:     '
    f'{projected_round1_prompt:,}'
)

print(
    f'  Round-1 completion tokens: '
    f'{projected_round1_completion:,}'
)

print(
    f'  Round-1 total tokens:      '
    f'{projected_round1_total:,}'
)

print()

print(
    'Checkpoint:',
    ROUND1_CKPT
)

print()

print(
    'PAID CALLS MADE BY THIS CELL: 1'
)

print(
    'PAID LLM CALLS ARE BLOCKED AGAIN:',
    not RUN_PAID_LLM
)

print(
    "==============================================="
)


In [ ]:

# step 4 round 1 — replan after 500-judgment smoke test

# zero paid llm calls

# keeps:
# - all 480 valid judgments from the smoke test

# requeues:
# - the 20 omitted smoke-test judgments
# - every other untested round-1 judgment

# new safer batch size:
# 350 judgments per call

import math
import json
import numpy as np
import pandas as pd

# safety

RUN_PAID_LLM = False

# files

ASSIGNMENT_FILE = (
    OUTPUT_DIR /
    'step4_round1_assignments.csv'
)

ROUND1_CKPT = (
    OUTPUT_DIR /
    'step4_round1_shared_checkpoint.jsonl'
)

REPLAN_FILE = (
    OUTPUT_DIR /
    'step4_round1_remaining_plan_350.csv'
)

REPLAN_AUDIT_FILE = (
    OUTPUT_DIR /
    'step4_round1_remaining_plan_350_audit.csv'
)

# load all 15,000 required round-1 tasks

assignments = pd.read_csv(
    ASSIGNMENT_FILE,
    keep_default_na=False
)

assignments['pair_id'] = (
    assignments['pair_id']
    .astype(str)
)

assignments['representative_id'] = (
    assignments['representative_id']
    .astype(str)
)

if len(assignments) != 15000:
    raise RuntimeError(
        f'Expected 15,000 Round-1 assignments; '
        f'found {len(assignments):,}.'
    )

# unique task identifier.
assignments['task_id'] = (
    assignments['representative_id']
    + '|'
    + assignments['pair_id']
)

if assignments['task_id'].duplicated().any():
    raise RuntimeError(
        'Duplicate principle/pair tasks exist '
        'in Round-1 assignments.'
    )

# read already-paid valid judgments

checkpoint_records = read_jsonl(
    ROUND1_CKPT
)

completed_rows = []

for record in checkpoint_records:

    for vote_row in record.get(
        'votes',
        []
    ):

        rid = str(
            vote_row.get(
                'representative_id',
                ''
            )
        )

        pid = str(
            vote_row.get(
                'pair_id',
                ''
            )
        )

        vote = str(
            vote_row.get(
                'vote',
                ''
            )
        )

        if (
            rid
            and pid
            and vote in {
                'A',
                'B',
                'None'
            }
        ):

            completed_rows.append({

                'representative_id':
                    rid,

                'pair_id':
                    pid,

                'task_id':
                    rid + '|' + pid,

                'vote':
                    vote

            })

completed_df = pd.DataFrame(
    completed_rows
)

if len(completed_df):

    completed_df = (
        completed_df
        .drop_duplicates(
            'task_id'
        )
        .copy()
    )

    completed_task_ids = set(
        completed_df[
            'task_id'
        ]
    )

else:

    completed_task_ids = set()

# keep only unfinished tasks

remaining = (

    assignments[
        ~assignments[
            'task_id'
        ].isin(
            completed_task_ids
        )
    ]

    .copy()

)

print(
    "========== EXISTING PAID WORK =========="
)

print(
    f'Round-1 tasks required:       '
    f'{len(assignments):,}'
)

print(
    f'Valid tasks already saved:    '
    f'{len(completed_task_ids):,}'
)

print(
    f'Tasks still required:         '
    f'{len(remaining):,}'
)

print(
    "========================================"
)

# safer new batch size

NEW_JUDGMENTS_PER_CALL = 350

# sort by pair first.

# this keeps shared reddit text together as much as possible.

remaining = (

    remaining

    .sort_values(
        [
            'pair_id',
            'embedding_relevance',
            'representative_id'
        ],

        ascending=[
            True,
            False,
            True
        ]
    )

    .reset_index(drop=True)

)

remaining[
    'planned_call_id_350'
] = (

    np.arange(
        len(remaining)
    )

    //
    NEW_JUDGMENTS_PER_CALL

    +
    1

)

remaining_calls = int(
    remaining[
        'planned_call_id_350'
    ].max()
)

# prompt-size audit

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order['pair_id'] = (
    pair_order['pair_id']
    .astype(str)
)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

audit_rows = []

for call_id, g in (
    remaining
    .groupby(
        'planned_call_id_350'
    )
):

    pair_chars = 0

    for pair_id in (
        g['pair_id']
        .drop_duplicates()
    ):

        q = pair_lookup.loc[
            str(pair_id)
        ]

        pair_chars += (
            len(
                str(
                    q['post_a']
                )
            )
            +
            len(
                str(
                    q['post_b']
                )
            )
        )

    principle_chars = (
        g['principle']
        .astype(str)
        .str.len()
        .sum()
    )

    total_chars = (
        pair_chars
        +
        principle_chars
    )

    audit_rows.append({

        'planned_call_id_350':
            int(call_id),

        'judgments':
            int(len(g)),

        'distinct_pairs':
            int(
                g[
                    'pair_id'
                ].nunique()
            ),

        'approx_prompt_chars':
            int(
                total_chars
            ),

        'very_rough_prompt_tokens':
            float(
                total_chars / 4.0
            )

    })

audit = pd.DataFrame(
    audit_rows
)

# save new plan

remaining.to_csv(
    REPLAN_FILE,
    index=False
)

audit.to_csv(
    REPLAN_AUDIT_FILE,
    index=False
)

# principle coverage audit

# some principles already have one of their 3 judgments
# completed in the smoke test; others have none.
# that is fine.

completed_counts = (
    assignments[
        assignments[
            'task_id'
        ].isin(
            completed_task_ids
        )
    ]

    .groupby(
        'representative_id'
    )

    .size()
)

remaining_counts = (
    remaining

    .groupby(
        'representative_id'
    )

    .size()
)

coverage_rows = []

for rid in (
    assignments[
        'representative_id'
    ]
    .unique()
):

    done = int(
        completed_counts.get(
            rid,
            0
        )
    )

    left = int(
        remaining_counts.get(
            rid,
            0
        )
    )

    coverage_rows.append({

        'representative_id':
            rid,

        'already_complete':
            done,

        'still_needed':
            left,

        'eventual_total':
            done + left

    })

coverage_df = pd.DataFrame(
    coverage_rows
)

if not (
    coverage_df[
        'eventual_total'
    ] == 3
).all():

    raise RuntimeError(
        'Replanned coverage does not preserve '
        'exactly 3 Round-1 judgments per principle.'
    )

coverage_df.to_csv(

    OUTPUT_DIR /
    'step4_round1_replan_coverage.csv',

    index=False
)

# final summary

print()
print(
    "========== ROUND-1 350-BATCH REPLAN =========="
)

print(
    f'Valid paid judgments retained: '
    f'{len(completed_task_ids):,}'
)

print(
    f'Judgments still needed:        '
    f'{len(remaining):,}'
)

print(
    f'New judgments/call:            '
    f'{NEW_JUDGMENTS_PER_CALL:,}'
)

print(
    f'New remaining calls:           '
    f'{remaining_calls:,}'
)

print()

print(
    f'Mean rough prompt tokens/call: '
    f'{audit.very_rough_prompt_tokens.mean():,.0f}'
)

print(
    f'Max rough prompt tokens/call:  '
    f'{audit.very_rough_prompt_tokens.max():,.0f}'
)

print()

print(
    'Coverage after completing this plan:'
)

print(
    f'  minimum judgments/principle: '
    f'{coverage_df.eventual_total.min()}'
)

print(
    f'  maximum judgments/principle: '
    f'{coverage_df.eventual_total.max()}'
)

print()

print(
    'PAID LLM CALLS MADE: 0'
)

print(
    'RUN_PAID_LLM remains:',
    RUN_PAID_LLM
)

print(
    "=============================================="
)


In [ ]:

# step 4 round 1 — 350-judgment paid smoke test

# makes exactly one paid call from the new 350-task plan.

# safety:
# - maximum 1 paid request
# - retries=1 (no automatic second paid request)
# - keeps the previous 480 valid judgments
# - checkpoints all valid votes from this call
# - omitted votes remain missing, not converted to "none"
# - run_paid_llm=false again afterward

import getpass
import json
from collections import defaultdict

import pandas as pd
from openai import AzureOpenAI

# files

REPLAN_FILE = (
    OUTPUT_DIR /
    'step4_round1_remaining_plan_350.csv'
)

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

ROUND1_CKPT = (
    OUTPUT_DIR /
    'step4_round1_shared_checkpoint.jsonl'
)

# begin with paid calls blocked

RUN_PAID_LLM = False

# load the 350-judgment plan

plan350 = pd.read_csv(
    REPLAN_FILE,
    keep_default_na=False
)

plan350['pair_id'] = (
    plan350['pair_id']
    .astype(str)
)

plan350['representative_id'] = (
    plan350['representative_id']
    .astype(str)
)

if len(plan350) != 14520:
    raise RuntimeError(
        f'Expected 14,520 remaining judgments; '
        f'found {len(plan350):,}.'
    )

if plan350['planned_call_id_350'].nunique() != 42:
    raise RuntimeError(
        f'Expected 42 planned 350-batches; found '
        f'{plan350["planned_call_id_350"].nunique():,}.'
    )

# load fixed a/b order

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order['pair_id'] = (
    pair_order['pair_id']
    .astype(str)
)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

# select only the first 350-task batch

SMOKE_BATCH_ID = 1

stage_name = (
    f'step4_round1_350_smoke_'
    f'{SMOKE_BATCH_ID:03d}'
)

call_df = (
    plan350[
        plan350['planned_call_id_350']
        == SMOKE_BATCH_ID
    ]
    .copy()
)

if len(call_df) != 350:
    raise RuntimeError(
        f'Expected exactly 350 judgments in '
        f'smoke batch 1; found {len(call_df):,}.'
    )

# duplicate-spend protection

call_log_before = read_jsonl(
    CALL_LOG
)

already_logged_stages = {
    str(x.get('stage', ''))
    for x in call_log_before
}

checkpoint_records = read_jsonl(
    ROUND1_CKPT
)

already_checkpointed_stages = {
    str(x.get('stage', ''))
    for x in checkpoint_records
}

if stage_name in already_checkpointed_stages:

    raise RuntimeError(
        f'{stage_name} is already checkpointed. '
        'No paid request was made.'
    )

if stage_name in already_logged_stages:

    raise RuntimeError(
        f'{stage_name} already appears in CALL_LOG '
        'but is not safely checkpointed. '
        'Do NOT pay for it again automatically.'
    )

# extra safety:
# make sure none of these 350 tasks are already saved

completed_task_ids = set()

for record in checkpoint_records:

    for vote_row in record.get(
        'votes',
        []
    ):

        rid = str(
            vote_row.get(
                'representative_id',
                ''
            )
        )

        pid = str(
            vote_row.get(
                'pair_id',
                ''
            )
        )

        if rid and pid:

            completed_task_ids.add(
                rid + '|' + pid
            )

call_df['task_id'] = (
    call_df['representative_id']
    + '|'
    + call_df['pair_id']
)

already_done_here = (
    call_df[
        'task_id'
    ].isin(
        completed_task_ids
    )
)

if already_done_here.any():

    raise RuntimeError(
        f'{already_done_here.sum():,} tasks in this smoke '
        'batch are already checkpointed. '
        'Rebuild the zero-cost replan before paying.'
    )

# build compact shared-pair payload

# reddit text appears once per distinct pair.

pair_payload = []

for pair_id, g in (
    call_df.groupby(
        'pair_id',
        sort=False
    )
):

    pid = str(
        pair_id
    )

    q = pair_lookup.loc[
        pid
    ]

    principles_for_pair = [

        {
            'id':
                str(r['representative_id']),

            'principle':
                str(r['principle'])
        }

        for _, r in g.iterrows()

    ]

    pair_payload.append({

        'pair_id':
            pid,

        'post_a':
            str(q['post_a']),

        'post_b':
            str(q['post_b']),

        'principles':
            principles_for_pair

    })

print(
    "========== 350-JUDGMENT SMOKE PREVIEW =========="
)

print(
    f'Batch ID:               '
    f'{SMOKE_BATCH_ID}'
)

print(
    f'Judgments:              '
    f'{len(call_df):,}'
)

print(
    f'Distinct Reddit pairs:  '
    f'{call_df.pair_id.nunique():,}'
)

print(
    'Paid calls to be made:  1'
)

print(
    'Automatic retries:     0'
)

print(
    "================================================"
)

# prompts

system_prompt = """
Apply each explicit principle independently to its assigned
pair of Reddit posts.

For every principle:
- Vote "A" if Post A better satisfies that principle.
- Vote "B" if Post B better satisfies that principle.
- Vote "None" if the principle is not applicable, does not
  meaningfully distinguish the posts, or neither post is
  clearly favored by that principle.

Judge ONLY by the supplied principle.
Do not judge overall post quality.
Do not use popularity, scores, votes, or outside information.
Treat every principle independently.

Return valid JSON only.
""".strip()

user_prompt = f"""
Evaluate EVERY supplied principle.

You must return:
- every supplied pair_id exactly once
- every supplied principle ID exactly once

Do not omit any principle.

Return ONLY this compact JSON structure:

{{
  "results": [
    {{
      "pair_id": "PAIR_ID",
      "votes": {{
        "C0000": "A",
        "C0001": "B",
        "C0002": "None"
      }}
    }}
  ]
}}

Allowed vote strings are ONLY:

"A"
"B"
"None"

DATA:

{json.dumps(pair_payload, ensure_ascii=False)}
""".strip()

# enable one paid request

subscription_key = getpass.getpass(
    'Azure OpenAI API key: '
)

client = AzureOpenAI(
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=subscription_key
)

RUN_PAID_LLM = True

try:

    # exactly one paid call

    result = chat_json(

        system_prompt,

        user_prompt,

        stage_name,

        max_tokens=10000,

        retries=1

    )

    # normalize returned votes

    expected_by_pair = defaultdict(
        set
    )

    for _, r in call_df.iterrows():

        expected_by_pair[
            str(r['pair_id'])
        ].add(
            str(
                r['representative_id']
            )
        )

    returned_votes = defaultdict(
        dict
    )

    for item in result.get(
        'results',
        []
    ):

        pid = str(
            item.get(
                'pair_id',
                ''
            )
        )

        if pid not in expected_by_pair:
            continue

        votes = item.get(
            'votes',
            {}
        )

        if not isinstance(
            votes,
            dict
        ):
            continue

        for rid, raw_vote in votes.items():

            rid = str(
                rid
            )

            if rid not in expected_by_pair[
                pid
            ]:
                continue

            # json null is not treated as "none".
            if not isinstance(
                raw_vote,
                str
            ):
                continue

            v = (
                raw_vote
                .strip()
                .upper()
            )

            if v == 'A':

                returned_votes[
                    pid
                ][rid] = 'A'

            elif v == 'B':

                returned_votes[
                    pid
                ][rid] = 'B'

            elif v == 'NONE':

                returned_votes[
                    pid
                ][rid] = 'None'

    # create valid checkpoint rows

    valid_rows = []

    missing_tasks = []

    for _, r in call_df.iterrows():

        pid = str(
            r['pair_id']
        )

        rid = str(
            r['representative_id']
        )

        vote = (
            returned_votes
            .get(
                pid,
                {}
            )
            .get(
                rid
            )
        )

        if vote is None:

            missing_tasks.append(
                rid + '|' + pid
            )

            continue

        correct_vote = str(
            pair_lookup.loc[
                pid,
                'correct_vote'
            ]
        )

        if vote == 'None':

            outcome = (
                'not_applicable'
            )

        elif vote == correct_vote:

            outcome = (
                'correct'
            )

        else:

            outcome = (
                'incorrect'
            )

        valid_rows.append({

            'representative_id':
                rid,

            'pair_id':
                pid,

            'vote':
                vote,

            'correct_vote':
                correct_vote,

            'outcome':
                outcome

        })

    # checkpoint this paid response

    checkpoint_record = {

        'stage':
            stage_name,

        'batch_scheme':
            '350',

        'planned_call_id_350':
            SMOKE_BATCH_ID,

        'expected_judgments':
            int(
                len(call_df)
            ),

        'received_judgments':
            int(
                len(valid_rows)
            ),

        'missing_judgments':
            int(
                len(missing_tasks)
            ),

        'votes':
            valid_rows,

        'missing_task_ids':
            missing_tasks

    }

    append_jsonl(
        ROUND1_CKPT,
        [
            checkpoint_record
        ]
    )

finally:

    # always block paid calls again

    RUN_PAID_LLM = False

    client = None

# read actual token usage

stage_log = [

    x

    for x in read_jsonl(
        CALL_LOG
    )

    if str(
        x.get(
            'stage',
            ''
        )
    ) == stage_name

]

if not stage_log:

    raise RuntimeError(
        'Paid request returned, but its token log '
        'could not be found.'
    )

usage = stage_log[-1]

prompt_tokens = (
    usage.get(
        'prompt_tokens'
    )
    or 0
)

completion_tokens = (
    usage.get(
        'completion_tokens'
    )
    or 0
)

total_tokens = (
    usage.get(
        'total_tokens'
    )
    or 0
)

# summary

valid_df = pd.DataFrame(
    valid_rows
)

if len(valid_df):

    vote_counts = (
        valid_df[
            'vote'
        ]
        .value_counts()
        .to_dict()
    )

else:

    vote_counts = {}

coverage = (
    len(valid_rows)
    /
    len(call_df)
)

# if all 42 new calls were approximately like this one.
projected_42_prompt = (
    prompt_tokens * 42
)

projected_42_completion = (
    completion_tokens * 42
)

projected_42_total = (
    total_tokens * 42
)

print()
print(
    "========== 350-JUDGMENT SMOKE RESULT =========="
)

print(
    f'Expected judgments:       '
    f'{len(call_df):,}'
)

print(
    f'Valid judgments returned: '
    f'{len(valid_rows):,}'
)

print(
    f'Missing judgments:        '
    f'{len(missing_tasks):,}'
)

print(
    f'Coverage:                 '
    f'{coverage:.2%}'
)

print()

print(
    'Vote counts:'
)

print(
    f'  A:    '
    f'{vote_counts.get("A", 0):,}'
)

print(
    f'  B:    '
    f'{vote_counts.get("B", 0):,}'
)

print(
    f'  None: '
    f'{vote_counts.get("None", 0):,}'
)

print()

print(
    'ACTUAL TOKEN USAGE:'
)

print(
    f'  Prompt tokens:     '
    f'{prompt_tokens:,}'
)

print(
    f'  Completion tokens: '
    f'{completion_tokens:,}'
)

print(
    f'  Total tokens:      '
    f'{total_tokens:,}'
)

print()

print(
    'Rough 42-call extrapolation '
    'if calls were similar:'
)

print(
    f'  Prompt tokens:     '
    f'{projected_42_prompt:,}'
)

print(
    f'  Completion tokens: '
    f'{projected_42_completion:,}'
)

print(
    f'  Total tokens:      '
    f'{projected_42_total:,}'
)

print()

print(
    'PAID CALLS MADE BY THIS CELL: 1'
)

print(
    'PAID LLM CALLS ARE BLOCKED AGAIN:',
    not RUN_PAID_LLM
)

print(
    "================================================"
)


In [ ]:

# step 4 round 1 — controlled 10-call tranche

# runs planned 350-batches 2 through 11 only.

# maximum paid calls: 10
# automatic retries: 0

# preserves:
# - 480 valid votes from the 500-judgment smoke test
# - 350 valid votes from the 350-judgment smoke test

# every successful batch is checkpointed immediately.
# paid calls are blocked again when the tranche ends.

import getpass
import json
from collections import defaultdict

import pandas as pd
from openai import AzureOpenAI
from tqdm.auto import tqdm

# files

PLAN_FILE = (
    OUTPUT_DIR /
    'step4_round1_remaining_plan_350.csv'
)

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

ROUND1_CKPT = (
    OUTPUT_DIR /
    'step4_round1_shared_checkpoint.jsonl'
)

# safety

RUN_PAID_LLM = False

FIRST_BATCH_ID = 32
LAST_BATCH_ID = 41

MAX_NEW_CALLS_THIS_CELL = 10

# load plan

plan = pd.read_csv(
    PLAN_FILE,
    keep_default_na=False
)

plan['pair_id'] = (
    plan['pair_id']
    .astype(str)
)

plan['representative_id'] = (
    plan['representative_id']
    .astype(str)
)

if len(plan) != 14520:
    raise RuntimeError(
        f'Expected 14,520 tasks in the 350-plan; '
        f'found {len(plan):,}.'
    )

if plan['planned_call_id_350'].nunique() != 42:
    raise RuntimeError(
        'Expected exactly 42 planned 350-batches.'
    )

# load a/b order

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order['pair_id'] = (
    pair_order['pair_id']
    .astype(str)
)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

# helper — currently completed tasks

def get_completed_round1_tasks():

    completed = set()

    for record in read_jsonl(
        ROUND1_CKPT
    ):

        for vote_row in record.get(
            'votes',
            []
        ):

            rid = str(
                vote_row.get(
                    'representative_id',
                    ''
                )
            )

            pid = str(
                vote_row.get(
                    'pair_id',
                    ''
                )
            )

            vote = vote_row.get(
                'vote'
            )

            if (
                rid
                and pid
                and vote in {
                    'A',
                    'B',
                    'None'
                }
            ):

                completed.add(
                    rid + '|' + pid
                )

    return completed

completed_before = (
    get_completed_round1_tasks()
)

print(
    "========== 10-CALL TRANCHE PREVIEW =========="
)

print(
    f'Round-1 judgments already complete: '
    f'{len(completed_before):,} / 15,000'
)

print(
    f'Planned 350-batches: '
    f'{FIRST_BATCH_ID} through {LAST_BATCH_ID}'
)

print(
    f'Maximum new paid calls: '
    f'{MAX_NEW_CALLS_THIS_CELL}'
)

print(
    'Automatic retries: 0'
)

print(
    'Expected judgments in tranche: 3,500'
)

print(
    "============================================="
)

# exact same prompt format as successful 350 smoke test

system_prompt = """
Apply each explicit principle independently to its assigned
pair of Reddit posts.

For every principle:
- Vote "A" if Post A better satisfies that principle.
- Vote "B" if Post B better satisfies that principle.
- Vote "None" if the principle is not applicable, does not
  meaningfully distinguish the posts, or neither post is
  clearly favored by that principle.

Judge ONLY by the supplied principle.
Do not judge overall post quality.
Do not use popularity, scores, votes, or outside information.
Treat every principle independently.

Return valid JSON only.
""".strip()

# initialize azure

subscription_key = getpass.getpass(
    'Azure OpenAI API key: '
)

client = AzureOpenAI(
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=subscription_key
)

RUN_PAID_LLM = True

# run exactly batches 2–11

new_calls_sent = 0
stages_this_tranche = []

tranche_expected = 0
tranche_received = 0
tranche_missing = 0

stopped_due_to_error = False
error_message = None

try:

    for batch_id in tqdm(
        range(
            FIRST_BATCH_ID,
            LAST_BATCH_ID + 1
        ),
        desc='Round-1 350 tranche'
    ):

        # local hard cap

        if (
            new_calls_sent
            >= MAX_NEW_CALLS_THIS_CELL
        ):
            break

        stage_name = (
            f'step4_round1_350_batch_'
            f'{batch_id:03d}'
        )

        # duplicate-spend protection

        checkpoint_records = read_jsonl(
            ROUND1_CKPT
        )

        checkpointed_stages = {
            str(
                x.get(
                    'stage',
                    ''
                )
            )
            for x in checkpoint_records
        }

        logged_stages = {
            str(
                x.get(
                    'stage',
                    ''
                )
            )
            for x in read_jsonl(
                CALL_LOG
            )
        }

        if stage_name in checkpointed_stages:

            print(
                f'\nSkipping batch {batch_id}: '
                'already checkpointed.'
            )

            continue

        if stage_name in logged_stages:

            raise RuntimeError(
                f'{stage_name} already appears in CALL_LOG '
                'but is not safely checkpointed. '
                'Stopping rather than paying for it again.'
            )

        # get this exact planned batch

        batch_df = (
            plan[
                plan[
                    'planned_call_id_350'
                ]
                == batch_id
            ]
            .copy()
        )

        if len(batch_df) == 0:
            raise RuntimeError(
                f'No tasks found for batch '
                f'{batch_id}.'
            )

        # normally 350, except the final batch 42.
        if (
            batch_id < 42
            and len(batch_df) != 350
        ):
            raise RuntimeError(
                f'Batch {batch_id} should contain '
                f'350 tasks; found {len(batch_df)}.'
            )

        # make sure none are already complete

        completed_now = (
            get_completed_round1_tasks()
        )

        batch_df['task_id'] = (
            batch_df[
                'representative_id'
            ]
            +
            '|'
            +
            batch_df[
                'pair_id'
            ]
        )

        already_done_mask = (
            batch_df[
                'task_id'
            ].isin(
                completed_now
            )
        )

        if already_done_mask.any():

            raise RuntimeError(
                f'Batch {batch_id} contains '
                f'{already_done_mask.sum():,} '
                'already-completed tasks. '
                'Stopping to avoid duplicate spending.'
            )

        # build shared-pair payload

        pair_payload = []

        for pair_id, g in (
            batch_df.groupby(
                'pair_id',
                sort=False
            )
        ):

            pid = str(
                pair_id
            )

            q = pair_lookup.loc[
                pid
            ]

            principles_for_pair = [

                {
                    'id':
                        str(
                            r[
                                'representative_id'
                            ]
                        ),

                    'principle':
                        str(
                            r[
                                'principle'
                            ]
                        )
                }

                for _, r
                in g.iterrows()

            ]

            pair_payload.append({

                'pair_id':
                    pid,

                'post_a':
                    str(
                        q['post_a']
                    ),

                'post_b':
                    str(
                        q['post_b']
                    ),

                'principles':
                    principles_for_pair

            })

        user_prompt = f"""
Evaluate EVERY supplied principle.

You must return:
- every supplied pair_id exactly once
- every supplied principle ID exactly once

Do not omit any principle.

Return ONLY this compact JSON structure:

{{
  "results": [
    {{
      "pair_id": "PAIR_ID",
      "votes": {{
        "C0000": "A",
        "C0001": "B",
        "C0002": "None"
      }}
    }}
  ]
}}

Allowed vote strings are ONLY:

"A"
"B"
"None"

DATA:

{json.dumps(pair_payload, ensure_ascii=False)}
""".strip()

        # one paid request for this batch

        try:

            result = chat_json(

                system_prompt,

                user_prompt,

                stage_name,

                max_tokens=10000,

                retries=1

            )

            new_calls_sent += 1

            stages_this_tranche.append(
                stage_name
            )

        except Exception as exc:

            stopped_due_to_error = True

            error_message = (
                f'{type(exc).__name__}: '
                f'{exc}'
            )

            print()
            print(
                f'Stopping safely at batch '
                f'{batch_id}.'
            )

            print(
                error_message
            )

            break

        # normalize returned votes

        expected_by_pair = defaultdict(
            set
        )

        for _, r in (
            batch_df.iterrows()
        ):

            expected_by_pair[
                str(r['pair_id'])
            ].add(
                str(
                    r[
                        'representative_id'
                    ]
                )
            )

        returned_votes = defaultdict(
            dict
        )

        for item in result.get(
            'results',
            []
        ):

            pid = str(
                item.get(
                    'pair_id',
                    ''
                )
            )

            if (
                pid
                not in expected_by_pair
            ):
                continue

            votes = item.get(
                'votes',
                {}
            )

            if not isinstance(
                votes,
                dict
            ):
                continue

            for rid, raw_vote in (
                votes.items()
            ):

                rid = str(
                    rid
                )

                if (
                    rid
                    not in expected_by_pair[
                        pid
                    ]
                ):
                    continue

                if not isinstance(
                    raw_vote,
                    str
                ):
                    continue

                v = (
                    raw_vote
                    .strip()
                    .upper()
                )

                if v == 'A':

                    returned_votes[
                        pid
                    ][rid] = 'A'

                elif v == 'B':

                    returned_votes[
                        pid
                    ][rid] = 'B'

                elif v == 'NONE':

                    returned_votes[
                        pid
                    ][rid] = 'None'

        # create checkpoint rows

        valid_rows = []
        missing_tasks = []

        for _, r in (
            batch_df.iterrows()
        ):

            pid = str(
                r['pair_id']
            )

            rid = str(
                r[
                    'representative_id'
                ]
            )

            vote = (
                returned_votes
                .get(
                    pid,
                    {}
                )
                .get(
                    rid
                )
            )

            if vote is None:

                missing_tasks.append(
                    rid + '|' + pid
                )

                continue

            correct_vote = str(
                pair_lookup.loc[
                    pid,
                    'correct_vote'
                ]
            )

            if vote == 'None':

                outcome = (
                    'not_applicable'
                )

            elif vote == correct_vote:

                outcome = (
                    'correct'
                )

            else:

                outcome = (
                    'incorrect'
                )

            valid_rows.append({

                'representative_id':
                    rid,

                'pair_id':
                    pid,

                'vote':
                    vote,

                'correct_vote':
                    correct_vote,

                'outcome':
                    outcome

            })

        # checkpoint immediately

        checkpoint_record = {

            'stage':
                stage_name,

            'batch_scheme':
                '350',

            'planned_call_id_350':
                int(
                    batch_id
                ),

            'expected_judgments':
                int(
                    len(batch_df)
                ),

            'received_judgments':
                int(
                    len(valid_rows)
                ),

            'missing_judgments':
                int(
                    len(missing_tasks)
                ),

            'votes':
                valid_rows,

            'missing_task_ids':
                missing_tasks

        }

        append_jsonl(

            ROUND1_CKPT,

            [
                checkpoint_record
            ]

        )

        tranche_expected += (
            len(batch_df)
        )

        tranche_received += (
            len(valid_rows)
        )

        tranche_missing += (
            len(missing_tasks)
        )

        print(
            f'\nBatch {batch_id}: '
            f'{len(valid_rows)}/{len(batch_df)} '
            f'valid; '
            f'{len(missing_tasks)} missing.'
        )

finally:

    # always turn paid calls back off

    RUN_PAID_LLM = False

    client = None

# token usage for this tranche

call_log = read_jsonl(
    CALL_LOG
)

tranche_logs = [

    x

    for x in call_log

    if str(
        x.get(
            'stage',
            ''
        )
    ) in set(
        stages_this_tranche
    )

]

tranche_prompt_tokens = sum(

    x.get(
        'prompt_tokens'
    )
    or 0

    for x in tranche_logs

)

tranche_completion_tokens = sum(

    x.get(
        'completion_tokens'
    )
    or 0

    for x in tranche_logs

)

tranche_total_tokens = sum(

    x.get(
        'total_tokens'
    )
    or 0

    for x in tranche_logs

)

# cumulative round-1 status

completed_after = (
    get_completed_round1_tasks()
)

total_complete = len(
    completed_after
)

total_remaining = (
    15000
    -
    total_complete
)

# how many of the 5,000 principles currently have 0, 1, 2,
# or all 3 judgments?
all_assignments = pd.read_csv(
    OUTPUT_DIR /
    'step4_round1_assignments.csv',
    keep_default_na=False
)

all_assignments[
    'representative_id'
] = (
    all_assignments[
        'representative_id'
    ]
    .astype(str)
)

all_assignments[
    'pair_id'
] = (
    all_assignments[
        'pair_id'
    ]
    .astype(str)
)

all_assignments[
    'task_id'
] = (
    all_assignments[
        'representative_id'
    ]
    +
    '|'
    +
    all_assignments[
        'pair_id'
    ]
)

completed_assignment_df = (
    all_assignments[
        all_assignments[
            'task_id'
        ].isin(
            completed_after
        )
    ]
)

judgments_per_principle = (

    completed_assignment_df

    .groupby(
        'representative_id'
    )

    .size()

    .reindex(
        all_assignments[
            'representative_id'
        ].unique(),
        fill_value=0
    )

)

coverage_counts = (
    judgments_per_principle
    .value_counts()
    .sort_index()
)

# final summary

tranche_coverage = (

    tranche_received
    /
    tranche_expected

    if tranche_expected

    else 0
)

print()
print(
    "========== 10-CALL TRANCHE RESULT =========="
)

print(
    f'Paid calls completed:        '
    f'{new_calls_sent:,} / 10'
)

print(
    f'Expected tranche judgments:  '
    f'{tranche_expected:,}'
)

print(
    f'Valid judgments returned:    '
    f'{tranche_received:,}'
)

print(
    f'Missing judgments:           '
    f'{tranche_missing:,}'
)

print(
    f'Tranche coverage:            '
    f'{tranche_coverage:.2%}'
)

print()

print(
    'TOKEN USAGE FOR THIS TRANCHE:'
)

print(
    f'  Prompt tokens:     '
    f'{tranche_prompt_tokens:,}'
)

print(
    f'  Completion tokens: '
    f'{tranche_completion_tokens:,}'
)

print(
    f'  Total tokens:      '
    f'{tranche_total_tokens:,}'
)

print()

print(
    'CUMULATIVE ROUND-1 STATUS:'
)

print(
    f'  Complete judgments: '
    f'{total_complete:,} / 15,000'
)

print(
    f'  Still missing:      '
    f'{total_remaining:,}'
)

print()

print(
    'Principles by completed judgment count:'
)

for n in range(4):

    print(
        f'  {n} judgments: '
        f'{int(coverage_counts.get(n, 0)):,}'
    )

print()

if stopped_due_to_error:

    print(
        'TRANCHE STOPPED EARLY DUE TO ERROR:'
    )

    print(
        error_message
    )

else:

    print(
        'All 10 planned tranche calls finished.'
    )

print()

print(
    'PAID LLM CALLS ARE BLOCKED AGAIN:',
    not RUN_PAID_LLM
)

print(
    "============================================"
)


In [ ]:

# step 4 round 1 — final normal batch 42

# runs only planned batch 42.

# expected tasks: 170
# maximum paid calls: 1
# automatic retries: 0

# checkpoints all valid results immediately.
# does not repair omissions.

import getpass
import json
from collections import defaultdict

import pandas as pd
from openai import AzureOpenAI

# files

PLAN_FILE = (
    OUTPUT_DIR /
    'step4_round1_remaining_plan_350.csv'
)

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

ROUND1_CKPT = (
    OUTPUT_DIR /
    'step4_round1_shared_checkpoint.jsonl'
)

# safety

RUN_PAID_LLM = False

BATCH_ID = 42

stage_name = (
    'step4_round1_350_batch_042'
)

# load plan

plan = pd.read_csv(
    PLAN_FILE,
    keep_default_na=False
)

plan['pair_id'] = (
    plan['pair_id']
    .astype(str)
)

plan['representative_id'] = (
    plan['representative_id']
    .astype(str)
)

batch_df = (
    plan[
        plan[
            'planned_call_id_350'
        ] == BATCH_ID
    ]
    .copy()
)

print(
    "========== FINAL NORMAL BATCH PREVIEW =========="
)

print(
    f'Batch ID:              {BATCH_ID}'
)

print(
    f'Planned judgments:     {len(batch_df):,}'
)

print(
    'Maximum paid calls:    1'
)

print(
    'Automatic retries:     0'
)

print(
    "==============================================="
)

if len(batch_df) != 170:

    raise RuntimeError(
        f'Expected 170 judgments in batch 42; '
        f'found {len(batch_df):,}.'
    )

# load a/b order

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order['pair_id'] = (
    pair_order['pair_id']
    .astype(str)
)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

# helper — find completed round-1 tasks

def get_completed_round1_tasks():

    completed = set()

    for record in read_jsonl(
        ROUND1_CKPT
    ):

        for vote_row in record.get(
            'votes',
            []
        ):

            rid = str(
                vote_row.get(
                    'representative_id',
                    ''
                )
            )

            pid = str(
                vote_row.get(
                    'pair_id',
                    ''
                )
            )

            vote = vote_row.get(
                'vote'
            )

            if (
                rid
                and pid
                and vote in {
                    'A',
                    'B',
                    'None'
                }
            ):

                completed.add(
                    rid + '|' + pid
                )

    return completed

completed_before = (
    get_completed_round1_tasks()
)

# duplicate-spend safety

checkpointed_stages = {

    str(x.get('stage', ''))

    for x in read_jsonl(
        ROUND1_CKPT
    )
}

logged_stages = {

    str(x.get('stage', ''))

    for x in read_jsonl(
        CALL_LOG
    )
}

if stage_name in checkpointed_stages:

    raise RuntimeError(
        'Batch 42 is already checkpointed. '
        'No paid call was made.'
    )

if stage_name in logged_stages:

    raise RuntimeError(
        'Batch 42 already appears in CALL_LOG '
        'but is not safely checkpointed. '
        'Stopping rather than paying twice.'
    )

batch_df['task_id'] = (
    batch_df['representative_id']
    +
    '|'
    +
    batch_df['pair_id']
)

already_done = (
    batch_df[
        'task_id'
    ].isin(
        completed_before
    )
)

if already_done.any():

    raise RuntimeError(
        f'{already_done.sum():,} batch-42 tasks '
        'are already checkpointed. '
        'Stopping to avoid duplicate spending.'
    )

# build shared-pair payload

pair_payload = []

for pair_id, g in (
    batch_df.groupby(
        'pair_id',
        sort=False
    )
):

    pid = str(
        pair_id
    )

    q = pair_lookup.loc[
        pid
    ]

    principles_for_pair = [

        {
            'id':
                str(
                    r['representative_id']
                ),

            'principle':
                str(
                    r['principle']
                )
        }

        for _, r
        in g.iterrows()

    ]

    pair_payload.append({

        'pair_id':
            pid,

        'post_a':
            str(
                q['post_a']
            ),

        'post_b':
            str(
                q['post_b']
            ),

        'principles':
            principles_for_pair

    })

print()
print(
    f'Distinct Reddit pairs in batch: '
    f'{batch_df.pair_id.nunique():,}'
)

# prompts

system_prompt = """
Apply each explicit principle independently to its assigned
pair of Reddit posts.

For every principle:
- Vote "A" if Post A better satisfies that principle.
- Vote "B" if Post B better satisfies that principle.
- Vote "None" if the principle is not applicable, does not
  meaningfully distinguish the posts, or neither post is
  clearly favored by that principle.

Judge ONLY by the supplied principle.
Do not judge overall post quality.
Do not use popularity, scores, votes, or outside information.
Treat every principle independently.

Return valid JSON only.
""".strip()

user_prompt = f"""
Evaluate EVERY supplied principle.

You must return:
- every supplied pair_id exactly once
- every supplied principle ID exactly once

Do not omit any principle.

Return ONLY this compact JSON structure:

{{
  "results": [
    {{
      "pair_id": "PAIR_ID",
      "votes": {{
        "C0000": "A",
        "C0001": "B",
        "C0002": "None"
      }}
    }}
  ]
}}

Allowed vote strings are ONLY:

"A"
"B"
"None"

DATA:

{json.dumps(pair_payload, ensure_ascii=False)}
""".strip()

# enable exactly one paid call

subscription_key = getpass.getpass(
    'Azure OpenAI API key: '
)

client = AzureOpenAI(
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=subscription_key
)

RUN_PAID_LLM = True

try:

    result = chat_json(

        system_prompt,

        user_prompt,

        stage_name,

        max_tokens=7000,

        retries=1

    )

    # normalize output

    expected_by_pair = defaultdict(
        set
    )

    for _, r in batch_df.iterrows():

        expected_by_pair[
            str(r['pair_id'])
        ].add(
            str(
                r['representative_id']
            )
        )

    returned_votes = defaultdict(
        dict
    )

    for item in result.get(
        'results',
        []
    ):

        pid = str(
            item.get(
                'pair_id',
                ''
            )
        )

        if pid not in expected_by_pair:
            continue

        votes = item.get(
            'votes',
            {}
        )

        if not isinstance(
            votes,
            dict
        ):
            continue

        for rid, raw_vote in votes.items():

            rid = str(
                rid
            )

            if rid not in expected_by_pair[
                pid
            ]:
                continue

            if not isinstance(
                raw_vote,
                str
            ):
                continue

            v = (
                raw_vote
                .strip()
                .upper()
            )

            if v == 'A':

                returned_votes[
                    pid
                ][rid] = 'A'

            elif v == 'B':

                returned_votes[
                    pid
                ][rid] = 'B'

            elif v == 'NONE':

                returned_votes[
                    pid
                ][rid] = 'None'

    # build valid rows

    valid_rows = []

    missing_tasks = []

    for _, r in batch_df.iterrows():

        pid = str(
            r['pair_id']
        )

        rid = str(
            r['representative_id']
        )

        vote = (
            returned_votes
            .get(
                pid,
                {}
            )
            .get(
                rid
            )
        )

        if vote is None:

            missing_tasks.append(
                rid + '|' + pid
            )

            continue

        correct_vote = str(
            pair_lookup.loc[
                pid,
                'correct_vote'
            ]
        )

        if vote == 'None':

            outcome = (
                'not_applicable'
            )

        elif vote == correct_vote:

            outcome = (
                'correct'
            )

        else:

            outcome = (
                'incorrect'
            )

        valid_rows.append({

            'representative_id':
                rid,

            'pair_id':
                pid,

            'vote':
                vote,

            'correct_vote':
                correct_vote,

            'outcome':
                outcome

        })

    # checkpoint immediately

    checkpoint_record = {

        'stage':
            stage_name,

        'batch_scheme':
            '350',

        'planned_call_id_350':
            BATCH_ID,

        'expected_judgments':
            int(
                len(batch_df)
            ),

        'received_judgments':
            int(
                len(valid_rows)
            ),

        'missing_judgments':
            int(
                len(missing_tasks)
            ),

        'votes':
            valid_rows,

        'missing_task_ids':
            missing_tasks

    }

    append_jsonl(
        ROUND1_CKPT,
        [
            checkpoint_record
        ]
    )

finally:

    RUN_PAID_LLM = False

    client = None

# token usage

stage_logs = [

    x

    for x in read_jsonl(
        CALL_LOG
    )

    if str(
        x.get(
            'stage',
            ''
        )
    ) == stage_name

]

if not stage_logs:

    raise RuntimeError(
        'Batch completed but token log '
        'could not be found.'
    )

usage = stage_logs[-1]

prompt_tokens = (
    usage.get(
        'prompt_tokens'
    )
    or 0
)

completion_tokens = (
    usage.get(
        'completion_tokens'
    )
    or 0
)

total_tokens = (
    usage.get(
        'total_tokens'
    )
    or 0
)

# cumulative status

completed_after = (
    get_completed_round1_tasks()
)

total_complete = len(
    completed_after
)

total_missing = (
    15000
    -
    total_complete
)

print()
print(
    "========== FINAL NORMAL BATCH RESULT =========="
)

print(
    f'Expected judgments:       '
    f'{len(batch_df):,}'
)

print(
    f'Valid judgments returned: '
    f'{len(valid_rows):,}'
)

print(
    f'Missing from batch 42:     '
    f'{len(missing_tasks):,}'
)

print(
    f'Coverage:                 '
    f'{len(valid_rows)/len(batch_df):.2%}'
)

print()

print(
    'TOKEN USAGE:'
)

print(
    f'  Prompt tokens:     '
    f'{prompt_tokens:,}'
)

print(
    f'  Completion tokens: '
    f'{completion_tokens:,}'
)

print(
    f'  Total tokens:      '
    f'{total_tokens:,}'
)

print()

print(
    'CUMULATIVE ROUND-1 STATUS:'
)

print(
    f'  Complete judgments: '
    f'{total_complete:,} / 15,000'
)

print(
    f'  Still missing:      '
    f'{total_missing:,}'
)

print()

print(
    'PAID CALLS MADE: 1'
)

print(
    'PAID LLM CALLS ARE BLOCKED AGAIN:',
    not RUN_PAID_LLM
)

print(
    "================================================"
)


In [ ]:

# step 4 round 1 — repair all remaining omissions
# + compute round-1 metrics
# + select up to 500 survivors

# expected repair tasks: 13
# maximum paid calls: 1
# automatic retries: 0

# after repair:
# - verifies exactly 15,000 / 15,000 judgments
# - computes applicability + accuracy + net score
# - applies original round-1 continuation thresholds
# - ranks eligible principles
# - keeps at most 500 for round 2

import getpass
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from openai import AzureOpenAI

# files

ASSIGNMENT_FILE = (
    OUTPUT_DIR /
    'step4_round1_assignments.csv'
)

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

ROUND1_CKPT = (
    OUTPUT_DIR /
    'step4_round1_shared_checkpoint.jsonl'
)

REP_FILE = (
    OUTPUT_DIR /
    'cluster_representatives.csv'
)

# safety

RUN_PAID_LLM = False

REPAIR_STAGE = (
    'step4_round1_final_repair'
)

MAX_REPAIR_TASKS = 50

# load all required round-1 tasks

assignments = pd.read_csv(
    ASSIGNMENT_FILE,
    keep_default_na=False
)

assignments['representative_id'] = (
    assignments['representative_id']
    .astype(str)
)

assignments['pair_id'] = (
    assignments['pair_id']
    .astype(str)
)

assignments['task_id'] = (
    assignments['representative_id']
    + '|'
    + assignments['pair_id']
)

if len(assignments) != 15000:
    raise RuntimeError(
        f'Expected exactly 15,000 Round-1 tasks; '
        f'found {len(assignments):,}.'
    )

if assignments['task_id'].duplicated().any():
    raise RuntimeError(
        'Duplicate Round-1 task IDs found.'
    )

# load fixed a/b order

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order['pair_id'] = (
    pair_order['pair_id']
    .astype(str)
)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

# helper — read all valid checkpointed votes

def collect_round1_votes():

    rows = []

    for record in read_jsonl(
        ROUND1_CKPT
    ):

        for r in record.get(
            'votes',
            []
        ):

            rid = str(
                r.get(
                    'representative_id',
                    ''
                )
            )

            pid = str(
                r.get(
                    'pair_id',
                    ''
                )
            )

            vote = r.get(
                'vote'
            )

            if (
                rid
                and pid
                and vote in {
                    'A',
                    'B',
                    'None'
                }
            ):

                rows.append({
                    'representative_id': rid,
                    'pair_id': pid,
                    'task_id': rid + '|' + pid,
                    'vote': vote
                })

    if not rows:

        return pd.DataFrame(
            columns=[
                'representative_id',
                'pair_id',
                'task_id',
                'vote'
            ]
        )

    out = pd.DataFrame(
        rows
    )

    # safety against accidental duplicate checkpoint records.
    out = (
        out
        .drop_duplicates(
            'task_id',
            keep='first'
        )
        .reset_index(drop=True)
    )

    return out

saved_before = (
    collect_round1_votes()
)

completed_ids = set(
    saved_before[
        'task_id'
    ]
)

# identify exactly what is still missing

missing = (
    assignments[
        ~assignments[
            'task_id'
        ].isin(
            completed_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "========== FINAL REPAIR PREVIEW =========="
)

print(
    f'Complete judgments: '
    f'{len(saved_before):,} / 15,000'
)

print(
    f'Missing judgments:  '
    f'{len(missing):,}'
)

print(
    'Maximum paid calls: 1'
)

print(
    'Automatic retries:  0'
)

print(
    "=========================================="
)

if len(missing) > MAX_REPAIR_TASKS:

    raise RuntimeError(
        f'Expected only a small omission set, but '
        f'{len(missing):,} tasks are missing. '
        'No paid call was made.'
    )

# run one repair call only if something is missing

if len(missing) > 0:

    # duplicate-spend protection

    checkpointed_stages = {
        str(x.get('stage', ''))
        for x in read_jsonl(
            ROUND1_CKPT
        )
    }

    logged_stages = {
        str(x.get('stage', ''))
        for x in read_jsonl(
            CALL_LOG
        )
    }

    if REPAIR_STAGE in checkpointed_stages:

        raise RuntimeError(
            'The final repair is already checkpointed. '
            'No paid request was made.'
        )

    if REPAIR_STAGE in logged_stages:

        raise RuntimeError(
            'The final repair already appears in CALL_LOG '
            'but is not safely checkpointed. '
            'Stopping to avoid duplicate spending.'
        )

    # build compact shared-pair payload

    pair_payload = []

    for pair_id, g in (
        missing.groupby(
            'pair_id',
            sort=False
        )
    ):

        pid = str(
            pair_id
        )

        q = pair_lookup.loc[
            pid
        ]

        principles_for_pair = [

            {
                'id':
                    str(
                        r[
                            'representative_id'
                        ]
                    ),

                'principle':
                    str(
                        r[
                            'principle'
                        ]
                    )
            }

            for _, r
            in g.iterrows()

        ]

        pair_payload.append({

            'pair_id':
                pid,

            'post_a':
                str(
                    q['post_a']
                ),

            'post_b':
                str(
                    q['post_b']
                ),

            'principles':
                principles_for_pair

        })

    print()
    print(
        f'Repairing {len(missing):,} judgments '
        f'across {missing.pair_id.nunique():,} '
        'Reddit pairs.'
    )

    # same judging instructions used through round 1

    system_prompt = """
Apply each explicit principle independently to its assigned
pair of Reddit posts.

For every principle:
- Vote "A" if Post A better satisfies that principle.
- Vote "B" if Post B better satisfies that principle.
- Vote "None" if the principle is not applicable, does not
  meaningfully distinguish the posts, or neither post is
  clearly favored by that principle.

Judge ONLY by the supplied principle.
Do not judge overall post quality.
Do not use popularity, scores, votes, or outside information.
Treat every principle independently.

Return valid JSON only.
""".strip()

    user_prompt = f"""
Evaluate EVERY supplied principle.

You must return every supplied pair_id and every supplied
principle ID exactly once.

Do not omit any principle.

Return ONLY:

{{
  "results": [
    {{
      "pair_id": "PAIR_ID",
      "votes": {{
        "C0000": "A",
        "C0001": "B",
        "C0002": "None"
      }}
    }}
  ]
}}

Allowed votes are ONLY:
"A", "B", or "None".

DATA:

{json.dumps(pair_payload, ensure_ascii=False)}
""".strip()

    # enable exactly one paid call

    subscription_key = getpass.getpass(
        'Azure OpenAI API key: '
    )

    client = AzureOpenAI(
        api_version=AZURE_API_VERSION,
        azure_endpoint=AZURE_ENDPOINT,
        api_key=subscription_key
    )

    RUN_PAID_LLM = True

    try:

        result = chat_json(

            system_prompt,

            user_prompt,

            REPAIR_STAGE,

            max_tokens=2500,

            retries=1

        )

        # parse output

        expected_by_pair = defaultdict(
            set
        )

        for _, r in missing.iterrows():

            expected_by_pair[
                str(r['pair_id'])
            ].add(
                str(
                    r[
                        'representative_id'
                    ]
                )
            )

        returned_votes = defaultdict(
            dict
        )

        for item in result.get(
            'results',
            []
        ):

            pid = str(
                item.get(
                    'pair_id',
                    ''
                )
            )

            if pid not in expected_by_pair:
                continue

            votes = item.get(
                'votes',
                {}
            )

            if not isinstance(
                votes,
                dict
            ):
                continue

            for rid, raw_vote in votes.items():

                rid = str(
                    rid
                )

                if rid not in expected_by_pair[
                    pid
                ]:
                    continue

                if not isinstance(
                    raw_vote,
                    str
                ):
                    continue

                v = (
                    raw_vote
                    .strip()
                    .upper()
                )

                if v == 'A':

                    returned_votes[
                        pid
                    ][rid] = 'A'

                elif v == 'B':

                    returned_votes[
                        pid
                    ][rid] = 'B'

                elif v == 'NONE':

                    returned_votes[
                        pid
                    ][rid] = 'None'

        # build repair checkpoint

        repaired_rows = []

        still_missing_ids = []

        for _, r in missing.iterrows():

            pid = str(
                r['pair_id']
            )

            rid = str(
                r[
                    'representative_id'
                ]
            )

            vote = (
                returned_votes
                .get(
                    pid,
                    {}
                )
                .get(
                    rid
                )
            )

            if vote is None:

                still_missing_ids.append(
                    rid + '|' + pid
                )

                continue

            correct_vote = str(
                pair_lookup.loc[
                    pid,
                    'correct_vote'
                ]
            )

            if vote == 'None':

                outcome = (
                    'not_applicable'
                )

            elif vote == correct_vote:

                outcome = (
                    'correct'
                )

            else:

                outcome = (
                    'incorrect'
                )

            repaired_rows.append({

                'representative_id':
                    rid,

                'pair_id':
                    pid,

                'vote':
                    vote,

                'correct_vote':
                    correct_vote,

                'outcome':
                    outcome

            })

        append_jsonl(

            ROUND1_CKPT,

            [{
                'stage':
                    REPAIR_STAGE,

                'batch_scheme':
                    'final_repair',

                'expected_judgments':
                    int(
                        len(missing)
                    ),

                'received_judgments':
                    int(
                        len(repaired_rows)
                    ),

                'missing_judgments':
                    int(
                        len(
                            still_missing_ids
                        )
                    ),

                'votes':
                    repaired_rows,

                'missing_task_ids':
                    still_missing_ids
            }]

        )

    finally:

        RUN_PAID_LLM = False

        client = None

# verify complete round 1

round1_votes = (
    collect_round1_votes()
)

all_required_ids = set(
    assignments[
        'task_id'
    ]
)

all_complete_ids = set(
    round1_votes[
        'task_id'
    ]
)

remaining_ids = (
    all_required_ids
    -
    all_complete_ids
)

print()
print(
    "========== ROUND-1 COMPLETENESS =========="
)

print(
    f'Valid judgments: '
    f'{len(round1_votes):,} / 15,000'
)

print(
    f'Still missing:   '
    f'{len(remaining_ids):,}'
)

print(
    'Paid calls are blocked again:',
    not RUN_PAID_LLM
)

print(
    "=========================================="
)

if remaining_ids:

    print()
    print(
        'Do NOT continue to Round 2 yet.'
    )

    print(
        'The remaining task IDs are:'
    )

    for x in sorted(
        remaining_ids
    ):

        print(
            ' ',
            x
        )

else:

    # round-1 metrics — local / free

    scored = (
        round1_votes

        .merge(
            pair_order[
                [
                    'pair_id',
                    'correct_vote'
                ]
            ],
            on='pair_id',
            how='left',
            suffixes=(
                '',
                '_from_pair'
            )
        )
    )

    scored[
        'applicable'
    ] = scored[
        'vote'
    ].isin(
        [
            'A',
            'B'
        ]
    )

    scored[
        'correct'
    ] = (

        scored[
            'applicable'
        ]

        &

        scored[
            'vote'
        ].eq(
            scored[
                'correct_vote'
            ]
        )

    )

    scored[
        'incorrect'
    ] = (

        scored[
            'applicable'
        ]

        &

        ~scored[
            'vote'
        ].eq(
            scored[
                'correct_vote'
            ]
        )

    )

    # verify every principle now has exactly 3 judgments.

    counts = (

        scored

        .groupby(
            'representative_id'
        )

        .size()

    )

    if (
        len(counts) != 5000
        or counts.min() != 3
        or counts.max() != 3
    ):

        raise RuntimeError(
            'Round-1 coverage is not exactly '
            '3 judgments for all 5,000 principles.'
        )

    # aggregate principle metrics

    stats = (

        scored

        .groupby(
            'representative_id'
        )

        .agg(

            tested=(
                'task_id',
                'size'
            ),

            applicable=(
                'applicable',
                'sum'
            ),

            correct=(
                'correct',
                'sum'
            ),

            incorrect=(
                'incorrect',
                'sum'
            )

        )

        .reset_index()

    )

    stats[
        'not_applicable'
    ] = (

        stats[
            'tested'
        ]

        -
        stats[
            'applicable'
        ]

    )

    stats[
        'applicability_rate'
    ] = (

        stats[
            'applicable'
        ]

        /
        stats[
            'tested'
        ]

    )

    stats[
        'accuracy_when_applicable'
    ] = (

        stats[
            'correct'
        ]

        /
        stats[
            'applicable'
        ].replace(
            0,
            np.nan
        )

    )

    stats[
        'net_score'
    ] = (

        stats[
            'correct'
        ]

        -
        stats[
            'incorrect'
        ]

    )

    # merge principle text + embedding diagnostics

    representatives_local = pd.read_csv(
        REP_FILE,
        keep_default_na=False
    )

    representatives_local[
        'representative_id'
    ] = (
        representatives_local[
            'representative_id'
        ]
        .astype(str)
    )

    round1_metrics = (

        representatives_local

        .merge(
            stats,
            on='representative_id',
            how='left'
        )

    )

    # original round-1 continuation rule

    # min_applicable_by_round[3] = 1
    # min_round_accuracy = .52

    # then ranking:
    # net score
    # accuracy
    # applicability count
    # embedding accuracy
    # embedding mean margin

    # cap at 500.

    ROUND1_MIN_APPLICABLE = 1
    ROUND1_MIN_ACCURACY = .52
    ROUND1_SURVIVOR_CAP = 500

    eligible = (

        round1_metrics[

            (
                round1_metrics[
                    'applicable'
                ]
                >= ROUND1_MIN_APPLICABLE
            )

            &

            (
                round1_metrics[
                    'accuracy_when_applicable'
                ]
                >= ROUND1_MIN_ACCURACY
            )

        ]

        .copy()

    )

    eligible = (

        eligible

        .sort_values(

            [
                'net_score',
                'accuracy_when_applicable',
                'applicable',
                'embedding_accuracy',
                'embedding_mean_margin'
            ],

            ascending=False

        )

        .reset_index(
            drop=True
        )

    )

    round1_survivors = (

        eligible

        .head(
            ROUND1_SURVIVOR_CAP
        )

        .copy()

    )

    round1_survivors.insert(

        0,

        'round1_rank',

        np.arange(
            1,
            len(
                round1_survivors
            ) + 1
        )

    )

    # save everything

    scored.to_csv(

        OUTPUT_DIR /
        'step4_round1_all_votes.csv',

        index=False
    )

    round1_metrics.to_csv(

        OUTPUT_DIR /
        'step4_round1_principle_metrics.csv',

        index=False
    )

    eligible.to_csv(

        OUTPUT_DIR /
        'step4_round1_eligible_principles.csv',

        index=False
    )

    round1_survivors.to_csv(

        OUTPUT_DIR /
        'step4_round1_survivors_500.csv',

        index=False
    )

    # summary

    print()
    print(
        "========== ROUND-1 RESULTS =========="
    )

    print(
        f'Candidate principles tested: '
        f'{len(round1_metrics):,}'
    )

    print(
        'Judgments per principle:     '
        '3'
    )

    print(
        f'Eligible after thresholds:   '
        f'{len(eligible):,}'
    )

    print(
        f'Continuing to Round 2:       '
        f'{len(round1_survivors):,}'
    )

    print()

    print(
        'Applicability distribution:'
    )

    print(
        round1_metrics[
            'applicable'
        ]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print()

    print(
        'Top 10 Round-1 principles:'
    )

    print(

        round1_survivors[

            [
                'round1_rank',
                'representative_id',
                'cleaned_principle',
                'applicable',
                'accuracy_when_applicable',
                'net_score'
            ]

        ]

        .head(10)

        .to_string(
            index=False
        )

    )

    print()

    print(
        'Saved:'
    )

    print(
        '  step4_round1_all_votes.csv'
    )

    print(
        '  step4_round1_principle_metrics.csv'
    )

    print(
        '  step4_round1_eligible_principles.csv'
    )

    print(
        '  step4_round1_survivors_500.csv'
    )

    print()

    print(
        'NO MORE PAID CALLS WERE MADE '
        'AFTER THE SINGLE REPAIR REQUEST.'
    )

    print(
        "====================================="
    )


In [ ]:

# step 4 round 2 — zero-cost preparation / audit

# round 1:
# 500 survivors already have 3 judgments each.

# round 2:
# add 7 new judgments per survivor.

# result:
# 500 x 7 = 3,500 new judgments
# each survivor reaches exactly 10 total judgments.

# no azure/openai calls are made by this cell.

import math
import numpy as np
import pandas as pd

# safety

RUN_PAID_LLM = False

# settings

ROUND2_SURVIVORS = 500

ROUND1_TOTAL_PER_PRINCIPLE = 3
ROUND2_TOTAL_PER_PRINCIPLE = 10

ROUND2_ADDITIONAL_PER_PRINCIPLE = (
    ROUND2_TOTAL_PER_PRINCIPLE
    -
    ROUND1_TOTAL_PER_PRINCIPLE
)

ROUND2_JUDGMENTS_PER_CALL = 350

# files

SURVIVOR_FILE = (
    OUTPUT_DIR /
    'step4_round1_survivors_500.csv'
)

ROUND1_ASSIGNMENT_FILE = (
    OUTPUT_DIR /
    'step4_round1_assignments.csv'
)

POOL_FILE = (
    OUTPUT_DIR /
    'step4_representative_pair_pool.csv'
)

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

PAIR_EMBED_FILE = (
    OUTPUT_DIR /
    'pair_post_embeddings.npz'
)

PRINCIPLE_EMBED_FILE = (
    OUTPUT_DIR /
    'principle_embeddings.npy'
)

ROUND2_PLAN_FILE = (
    OUTPUT_DIR /
    'step4_round2_assignments.csv'
)

ROUND2_CALL_PLAN_FILE = (
    OUTPUT_DIR /
    'step4_round2_call_plan_350.csv'
)

ROUND2_AUDIT_FILE = (
    OUTPUT_DIR /
    'step4_round2_call_audit_350.csv'
)

# load round-1 survivors

survivors = pd.read_csv(
    SURVIVOR_FILE,
    keep_default_na=False
)

survivors[
    'representative_id'
] = survivors[
    'representative_id'
].astype(str)

if len(survivors) != ROUND2_SURVIVORS:

    raise RuntimeError(
        f'Expected {ROUND2_SURVIVORS:,} Round-1 survivors; '
        f'found {len(survivors):,}.'
    )

print(
    f'Validated Round-1 survivors: '
    f'{len(survivors):,}'
)

# load round-1 assignments

# these are the 3 pairs already used for each principle.
# round 2 must not repeat them.

round1_assignments = pd.read_csv(
    ROUND1_ASSIGNMENT_FILE,
    keep_default_na=False
)

round1_assignments[
    'representative_id'
] = round1_assignments[
    'representative_id'
].astype(str)

round1_assignments[
    'pair_id'
] = round1_assignments[
    'pair_id'
].astype(str)

round1_used_pairs = {

    rid: set(
        g['pair_id'].astype(str)
    )

    for rid, g in (
        round1_assignments
        .groupby('representative_id')
    )

}

# safety:
# every survivor should have exactly 3 prior pairs.

for rid in survivors['representative_id']:

    used = round1_used_pairs.get(
        rid,
        set()
    )

    if len(used) != 3:

        raise RuntimeError(
            f'{rid} has {len(used)} Round-1 pairs; '
            'expected exactly 3.'
        )

# load the same diverse 250-pair pool

pool = pd.read_csv(
    POOL_FILE,
    keep_default_na=False
)

pool['pair_id'] = (
    pool['pair_id']
    .astype(str)
)

if len(pool) != 250:

    raise RuntimeError(
        f'Expected 250 Step-4 pool pairs; '
        f'found {len(pool):,}.'
    )

# load fixed a/b order

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order['pair_id'] = (
    pair_order['pair_id']
    .astype(str)
)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

# load saved embeddings

z = np.load(
    PAIR_EMBED_FILE,
    allow_pickle=True
)

pref_emb = z['preferred']
non_emb = z['nonpreferred']

princ_emb = np.load(
    PRINCIPLE_EMBED_FILE
)

# exact step 3 saved the row in princ_emb used by each rep.

if 'principle_row_index' not in survivors.columns:

    raise RuntimeError(
        'Survivor file is missing principle_row_index.'
    )

rep_indices = (
    survivors[
        'principle_row_index'
    ]
    .astype(int)
    .to_numpy()
)

survivor_emb = princ_emb[
    rep_indices
]

# get embeddings for the 250-pair pool

if 'original_df_index' not in pool.columns:

    raise RuntimeError(
        'Pair-pool file is missing original_df_index.'
    )

pool_indices = (
    pool[
        'original_df_index'
    ]
    .astype(int)
    .to_numpy()
)

pool_pref_emb = (
    pref_emb[
        pool_indices
    ]
)

pool_non_emb = (
    non_emb[
        pool_indices
    ]
)

# relevance matrix

# same relevance concept used for round 1:

# abs(
# similarity(principle, preferred)

# similarity(principle, nonpreferred)
# )

# polarity is included for diagnostic signed margin.

raw_margin = (

    survivor_emb
    @ pool_pref_emb.T

    -

    survivor_emb
    @ pool_non_emb.T
)

polarity = np.where(

    survivors[
        'cleaned_principle'
    ]
    .str.lower()
    .str.startswith('avoid'),

    -1.0,

    1.0

)[:, None]

signed_margin = (
    polarity
    *
    raw_margin
)

relevance = np.abs(
    signed_margin
)

# select next 7 most relevant unused pairs

round2_rows = []

for survivor_i, survivor in (
    survivors
    .reset_index(drop=True)
    .iterrows()
):

    rid = str(
        survivor[
            'representative_id'
        ]
    )

    principle = str(
        survivor[
            'cleaned_principle'
        ]
    )

    previously_used = (
        round1_used_pairs[
            rid
        ]
    )

    candidate_positions = [

        j
        for j in range(len(pool))

        if str(
            pool.iloc[j][
                'pair_id'
            ]
        )
        not in previously_used

    ]

    if (
        len(candidate_positions)
        <
        ROUND2_ADDITIONAL_PER_PRINCIPLE
    ):

        raise RuntimeError(
            f'Not enough unused pairs for {rid}.'
        )

    candidate_positions = np.array(
        candidate_positions,
        dtype=int
    )

    scores = relevance[
        survivor_i,
        candidate_positions
    ]

    order = np.argsort(
        -scores
    )

    chosen_positions = (
        candidate_positions[
            order[
                :ROUND2_ADDITIONAL_PER_PRINCIPLE
            ]
        ]
    )

    for additional_rank, pool_pos in enumerate(
        chosen_positions,
        start=1
    ):

        pool_pos = int(
            pool_pos
        )

        pid = str(
            pool.iloc[
                pool_pos
            ]['pair_id']
        )

        q = pair_lookup.loc[
            pid
        ]

        round2_rows.append({

            'representative_id':
                rid,

            'round1_rank':
                survivor[
                    'round1_rank'
                ],

            'principle':
                principle,

            'pair_id':
                pid,

            'round2_additional_rank':
                additional_rank,

            'embedding_relevance':
                float(
                    relevance[
                        survivor_i,
                        pool_pos
                    ]
                ),

            'signed_embedding_margin':
                float(
                    signed_margin[
                        survivor_i,
                        pool_pos
                    ]
                ),

            'post_a':
                str(
                    q[
                        'post_a'
                    ]
                ),

            'post_b':
                str(
                    q[
                        'post_b'
                    ]
                ),

            'correct_vote':
                str(
                    q[
                        'correct_vote'
                    ]
                )

        })

round2_assignments = pd.DataFrame(
    round2_rows
)

# validation

expected_round2_judgments = (

    ROUND2_SURVIVORS
    *
    ROUND2_ADDITIONAL_PER_PRINCIPLE
)

if len(round2_assignments) != expected_round2_judgments:

    raise RuntimeError(
        f'Expected {expected_round2_judgments:,} '
        f'Round-2 judgments; '
        f'created {len(round2_assignments):,}.'
    )

# no duplicate principle/pair combinations.

duplicate_tasks = (
    round2_assignments
    .duplicated(
        [
            'representative_id',
            'pair_id'
        ]
    )
    .sum()
)

if duplicate_tasks:

    raise RuntimeError(
        f'Found {duplicate_tasks:,} duplicate '
        'Round-2 tasks.'
    )

# confirm no round-1 pair was reused.

reuse_count = 0

for _, r in round2_assignments.iterrows():

    if str(
        r['pair_id']
    ) in round1_used_pairs[
        str(
            r['representative_id']
        )
    ]:

        reuse_count += 1

if reuse_count:

    raise RuntimeError(
        f'Round 2 accidentally reused '
        f'{reuse_count:,} Round-1 tasks.'
    )

# exactly seven additional judgments per survivor.

round2_counts = (

    round2_assignments

    .groupby(
        'representative_id'
    )

    .size()

)

if (
    len(round2_counts) != 500
    or round2_counts.min() != 7
    or round2_counts.max() != 7
):

    raise RuntimeError(
        'Round-2 coverage is not exactly '
        '7 additional judgments per survivor.'
    )

# save assignments

round2_assignments.to_csv(
    ROUND2_PLAN_FILE,
    index=False
)

# build 350-judgment call plan

# pair-first ordering maximizes reddit-text reuse.

round2_call_plan = (

    round2_assignments

    .sort_values(

        [
            'pair_id',
            'embedding_relevance',
            'representative_id'
        ],

        ascending=[
            True,
            False,
            True
        ]

    )

    .reset_index(drop=True)

)

round2_call_plan[
    'planned_call_id'
] = (

    np.arange(
        len(round2_call_plan)
    )

    //
    ROUND2_JUDGMENTS_PER_CALL

    +
    1

)

round2_call_plan.to_csv(
    ROUND2_CALL_PLAN_FILE,
    index=False
)

round2_calls = int(
    round2_call_plan[
        'planned_call_id'
    ].max()
)

# prompt-size audit

audit_rows = []

for call_id, g in (
    round2_call_plan
    .groupby(
        'planned_call_id'
    )
):

    pair_chars = 0

    for pair_id in (
        g[
            'pair_id'
        ]
        .drop_duplicates()
    ):

        q = pair_lookup.loc[
            str(
                pair_id
            )
        ]

        pair_chars += (

            len(
                str(
                    q[
                        'post_a'
                    ]
                )
            )

            +

            len(
                str(
                    q[
                        'post_b'
                    ]
                )
            )

        )

    principle_chars = (

        g[
            'principle'
        ]
        .astype(str)
        .str.len()
        .sum()

    )

    total_chars = (
        pair_chars
        +
        principle_chars
    )

    audit_rows.append({

        'planned_call_id':
            int(
                call_id
            ),

        'judgments':
            int(
                len(g)
            ),

        'distinct_pairs':
            int(
                g[
                    'pair_id'
                ].nunique()
            ),

        'approx_prompt_chars':
            int(
                total_chars
            ),

        'very_rough_prompt_tokens':
            float(
                total_chars / 4.0
            )

    })

round2_audit = pd.DataFrame(
    audit_rows
)

round2_audit.to_csv(
    ROUND2_AUDIT_FILE,
    index=False
)

# pair-reuse diagnostics

pair_counts = (

    round2_assignments

    .groupby(
        'pair_id'
    )

    .size()

)

# final zero-cost audit

print()
print(
    "========== ROUND-2 ZERO-COST PLAN =========="
)

print(
    f'Survivor principles:             '
    f'{len(survivors):,}'
)

print(
    f'Existing judgments/survivor:     '
    f'{ROUND1_TOTAL_PER_PRINCIPLE}'
)

print(
    f'New judgments/survivor:          '
    f'{ROUND2_ADDITIONAL_PER_PRINCIPLE}'
)

print(
    f'Eventual judgments/survivor:     '
    f'{ROUND2_TOTAL_PER_PRINCIPLE}'
)

print()

print(
    f'New Round-2 judgments:           '
    f'{len(round2_assignments):,}'
)

print(
    f'Judgments/call:                  '
    f'{ROUND2_JUDGMENTS_PER_CALL:,}'
)

print(
    f'Exact planned paid calls:        '
    f'{round2_calls:,}'
)

print()

print(
    f'Unique pool pairs used:          '
    f'{round2_assignments.pair_id.nunique():,}'
)

print(
    f'Mean principles per used pair:   '
    f'{pair_counts.mean():.1f}'
)

print(
    f'Max principles on one pair:      '
    f'{pair_counts.max():,}'
)

print()

print(
    'Prompt-size diagnostic:'
)

print(
    f'  mean rough tokens/call:        '
    f'{round2_audit.very_rough_prompt_tokens.mean():,.0f}'
)

print(
    f'  max rough tokens/call:         '
    f'{round2_audit.very_rough_prompt_tokens.max():,.0f}'
)

print()

print(
    'Coverage validation:'
)

print(
    f'  min new judgments/survivor:    '
    f'{round2_counts.min()}'
)

print(
    f'  max new judgments/survivor:    '
    f'{round2_counts.max()}'
)

print(
    f'  Round-1 pair reuses:            '
    f'{reuse_count}'
)

print()

print(
    'Files saved:'
)

print(
    '  step4_round2_assignments.csv'
)

print(
    '  step4_round2_call_plan_350.csv'
)

print(
    '  step4_round2_call_audit_350.csv'
)

print()

print(
    'PAID LLM CALLS MADE: 0'
)

print(
    'RUN_PAID_LLM remains:',
    RUN_PAID_LLM
)

print(
    "============================================"
)


In [ ]:

# step 4 round 2 — run all 10 paid calls

# 500 surviving principles
# 7 additional judgments each
# = 3,500 new judgments

# 350 judgments/call
# = exactly 10 calls

# safety:
# - max 10 paid calls
# - retries=1
# - immediate checkpoint after every batch
# - no automatic omission repair
# - paid calls blocked again afterward

import getpass
import json
from collections import defaultdict

import pandas as pd
from openai import AzureOpenAI
from tqdm.auto import tqdm

# files

PLAN_FILE = (
    OUTPUT_DIR /
    'step4_round2_call_plan_350.csv'
)

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

ROUND2_CKPT = (
    OUTPUT_DIR /
    'step4_round2_shared_checkpoint.jsonl'
)

# safety

RUN_PAID_LLM = False

FIRST_BATCH_ID = 1
LAST_BATCH_ID = 10
MAX_NEW_CALLS_THIS_CELL = 10

# load round-2 plan

plan = pd.read_csv(
    PLAN_FILE,
    keep_default_na=False
)

plan['pair_id'] = (
    plan['pair_id']
    .astype(str)
)

plan['representative_id'] = (
    plan['representative_id']
    .astype(str)
)

if len(plan) != 3500:
    raise RuntimeError(
        f'Expected 3,500 Round-2 tasks; '
        f'found {len(plan):,}.'
    )

if plan['planned_call_id'].nunique() != 10:
    raise RuntimeError(
        f'Expected exactly 10 Round-2 calls; '
        f'found {plan["planned_call_id"].nunique():,}.'
    )

# load fixed a/b order

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order['pair_id'] = (
    pair_order['pair_id']
    .astype(str)
)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

# helper — current completed round-2 tasks

def get_completed_round2_tasks():

    completed = set()

    for record in read_jsonl(
        ROUND2_CKPT
    ):

        for vote_row in record.get(
            'votes',
            []
        ):

            rid = str(
                vote_row.get(
                    'representative_id',
                    ''
                )
            )

            pid = str(
                vote_row.get(
                    'pair_id',
                    ''
                )
            )

            vote = vote_row.get(
                'vote'
            )

            if (
                rid
                and pid
                and vote in {
                    'A',
                    'B',
                    'None'
                }
            ):

                completed.add(
                    rid + '|' + pid
                )

    return completed

completed_before = (
    get_completed_round2_tasks()
)

print(
    "========== ROUND-2 PAID RUN PREVIEW =========="
)

print(
    f'Round-2 judgments already complete: '
    f'{len(completed_before):,} / 3,500'
)

print(
    'Planned batches: 1 through 10'
)

print(
    'Maximum new paid calls: 10'
)

print(
    'Automatic retries: 0'
)

print(
    'Expected new judgments: 3,500'
)

print(
    "=============================================="
)

# same proven 350-judgment prompt

system_prompt = """
Apply each explicit principle independently to its assigned
pair of Reddit posts.

For every principle:
- Vote "A" if Post A better satisfies that principle.
- Vote "B" if Post B better satisfies that principle.
- Vote "None" if the principle is not applicable, does not
  meaningfully distinguish the posts, or neither post is
  clearly favored by that principle.

Judge ONLY by the supplied principle.
Do not judge overall post quality.
Do not use popularity, scores, votes, or outside information.
Treat every principle independently.

Return valid JSON only.
""".strip()

# connect to azure

subscription_key = getpass.getpass(
    'Azure OpenAI API key: '
)

client = AzureOpenAI(
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=subscription_key
)

RUN_PAID_LLM = True

# run all 10 round-2 batches

new_calls_sent = 0

stages_this_run = []

run_expected = 0
run_received = 0
run_missing = 0

stopped_due_to_error = False
error_message = None

try:

    for batch_id in tqdm(
        range(
            FIRST_BATCH_ID,
            LAST_BATCH_ID + 1
        ),
        desc='Round-2 350 batches'
    ):

        if (
            new_calls_sent
            >= MAX_NEW_CALLS_THIS_CELL
        ):
            break

        stage_name = (
            f'step4_round2_350_batch_'
            f'{batch_id:03d}'
        )

        # duplicate-spend protection

        checkpointed_stages = {

            str(
                x.get(
                    'stage',
                    ''
                )
            )

            for x in read_jsonl(
                ROUND2_CKPT
            )

        }

        logged_stages = {

            str(
                x.get(
                    'stage',
                    ''
                )
            )

            for x in read_jsonl(
                CALL_LOG
            )

        }

        if stage_name in checkpointed_stages:

            print(
                f'\nSkipping batch {batch_id}: '
                'already checkpointed.'
            )

            continue

        if stage_name in logged_stages:

            raise RuntimeError(
                f'{stage_name} already appears in CALL_LOG '
                'but is not checkpointed. '
                'Stopping to avoid duplicate spending.'
            )

        # load this batch

        batch_df = (
            plan[
                plan[
                    'planned_call_id'
                ] == batch_id
            ]
            .copy()
        )

        if len(batch_df) != 350:

            raise RuntimeError(
                f'Round-2 batch {batch_id} should contain '
                f'350 tasks; found {len(batch_df):,}.'
            )

        # make sure none are already complete

        completed_now = (
            get_completed_round2_tasks()
        )

        batch_df['task_id'] = (
            batch_df[
                'representative_id'
            ]
            +
            '|'
            +
            batch_df[
                'pair_id'
            ]
        )

        already_done = (
            batch_df[
                'task_id'
            ].isin(
                completed_now
            )
        )

        if already_done.any():

            raise RuntimeError(
                f'Batch {batch_id} contains '
                f'{already_done.sum():,} '
                'already-completed tasks.'
            )

        # build shared-pair payload

        pair_payload = []

        for pair_id, g in (
            batch_df.groupby(
                'pair_id',
                sort=False
            )
        ):

            pid = str(
                pair_id
            )

            q = pair_lookup.loc[
                pid
            ]

            principles_for_pair = [

                {
                    'id':
                        str(
                            r[
                                'representative_id'
                            ]
                        ),

                    'principle':
                        str(
                            r[
                                'principle'
                            ]
                        )
                }

                for _, r
                in g.iterrows()

            ]

            pair_payload.append({

                'pair_id':
                    pid,

                'post_a':
                    str(
                        q['post_a']
                    ),

                'post_b':
                    str(
                        q['post_b']
                    ),

                'principles':
                    principles_for_pair

            })

        user_prompt = f"""
Evaluate EVERY supplied principle.

You must return:
- every supplied pair_id exactly once
- every supplied principle ID exactly once

Do not omit any principle.

Return ONLY this compact JSON structure:

{{
  "results": [
    {{
      "pair_id": "PAIR_ID",
      "votes": {{
        "C0000": "A",
        "C0001": "B",
        "C0002": "None"
      }}
    }}
  ]
}}

Allowed vote strings are ONLY:

"A"
"B"
"None"

DATA:

{json.dumps(pair_payload, ensure_ascii=False)}
""".strip()

        # one paid call

        try:

            result = chat_json(

                system_prompt,

                user_prompt,

                stage_name,

                max_tokens=10000,

                retries=1

            )

            new_calls_sent += 1

            stages_this_run.append(
                stage_name
            )

        except Exception as exc:

            stopped_due_to_error = True

            error_message = (
                f'{type(exc).__name__}: {exc}'
            )

            print()
            print(
                f'Stopping safely at '
                f'Round-2 batch {batch_id}.'
            )

            print(
                error_message
            )

            break

        # normalize returned votes

        expected_by_pair = defaultdict(
            set
        )

        for _, r in batch_df.iterrows():

            expected_by_pair[
                str(r['pair_id'])
            ].add(
                str(
                    r[
                        'representative_id'
                    ]
                )
            )

        returned_votes = defaultdict(
            dict
        )

        for item in result.get(
            'results',
            []
        ):

            pid = str(
                item.get(
                    'pair_id',
                    ''
                )
            )

            if pid not in expected_by_pair:
                continue

            votes = item.get(
                'votes',
                {}
            )

            if not isinstance(
                votes,
                dict
            ):
                continue

            for rid, raw_vote in votes.items():

                rid = str(
                    rid
                )

                if rid not in expected_by_pair[
                    pid
                ]:
                    continue

                if not isinstance(
                    raw_vote,
                    str
                ):
                    continue

                v = (
                    raw_vote
                    .strip()
                    .upper()
                )

                if v == 'A':

                    returned_votes[
                        pid
                    ][rid] = 'A'

                elif v == 'B':

                    returned_votes[
                        pid
                    ][rid] = 'B'

                elif v == 'NONE':

                    returned_votes[
                        pid
                    ][rid] = 'None'

        # build valid rows

        valid_rows = []
        missing_tasks = []

        for _, r in batch_df.iterrows():

            pid = str(
                r['pair_id']
            )

            rid = str(
                r[
                    'representative_id'
                ]
            )

            vote = (
                returned_votes
                .get(
                    pid,
                    {}
                )
                .get(
                    rid
                )
            )

            if vote is None:

                missing_tasks.append(
                    rid + '|' + pid
                )

                continue

            correct_vote = str(
                pair_lookup.loc[
                    pid,
                    'correct_vote'
                ]
            )

            if vote == 'None':

                outcome = (
                    'not_applicable'
                )

            elif vote == correct_vote:

                outcome = (
                    'correct'
                )

            else:

                outcome = (
                    'incorrect'
                )

            valid_rows.append({

                'representative_id':
                    rid,

                'pair_id':
                    pid,

                'vote':
                    vote,

                'correct_vote':
                    correct_vote,

                'outcome':
                    outcome

            })

        # checkpoint immediately

        append_jsonl(

            ROUND2_CKPT,

            [{

                'stage':
                    stage_name,

                'batch_scheme':
                    'round2_350',

                'planned_call_id':
                    int(
                        batch_id
                    ),

                'expected_judgments':
                    int(
                        len(batch_df)
                    ),

                'received_judgments':
                    int(
                        len(valid_rows)
                    ),

                'missing_judgments':
                    int(
                        len(missing_tasks)
                    ),

                'votes':
                    valid_rows,

                'missing_task_ids':
                    missing_tasks

            }]

        )

        run_expected += (
            len(batch_df)
        )

        run_received += (
            len(valid_rows)
        )

        run_missing += (
            len(missing_tasks)
        )

        print(
            f'\nBatch {batch_id}: '
            f'{len(valid_rows)}/{len(batch_df)} '
            f'valid; '
            f'{len(missing_tasks)} missing.'
        )

finally:

    # always disable paid calls

    RUN_PAID_LLM = False

    client = None

# token usage

call_log = read_jsonl(
    CALL_LOG
)

run_logs = [

    x

    for x in call_log

    if str(
        x.get(
            'stage',
            ''
        )
    ) in set(
        stages_this_run
    )

]

prompt_tokens = sum(

    x.get(
        'prompt_tokens'
    )
    or 0

    for x in run_logs

)

completion_tokens = sum(

    x.get(
        'completion_tokens'
    )
    or 0

    for x in run_logs

)

total_tokens = sum(

    x.get(
        'total_tokens'
    )
    or 0

    for x in run_logs

)

# cumulative round-2 status

completed_after = (
    get_completed_round2_tasks()
)

total_complete = len(
    completed_after
)

total_remaining = (
    3500
    -
    total_complete
)

coverage = (

    run_received
    /
    run_expected

    if run_expected

    else 0
)

print()
print(
    "========== ROUND-2 PAID RUN RESULT =========="
)

print(
    f'Paid calls completed:       '
    f'{new_calls_sent:,} / 10'
)

print(
    f'Expected judgments:         '
    f'{run_expected:,}'
)

print(
    f'Valid judgments returned:   '
    f'{run_received:,}'
)

print(
    f'Missing judgments:          '
    f'{run_missing:,}'
)

print(
    f'Coverage:                   '
    f'{coverage:.2%}'
)

print()

print(
    'TOKEN USAGE:'
)

print(
    f'  Prompt tokens:     '
    f'{prompt_tokens:,}'
)

print(
    f'  Completion tokens: '
    f'{completion_tokens:,}'
)

print(
    f'  Total tokens:      '
    f'{total_tokens:,}'
)

print()

print(
    'CUMULATIVE ROUND-2 STATUS:'
)

print(
    f'  Complete judgments: '
    f'{total_complete:,} / 3,500'
)

print(
    f'  Still missing:      '
    f'{total_remaining:,}'
)

print()

if stopped_due_to_error:

    print(
        'RUN STOPPED EARLY:'
    )

    print(
        error_message
    )

else:

    print(
        'All 10 Round-2 batches finished.'
    )

print()

print(
    'PAID LLM CALLS ARE BLOCKED AGAIN:',
    not RUN_PAID_LLM
)

print(
    "============================================"
)


In [ ]:

# step 4 round 2 results
# + select top 100
# + prepare final round 3

# zero paid llm calls

# round 1: 3 judgments
# round 2: +7 = 10 total

# original continuation rule at 10:
# applicable >= 3
# accuracy >= .52
# rank eligible principles
# keep top 100

# round 3:
# +15 new judgments
# = 25 total per surviving principle

import math
import numpy as np
import pandas as pd

# safety

RUN_PAID_LLM = False

# files

ROUND1_CKPT = (
    OUTPUT_DIR /
    'step4_round1_shared_checkpoint.jsonl'
)

ROUND2_CKPT = (
    OUTPUT_DIR /
    'step4_round2_shared_checkpoint.jsonl'
)

ROUND1_ASSIGNMENT_FILE = (
    OUTPUT_DIR /
    'step4_round1_assignments.csv'
)

ROUND2_ASSIGNMENT_FILE = (
    OUTPUT_DIR /
    'step4_round2_assignments.csv'
)

ROUND1_SURVIVOR_FILE = (
    OUTPUT_DIR /
    'step4_round1_survivors_500.csv'
)

POOL_FILE = (
    OUTPUT_DIR /
    'step4_representative_pair_pool.csv'
)

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

PAIR_EMBED_FILE = (
    OUTPUT_DIR /
    'pair_post_embeddings.npz'
)

PRINCIPLE_EMBED_FILE = (
    OUTPUT_DIR /
    'principle_embeddings.npy'
)

# helper — extract valid votes

def collect_votes(path):

    rows = []

    for record in read_jsonl(path):

        for r in record.get(
            'votes',
            []
        ):

            rid = str(
                r.get(
                    'representative_id',
                    ''
                )
            )

            pid = str(
                r.get(
                    'pair_id',
                    ''
                )
            )

            vote = r.get(
                'vote'
            )

            if (
                rid
                and pid
                and vote in {
                    'A',
                    'B',
                    'None'
                }
            ):

                rows.append({

                    'representative_id':
                        rid,

                    'pair_id':
                        pid,

                    'task_id':
                        rid + '|' + pid,

                    'vote':
                        vote

                })

    out = pd.DataFrame(rows)

    if len(out):

        out = (
            out
            .drop_duplicates(
                'task_id',
                keep='first'
            )
            .reset_index(drop=True)
        )

    return out

# load the 500 round-1 survivors

survivors500 = pd.read_csv(
    ROUND1_SURVIVOR_FILE,
    keep_default_na=False
)

survivors500[
    'representative_id'
] = survivors500[
    'representative_id'
].astype(str)

if len(survivors500) != 500:

    raise RuntimeError(
        f'Expected 500 Round-1 survivors; '
        f'found {len(survivors500):,}.'
    )

survivor_ids = set(
    survivors500[
        'representative_id'
    ]
)

# load round-1 + round-2 votes

r1_votes_all = collect_votes(
    ROUND1_CKPT
)

r2_votes = collect_votes(
    ROUND2_CKPT
)

# only the 500 survivors' round-1 votes matter here.

r1_votes = (

    r1_votes_all[
        r1_votes_all[
            'representative_id'
        ].isin(
            survivor_ids
        )
    ]

    .copy()

)

print(
    "========== 10-JUDGMENT INPUT CHECK =========="
)

print(
    f'Round-1 votes for 500 survivors: '
    f'{len(r1_votes):,}'
)

print(
    f'Round-2 votes:                   '
    f'{len(r2_votes):,}'
)

print(
    f'Combined votes:                  '
    f'{len(r1_votes) + len(r2_votes):,}'
)

print(
    "============================================="
)

if len(r1_votes) != 1500:

    raise RuntimeError(
        f'Expected 1,500 Round-1 survivor votes; '
        f'found {len(r1_votes):,}.'
    )

if len(r2_votes) != 3500:

    raise RuntimeError(
        f'Expected 3,500 Round-2 votes; '
        f'found {len(r2_votes):,}.'
    )

all10 = pd.concat(
    [
        r1_votes,
        r2_votes
    ],
    ignore_index=True
)

if all10[
    'task_id'
].duplicated().any():

    raise RuntimeError(
        'Duplicate principle/pair judgments '
        'exist across Round 1 and Round 2.'
    )

# load correct a/b answers

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order[
    'pair_id'
] = pair_order[
    'pair_id'
].astype(str)

scored = (

    all10

    .merge(

        pair_order[
            [
                'pair_id',
                'correct_vote'
            ]
        ],

        on='pair_id',

        how='left'

    )

)

scored[
    'applicable'
] = scored[
    'vote'
].isin(
    [
        'A',
        'B'
    ]
)

scored[
    'correct'
] = (

    scored[
        'applicable'
    ]

    &

    scored[
        'vote'
    ].eq(
        scored[
            'correct_vote'
        ]
    )

)

scored[
    'incorrect'
] = (

    scored[
        'applicable'
    ]

    &

    ~scored[
        'vote'
    ].eq(
        scored[
            'correct_vote'
        ]
    )

)

# verify exactly 10 judgments per survivor

counts10 = (

    scored

    .groupby(
        'representative_id'
    )

    .size()

)

if (
    len(counts10) != 500
    or counts10.min() != 10
    or counts10.max() != 10
):

    raise RuntimeError(
        'Not every Round-2 survivor has '
        'exactly 10 total judgments.'
    )

# calculate 10-judgment metrics

stats10 = (

    scored

    .groupby(
        'representative_id'
    )

    .agg(

        tested=(
            'task_id',
            'size'
        ),

        applicable=(
            'applicable',
            'sum'
        ),

        correct=(
            'correct',
            'sum'
        ),

        incorrect=(
            'incorrect',
            'sum'
        )

    )

    .reset_index()

)

stats10[
    'not_applicable'
] = (

    stats10[
        'tested'
    ]

    -
    stats10[
        'applicable'
    ]

)

stats10[
    'applicability_rate'
] = (

    stats10[
        'applicable'
    ]

    /
    stats10[
        'tested'
    ]

)

stats10[
    'accuracy_when_applicable'
] = (

    stats10[
        'correct'
    ]

    /
    stats10[
        'applicable'
    ].replace(
        0,
        np.nan
    )

)

stats10[
    'net_score'
] = (

    stats10[
        'correct'
    ]

    -
    stats10[
        'incorrect'
    ]

)

# merge principle / embedding information

round2_metrics = (

    survivors500

    .merge(
        stats10,
        on='representative_id',
        how='left'
    )

)

# original 10-judgment continuation rule

# applicable >= 3
# accuracy >= .52
# cap = 100

MIN_APPLICABLE_10 = 3
MIN_ACCURACY_10 = .52
ROUND3_SURVIVOR_CAP = 100

eligible10 = (

    round2_metrics[

        (
            round2_metrics[
                'applicable'
            ]
            >= MIN_APPLICABLE_10
        )

        &

        (
            round2_metrics[
                'accuracy_when_applicable'
            ]
            >= MIN_ACCURACY_10
        )

    ]

    .copy()

)

# same ranking structure as original notebook.

eligible10 = (

    eligible10

    .sort_values(

        [
            'net_score',
            'accuracy_when_applicable',
            'applicable',
            'embedding_accuracy',
            'embedding_mean_margin'
        ],

        ascending=False

    )

    .reset_index(drop=True)

)

top100 = (

    eligible10

    .head(
        ROUND3_SURVIVOR_CAP
    )

    .copy()

)

top100.insert(

    0,

    'round2_rank',

    np.arange(
        1,
        len(top100) + 1
    )

)

# save round-2 results

scored.to_csv(

    OUTPUT_DIR /
    'step4_round2_all_10_votes.csv',

    index=False

)

round2_metrics.to_csv(

    OUTPUT_DIR /
    'step4_round2_metrics_10_total.csv',

    index=False

)

eligible10.to_csv(

    OUTPUT_DIR /
    'step4_round2_eligible_principles.csv',

    index=False

)

top100.to_csv(

    OUTPUT_DIR /
    'step4_round2_survivors_100.csv',

    index=False

)

# round 3 preparation

# each top principle already has:
# 3 round-1 pairs
# + 7 round-2 pairs
# = 10 unique pairs

# select 15 new relevant pairs from the same diverse pool.

r1_assign = pd.read_csv(
    ROUND1_ASSIGNMENT_FILE,
    keep_default_na=False
)

r2_assign = pd.read_csv(
    ROUND2_ASSIGNMENT_FILE,
    keep_default_na=False
)

for frame in [
    r1_assign,
    r2_assign
]:

    frame[
        'representative_id'
    ] = frame[
        'representative_id'
    ].astype(str)

    frame[
        'pair_id'
    ] = frame[
        'pair_id'
    ].astype(str)

# all previously used pairs for each principle.

previous_assignments = pd.concat(

    [

        r1_assign[
            [
                'representative_id',
                'pair_id'
            ]
        ],

        r2_assign[
            [
                'representative_id',
                'pair_id'
            ]
        ]

    ],

    ignore_index=True

)

used_pairs = {

    rid: set(
        g[
            'pair_id'
        ].astype(str)
    )

    for rid, g
    in previous_assignments.groupby(
        'representative_id'
    )

}

for rid in top100[
    'representative_id'
]:

    if len(
        used_pairs.get(
            rid,
            set()
        )
    ) != 10:

        raise RuntimeError(
            f'{rid} does not have exactly '
            '10 previously used pairs.'
        )

# load pair pool + embeddings

pool = pd.read_csv(
    POOL_FILE,
    keep_default_na=False
)

pool[
    'pair_id'
] = pool[
    'pair_id'
].astype(str)

z = np.load(
    PAIR_EMBED_FILE,
    allow_pickle=True
)

pref_emb = z[
    'preferred'
]

non_emb = z[
    'nonpreferred'
]

princ_emb = np.load(
    PRINCIPLE_EMBED_FILE
)

pool_indices = (
    pool[
        'original_df_index'
    ]
    .astype(int)
    .to_numpy()
)

pool_pref = pref_emb[
    pool_indices
]

pool_non = non_emb[
    pool_indices
]

top_indices = (

    top100[
        'principle_row_index'
    ]
    .astype(int)
    .to_numpy()

)

top_emb = princ_emb[
    top_indices
]

# embedding relevance

raw_margin = (

    top_emb
    @ pool_pref.T

    -

    top_emb
    @ pool_non.T

)

polarity = np.where(

    top100[
        'cleaned_principle'
    ]
    .str.lower()
    .str.startswith(
        'avoid'
    ),

    -1.0,

    1.0

)[:, None]

signed_margin = (

    polarity
    *
    raw_margin

)

relevance = np.abs(
    signed_margin
)

pair_lookup = (
    pair_order
    .set_index(
        'pair_id'
    )
)

# select 15 unused pairs for each top principle

ROUND3_ADDITIONAL = 15

round3_rows = []

for principle_i, p in (
    top100
    .reset_index(drop=True)
    .iterrows()
):

    rid = str(
        p[
            'representative_id'
        ]
    )

    prior = used_pairs[
        rid
    ]

    candidate_positions = [

        j

        for j in range(
            len(pool)
        )

        if str(
            pool.iloc[j][
                'pair_id'
            ]
        )
        not in prior

    ]

    candidate_positions = np.array(
        candidate_positions,
        dtype=int
    )

    scores = relevance[
        principle_i,
        candidate_positions
    ]

    best_order = np.argsort(
        -scores
    )

    chosen = candidate_positions[

        best_order[
            :ROUND3_ADDITIONAL
        ]

    ]

    for rank, pos in enumerate(
        chosen,
        start=1
    ):

        pos = int(
            pos
        )

        pid = str(
            pool.iloc[
                pos
            ][
                'pair_id'
            ]
        )

        q = pair_lookup.loc[
            pid
        ]

        round3_rows.append({

            'representative_id':
                rid,

            'round2_rank':
                p[
                    'round2_rank'
                ],

            'principle':
                str(
                    p[
                        'cleaned_principle'
                    ]
                ),

            'pair_id':
                pid,

            'round3_additional_rank':
                rank,

            'embedding_relevance':
                float(
                    relevance[
                        principle_i,
                        pos
                    ]
                ),

            'signed_embedding_margin':
                float(
                    signed_margin[
                        principle_i,
                        pos
                    ]
                ),

            'post_a':
                str(
                    q[
                        'post_a'
                    ]
                ),

            'post_b':
                str(
                    q[
                        'post_b'
                    ]
                ),

            'correct_vote':
                str(
                    q[
                        'correct_vote'
                    ]
                )

        })

round3_assignments = pd.DataFrame(
    round3_rows
)

# validate round 3

expected_round3 = (
    len(top100)
    *
    ROUND3_ADDITIONAL
)

if len(round3_assignments) != expected_round3:

    raise RuntimeError(
        f'Expected {expected_round3:,} '
        f'Round-3 judgments; '
        f'created {len(round3_assignments):,}.'
    )

r3_counts = (

    round3_assignments

    .groupby(
        'representative_id'
    )

    .size()

)

if (
    r3_counts.min() != 15
    or
    r3_counts.max() != 15
):

    raise RuntimeError(
        'Not every Round-3 principle '
        'received exactly 15 new tasks.'
    )

# confirm no previous pair reuse.

reuse_count = 0

for _, r in (
    round3_assignments.iterrows()
):

    if str(
        r[
            'pair_id'
        ]
    ) in used_pairs[
        str(
            r[
                'representative_id'
            ]
        )
    ]:

        reuse_count += 1

if reuse_count:

    raise RuntimeError(
        f'Round 3 reused '
        f'{reuse_count} earlier pairs.'
    )

# save round-3 assignments

round3_assignments.to_csv(

    OUTPUT_DIR /
    'step4_round3_assignments.csv',

    index=False

)

# build final 350-judgment call plan

ROUND3_JUDGMENTS_PER_CALL = 350

round3_plan = (

    round3_assignments

    .sort_values(

        [
            'pair_id',
            'embedding_relevance',
            'representative_id'
        ],

        ascending=[
            True,
            False,
            True
        ]

    )

    .reset_index(drop=True)

)

round3_plan[
    'planned_call_id'
] = (

    np.arange(
        len(round3_plan)
    )

    //
    ROUND3_JUDGMENTS_PER_CALL

    +
    1

)

round3_plan.to_csv(

    OUTPUT_DIR /
    'step4_round3_call_plan_350.csv',

    index=False

)

round3_calls = int(
    round3_plan[
        'planned_call_id'
    ].max()
)

# prompt-size audit

audit_rows = []

for call_id, g in (
    round3_plan
    .groupby(
        'planned_call_id'
    )
):

    pair_chars = 0

    for pid in (
        g[
            'pair_id'
        ]
        .drop_duplicates()
    ):

        q = pair_lookup.loc[
            str(pid)
        ]

        pair_chars += (

            len(
                str(
                    q[
                        'post_a'
                    ]
                )
            )

            +

            len(
                str(
                    q[
                        'post_b'
                    ]
                )
            )

        )

    principle_chars = (

        g[
            'principle'
        ]
        .astype(str)
        .str.len()
        .sum()

    )

    total_chars = (
        pair_chars
        +
        principle_chars
    )

    audit_rows.append({

        'planned_call_id':
            int(call_id),

        'judgments':
            int(len(g)),

        'distinct_pairs':
            int(
                g[
                    'pair_id'
                ].nunique()
            ),

        'very_rough_prompt_tokens':
            float(
                total_chars / 4
            )

    })

audit = pd.DataFrame(
    audit_rows
)

audit.to_csv(

    OUTPUT_DIR /
    'step4_round3_call_audit_350.csv',

    index=False

)

# final summary

print()
print(
    "========== ROUND-2 RESULTS =========="
)

print(
    f'Principles evaluated at 10 pairs: '
    f'{len(round2_metrics):,}'
)

print(
    f'Eligible after thresholds:       '
    f'{len(eligible10):,}'
)

print(
    f'Continuing to final Round 3:      '
    f'{len(top100):,}'
)

print()

print(
    'Top-100 selection threshold:'
)

print(
    '  minimum applicable = 3'
)

print(
    '  minimum accuracy   = .52'
)

print()

print(
    "========== ROUND-3 ZERO-COST PLAN =========="
)

print(
    f'Principles entering Round 3:      '
    f'{len(top100):,}'
)

print(
    f'Existing judgments/principle:     '
    f'10'
)

print(
    f'New judgments/principle:          '
    f'15'
)

print(
    f'Eventual judgments/principle:     '
    f'25'
)

print()

print(
    f'New Round-3 judgments:            '
    f'{len(round3_assignments):,}'
)

print(
    f'Exact planned paid calls:         '
    f'{round3_calls:,}'
)

print(
    f'Round-1/2 pair reuses:            '
    f'{reuse_count}'
)

print()

print(
    'Prompt-size diagnostic:'
)

print(
    f'  mean rough tokens/call: '
    f'{audit.very_rough_prompt_tokens.mean():,.0f}'
)

print(
    f'  max rough tokens/call:  '
    f'{audit.very_rough_prompt_tokens.max():,.0f}'
)

print()

print(
    'PAID LLM CALLS MADE: 0'
)

print(
    'RUN_PAID_LLM remains:',
    RUN_PAID_LLM
)

print(
    "============================================"
)


In [ ]:

# fix round-2 merge + continue to round-3 plan

# zero paid llm calls

# the previous cell failed because survivors500 already
# contained round-1 metric columns such as "applicable".
# merging stats10 created applicable_x / applicable_y.

# this cell keeps the new 10-judgment metrics and continues.

import numpy as np
import pandas as pd

RUN_PAID_LLM = False

# 1. fix the metric merge

OLD_METRIC_COLUMNS = [
    'tested',
    'applicable',
    'correct',
    'incorrect',
    'not_applicable',
    'applicability_rate',
    'accuracy_when_applicable',
    'net_score'
]

# remove round-1 versions before merging the new
# 10-judgment metrics.
survivor_base = survivors500.drop(
    columns=[
        c
        for c in OLD_METRIC_COLUMNS
        if c in survivors500.columns
    ],
    errors='ignore'
).copy()

round2_metrics = (
    survivor_base
    .merge(
        stats10,
        on='representative_id',
        how='left',
        validate='one_to_one'
    )
)

required_metric_columns = [
    'tested',
    'applicable',
    'correct',
    'incorrect',
    'accuracy_when_applicable',
    'net_score'
]

missing_metric_columns = [
    c
    for c in required_metric_columns
    if c not in round2_metrics.columns
]

if missing_metric_columns:
    raise RuntimeError(
        f'Missing expected 10-judgment metrics: '
        f'{missing_metric_columns}'
    )

if len(round2_metrics) != 500:
    raise RuntimeError(
        f'Expected 500 principles after corrected merge; '
        f'found {len(round2_metrics):,}.'
    )

print(
    'Corrected 10-judgment metric merge:',
    len(round2_metrics)
)

# 2. apply round-2 continuation rule

# at 10 judgments:
# applicable >= 3
# accuracy >= .52
# keep at most 100

MIN_APPLICABLE_10 = 3
MIN_ACCURACY_10 = 0.52
ROUND3_SURVIVOR_CAP = 100

eligible10 = (
    round2_metrics[
        (
            round2_metrics['applicable']
            >= MIN_APPLICABLE_10
        )
        &
        (
            round2_metrics[
                'accuracy_when_applicable'
            ]
            >= MIN_ACCURACY_10
        )
    ]
    .copy()
)

# same ranking rule as before.
sort_columns = [
    'net_score',
    'accuracy_when_applicable',
    'applicable'
]

# add embedding diagnostics if present.
for c in [
    'embedding_accuracy',
    'embedding_mean_margin'
]:
    if c in eligible10.columns:
        sort_columns.append(c)

eligible10 = (
    eligible10
    .sort_values(
        sort_columns,
        ascending=[False] * len(sort_columns)
    )
    .reset_index(drop=True)
)

top100 = (
    eligible10
    .head(
        ROUND3_SURVIVOR_CAP
    )
    .copy()
)

top100.insert(
    0,
    'round2_rank',
    np.arange(
        1,
        len(top100) + 1
    )
)

# 3. save round-2 results

scored.to_csv(
    OUTPUT_DIR /
    'step4_round2_all_10_votes.csv',
    index=False
)

round2_metrics.to_csv(
    OUTPUT_DIR /
    'step4_round2_metrics_10_total.csv',
    index=False
)

eligible10.to_csv(
    OUTPUT_DIR /
    'step4_round2_eligible_principles.csv',
    index=False
)

top100.to_csv(
    OUTPUT_DIR /
    'step4_round2_survivors_100.csv',
    index=False
)

print()
print(
    "========== ROUND-2 RESULTS =========="
)

print(
    f'Principles evaluated at 10 pairs: '
    f'{len(round2_metrics):,}'
)

print(
    f'Eligible after thresholds:       '
    f'{len(eligible10):,}'
)

print(
    f'Continuing to final Round 3:      '
    f'{len(top100):,}'
)

print(
    "====================================="
)

if len(top100) == 0:
    raise RuntimeError(
        'No principles survived Round 2.'
    )

# 4. prepare round 3

# each surviving principle already has 10 unique pairs.
# give it 15 new pairs -> 25 total.

ROUND1_ASSIGNMENT_FILE = (
    OUTPUT_DIR /
    'step4_round1_assignments.csv'
)

ROUND2_ASSIGNMENT_FILE = (
    OUTPUT_DIR /
    'step4_round2_assignments.csv'
)

POOL_FILE = (
    OUTPUT_DIR /
    'step4_representative_pair_pool.csv'
)

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

PAIR_EMBED_FILE = (
    OUTPUT_DIR /
    'pair_post_embeddings.npz'
)

PRINCIPLE_EMBED_FILE = (
    OUTPUT_DIR /
    'principle_embeddings.npy'
)

r1_assign = pd.read_csv(
    ROUND1_ASSIGNMENT_FILE,
    keep_default_na=False
)

r2_assign = pd.read_csv(
    ROUND2_ASSIGNMENT_FILE,
    keep_default_na=False
)

for frame in [
    r1_assign,
    r2_assign
]:
    frame['representative_id'] = (
        frame['representative_id']
        .astype(str)
    )

    frame['pair_id'] = (
        frame['pair_id']
        .astype(str)
    )

previous_assignments = pd.concat(
    [
        r1_assign[
            [
                'representative_id',
                'pair_id'
            ]
        ],
        r2_assign[
            [
                'representative_id',
                'pair_id'
            ]
        ]
    ],
    ignore_index=True
)

used_pairs = {
    rid: set(
        g['pair_id'].astype(str)
    )
    for rid, g in (
        previous_assignments
        .groupby('representative_id')
    )
}

for rid in top100['representative_id']:

    if len(
        used_pairs.get(
            str(rid),
            set()
        )
    ) != 10:

        raise RuntimeError(
            f'{rid} does not have exactly '
            '10 previously used pairs.'
        )

# 5. load pool / embeddings

pool = pd.read_csv(
    POOL_FILE,
    keep_default_na=False
)

pool['pair_id'] = (
    pool['pair_id']
    .astype(str)
)

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order['pair_id'] = (
    pair_order['pair_id']
    .astype(str)
)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

z = np.load(
    PAIR_EMBED_FILE,
    allow_pickle=True
)

pref_emb = z['preferred']
non_emb = z['nonpreferred']

princ_emb = np.load(
    PRINCIPLE_EMBED_FILE
)

pool_indices = (
    pool[
        'original_df_index'
    ]
    .astype(int)
    .to_numpy()
)

pool_pref = pref_emb[
    pool_indices
]

pool_non = non_emb[
    pool_indices
]

top_indices = (
    top100[
        'principle_row_index'
    ]
    .astype(int)
    .to_numpy()
)

top_emb = princ_emb[
    top_indices
]

# 6. compute embedding relevance

raw_margin = (
    top_emb @ pool_pref.T
    -
    top_emb @ pool_non.T
)

polarity = np.where(
    top100[
        'cleaned_principle'
    ]
    .str.lower()
    .str.startswith('avoid'),
    -1.0,
    1.0
)[:, None]

signed_margin = (
    polarity * raw_margin
)

relevance = np.abs(
    signed_margin
)

# 7. select 15 new pairs per principle

ROUND3_ADDITIONAL = 15

round3_rows = []

for principle_i, p in (
    top100
    .reset_index(drop=True)
    .iterrows()
):

    rid = str(
        p['representative_id']
    )

    prior = used_pairs[
        rid
    ]

    candidate_positions = np.array(
        [
            j
            for j in range(len(pool))
            if str(
                pool.iloc[j]['pair_id']
            ) not in prior
        ],
        dtype=int
    )

    if len(candidate_positions) < 15:
        raise RuntimeError(
            f'Not enough unused pairs for {rid}.'
        )

    scores = relevance[
        principle_i,
        candidate_positions
    ]

    order = np.argsort(
        -scores
    )

    chosen = candidate_positions[
        order[:ROUND3_ADDITIONAL]
    ]

    for rank, pos in enumerate(
        chosen,
        start=1
    ):

        pos = int(pos)

        pid = str(
            pool.iloc[pos]['pair_id']
        )

        q = pair_lookup.loc[
            pid
        ]

        round3_rows.append({
            'representative_id':
                rid,

            'round2_rank':
                p['round2_rank'],

            'principle':
                str(
                    p['cleaned_principle']
                ),

            'pair_id':
                pid,

            'round3_additional_rank':
                rank,

            'embedding_relevance':
                float(
                    relevance[
                        principle_i,
                        pos
                    ]
                ),

            'signed_embedding_margin':
                float(
                    signed_margin[
                        principle_i,
                        pos
                    ]
                ),

            'post_a':
                str(q['post_a']),

            'post_b':
                str(q['post_b']),

            'correct_vote':
                str(q['correct_vote'])
        })

round3_assignments = pd.DataFrame(
    round3_rows
)

expected_round3 = (
    len(top100) * 15
)

if len(round3_assignments) != expected_round3:
    raise RuntimeError(
        f'Expected {expected_round3:,} '
        f'Round-3 tasks; '
        f'created {len(round3_assignments):,}.'
    )

r3_counts = (
    round3_assignments
    .groupby('representative_id')
    .size()
)

if (
    r3_counts.min() != 15
    or
    r3_counts.max() != 15
):
    raise RuntimeError(
        'Round-3 coverage is not '
        'exactly 15 per principle.'
    )

# confirm no reuse of earlier pairs.
reuse_count = 0

for _, r in round3_assignments.iterrows():

    rid = str(
        r['representative_id']
    )

    pid = str(
        r['pair_id']
    )

    if pid in used_pairs[rid]:
        reuse_count += 1

if reuse_count:
    raise RuntimeError(
        f'Round 3 reused '
        f'{reuse_count} previous pairs.'
    )

round3_assignments.to_csv(
    OUTPUT_DIR /
    'step4_round3_assignments.csv',
    index=False
)

# 8. build 350-judgment call plan

ROUND3_JUDGMENTS_PER_CALL = 350

round3_plan = (
    round3_assignments
    .sort_values(
        [
            'pair_id',
            'embedding_relevance',
            'representative_id'
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

round3_plan[
    'planned_call_id'
] = (
    np.arange(
        len(round3_plan)
    )
    //
    ROUND3_JUDGMENTS_PER_CALL
    +
    1
)

round3_plan.to_csv(
    OUTPUT_DIR /
    'step4_round3_call_plan_350.csv',
    index=False
)

round3_calls = int(
    round3_plan[
        'planned_call_id'
    ].max()
)

# 9. prompt-size audit

audit_rows = []

for call_id, g in (
    round3_plan
    .groupby('planned_call_id')
):

    pair_chars = 0

    for pid in (
        g['pair_id']
        .drop_duplicates()
    ):

        q = pair_lookup.loc[
            str(pid)
        ]

        pair_chars += (
            len(str(q['post_a']))
            +
            len(str(q['post_b']))
        )

    principle_chars = (
        g['principle']
        .astype(str)
        .str.len()
        .sum()
    )

    total_chars = (
        pair_chars
        +
        principle_chars
    )

    audit_rows.append({
        'planned_call_id':
            int(call_id),

        'judgments':
            int(len(g)),

        'distinct_pairs':
            int(
                g['pair_id']
                .nunique()
            ),

        'very_rough_prompt_tokens':
            float(
                total_chars / 4
            )
    })

audit = pd.DataFrame(
    audit_rows
)

audit.to_csv(
    OUTPUT_DIR /
    'step4_round3_call_audit_350.csv',
    index=False
)

# final summary

print()
print(
    "========== ROUND-3 ZERO-COST PLAN =========="
)

print(
    f'Principles entering Round 3:      '
    f'{len(top100):,}'
)

print(
    'Existing judgments/principle:     10'
)

print(
    'New judgments/principle:          15'
)

print(
    'Eventual judgments/principle:     25'
)

print()

print(
    f'New Round-3 judgments:            '
    f'{len(round3_assignments):,}'
)

print(
    f'Exact planned paid calls:         '
    f'{round3_calls:,}'
)

print(
    f'Round-1/2 pair reuses:            '
    f'{reuse_count}'
)

print()

print(
    'Prompt-size diagnostic:'
)

print(
    f'  mean rough tokens/call: '
    f'{audit.very_rough_prompt_tokens.mean():,.0f}'
)

print(
    f'  max rough tokens/call:  '
    f'{audit.very_rough_prompt_tokens.max():,.0f}'
)

print()

print(
    'PAID LLM CALLS MADE: 0'
)

print(
    'RUN_PAID_LLM remains:',
    RUN_PAID_LLM
)

print(
    "============================================"
)


In [ ]:

# step 4 round 3 — final paid testing
# + final 25-judgment ranking

# 100 principles
# 15 new judgments each
# = 1,500 new judgments

# planned calls:
# 350 + 350 + 350 + 350 + 100
# = 5 calls

# safety:
# - maximum 5 paid calls
# - retries=1
# - checkpoint every successful batch immediately
# - no automatic repair calls
# - paid calls blocked afterward

# if complete:
# combines round 1 + round 2 + round 3
# = exactly 25 judgments per principle
# and saves final ranking.

import getpass
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from openai import AzureOpenAI
from tqdm.auto import tqdm

# files

ROUND3_PLAN_FILE = (
    OUTPUT_DIR /
    'step4_round3_call_plan_350.csv'
)

PAIR_ORDER_FILE = (
    OUTPUT_DIR /
    'step4_pair_order.csv'
)

ROUND1_CKPT = (
    OUTPUT_DIR /
    'step4_round1_shared_checkpoint.jsonl'
)

ROUND2_CKPT = (
    OUTPUT_DIR /
    'step4_round2_shared_checkpoint.jsonl'
)

ROUND3_CKPT = (
    OUTPUT_DIR /
    'step4_round3_shared_checkpoint.jsonl'
)

TOP100_FILE = (
    OUTPUT_DIR /
    'step4_round2_survivors_100.csv'
)

# safety

RUN_PAID_LLM = False

MAX_NEW_CALLS_THIS_CELL = 5

# load round-3 plan

plan = pd.read_csv(
    ROUND3_PLAN_FILE,
    keep_default_na=False
)

plan['pair_id'] = (
    plan['pair_id']
    .astype(str)
)

plan['representative_id'] = (
    plan['representative_id']
    .astype(str)
)

if len(plan) != 1500:
    raise RuntimeError(
        f'Expected 1,500 Round-3 tasks; '
        f'found {len(plan):,}.'
    )

if plan['planned_call_id'].nunique() != 5:
    raise RuntimeError(
        f'Expected exactly 5 Round-3 calls; '
        f'found {plan["planned_call_id"].nunique():,}.'
    )

# expected batch sizes:
# 350, 350, 350, 350, 100

batch_sizes = (
    plan
    .groupby('planned_call_id')
    .size()
    .to_dict()
)

expected_sizes = {
    1: 350,
    2: 350,
    3: 350,
    4: 350,
    5: 100
}

for batch_id, expected in expected_sizes.items():

    actual = int(
        batch_sizes.get(
            batch_id,
            0
        )
    )

    if actual != expected:

        raise RuntimeError(
            f'Round-3 batch {batch_id} should contain '
            f'{expected} tasks; found {actual}.'
        )

# load a/b order

pair_order = pd.read_csv(
    PAIR_ORDER_FILE,
    keep_default_na=False
)

pair_order['pair_id'] = (
    pair_order['pair_id']
    .astype(str)
)

pair_lookup = (
    pair_order
    .set_index('pair_id')
)

# helpers

def collect_checkpoint_votes(path):

    rows = []

    for record in read_jsonl(path):

        for r in record.get(
            'votes',
            []
        ):

            rid = str(
                r.get(
                    'representative_id',
                    ''
                )
            )

            pid = str(
                r.get(
                    'pair_id',
                    ''
                )
            )

            vote = r.get(
                'vote'
            )

            if (
                rid
                and pid
                and vote in {
                    'A',
                    'B',
                    'None'
                }
            ):

                rows.append({

                    'representative_id':
                        rid,

                    'pair_id':
                        pid,

                    'task_id':
                        rid + '|' + pid,

                    'vote':
                        vote

                })

    if not rows:

        return pd.DataFrame(
            columns=[
                'representative_id',
                'pair_id',
                'task_id',
                'vote'
            ]
        )

    out = pd.DataFrame(rows)

    return (
        out
        .drop_duplicates(
            'task_id',
            keep='first'
        )
        .reset_index(drop=True)
    )

def completed_round3_tasks():

    votes = collect_checkpoint_votes(
        ROUND3_CKPT
    )

    return set(
        votes['task_id']
    )

# preview

completed_before = (
    completed_round3_tasks()
)

print(
    "========== FINAL ROUND-3 PAID RUN PREVIEW =========="
)

print(
    f'Round-3 judgments already complete: '
    f'{len(completed_before):,} / 1,500'
)

print(
    'Planned paid calls: 5'
)

print(
    'Batch sizes: 350, 350, 350, 350, 100'
)

print(
    'Maximum new paid calls: 5'
)

print(
    'Automatic retries: 0'
)

print(
    "===================================================="
)

# prompt

system_prompt = """
Apply each explicit principle independently to its assigned
pair of Reddit posts.

For every principle:
- Vote "A" if Post A better satisfies that principle.
- Vote "B" if Post B better satisfies that principle.
- Vote "None" if the principle is not applicable, does not
  meaningfully distinguish the posts, or neither post is
  clearly favored by that principle.

Judge ONLY by the supplied principle.
Do not judge overall post quality.
Do not use popularity, scores, votes, or outside information.
Treat every principle independently.

Return valid JSON only.
""".strip()

# connect to azure

subscription_key = getpass.getpass(
    'Azure OpenAI API key: '
)

client = AzureOpenAI(
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=subscription_key
)

RUN_PAID_LLM = True

# run five calls

new_calls_sent = 0

stages_this_run = []

run_expected = 0
run_received = 0
run_missing = 0

stopped_due_to_error = False
error_message = None

try:

    for batch_id in tqdm(
        range(1, 6),
        desc='FINAL Round-3 testing'
    ):

        if (
            new_calls_sent
            >= MAX_NEW_CALLS_THIS_CELL
        ):
            break

        stage_name = (
            f'step4_round3_350_batch_'
            f'{batch_id:03d}'
        )

        # duplicate-spend protection

        checkpointed_stages = {

            str(
                x.get(
                    'stage',
                    ''
                )
            )

            for x in read_jsonl(
                ROUND3_CKPT
            )

        }

        logged_stages = {

            str(
                x.get(
                    'stage',
                    ''
                )
            )

            for x in read_jsonl(
                CALL_LOG
            )

        }

        if stage_name in checkpointed_stages:

            print(
                f'\nSkipping batch {batch_id}: '
                'already checkpointed.'
            )

            continue

        if stage_name in logged_stages:

            raise RuntimeError(
                f'{stage_name} appears in CALL_LOG '
                'but is not checkpointed. '
                'Stopping to avoid duplicate spending.'
            )

        # load batch

        batch_df = (
            plan[
                plan[
                    'planned_call_id'
                ] == batch_id
            ]
            .copy()
        )

        expected_batch_size = (
            expected_sizes[
                batch_id
            ]
        )

        if (
            len(batch_df)
            != expected_batch_size
        ):

            raise RuntimeError(
                f'Batch {batch_id}: expected '
                f'{expected_batch_size} tasks, '
                f'found {len(batch_df)}.'
            )

        # avoid duplicate tasks

        completed_now = (
            completed_round3_tasks()
        )

        batch_df['task_id'] = (
            batch_df[
                'representative_id'
            ]
            +
            '|'
            +
            batch_df[
                'pair_id'
            ]
        )

        already_done = (
            batch_df[
                'task_id'
            ].isin(
                completed_now
            )
        )

        if already_done.any():

            raise RuntimeError(
                f'Batch {batch_id} contains '
                f'{already_done.sum():,} already-completed '
                'tasks. Stopping.'
            )

        # group by shared reddit pair

        pair_payload = []

        for pair_id, g in (
            batch_df.groupby(
                'pair_id',
                sort=False
            )
        ):

            pid = str(
                pair_id
            )

            q = pair_lookup.loc[
                pid
            ]

            principles_for_pair = [

                {
                    'id':
                        str(
                            r[
                                'representative_id'
                            ]
                        ),

                    'principle':
                        str(
                            r[
                                'principle'
                            ]
                        )
                }

                for _, r in g.iterrows()

            ]

            pair_payload.append({

                'pair_id':
                    pid,

                'post_a':
                    str(
                        q[
                            'post_a'
                        ]
                    ),

                'post_b':
                    str(
                        q[
                            'post_b'
                        ]
                    ),

                'principles':
                    principles_for_pair

            })

        user_prompt = f"""
Evaluate EVERY supplied principle.

You must return:
- every supplied pair_id exactly once
- every supplied principle ID exactly once

Do not omit any principle.

Return ONLY this JSON structure:

{{
  "results": [
    {{
      "pair_id": "PAIR_ID",
      "votes": {{
        "C0000": "A",
        "C0001": "B",
        "C0002": "None"
      }}
    }}
  ]
}}

Allowed vote strings are ONLY:
"A", "B", or "None".

DATA:

{json.dumps(pair_payload, ensure_ascii=False)}
""".strip()

        # paid call

        try:

            result = chat_json(

                system_prompt,

                user_prompt,

                stage_name,

                max_tokens=10000,

                retries=1

            )

            new_calls_sent += 1

            stages_this_run.append(
                stage_name
            )

        except Exception as exc:

            stopped_due_to_error = True

            error_message = (
                f'{type(exc).__name__}: {exc}'
            )

            print()
            print(
                f'Stopping safely at '
                f'Round-3 batch {batch_id}.'
            )

            print(
                error_message
            )

            break

        # parse returned votes

        expected_by_pair = defaultdict(
            set
        )

        for _, r in batch_df.iterrows():

            expected_by_pair[
                str(
                    r['pair_id']
                )
            ].add(
                str(
                    r[
                        'representative_id'
                    ]
                )
            )

        returned_votes = defaultdict(
            dict
        )

        for item in result.get(
            'results',
            []
        ):

            pid = str(
                item.get(
                    'pair_id',
                    ''
                )
            )

            if pid not in expected_by_pair:
                continue

            votes = item.get(
                'votes',
                {}
            )

            if not isinstance(
                votes,
                dict
            ):
                continue

            for rid, raw_vote in votes.items():

                rid = str(
                    rid
                )

                if rid not in expected_by_pair[
                    pid
                ]:
                    continue

                if not isinstance(
                    raw_vote,
                    str
                ):
                    continue

                v = (
                    raw_vote
                    .strip()
                    .upper()
                )

                if v == 'A':

                    returned_votes[
                        pid
                    ][rid] = 'A'

                elif v == 'B':

                    returned_votes[
                        pid
                    ][rid] = 'B'

                elif v == 'NONE':

                    returned_votes[
                        pid
                    ][rid] = 'None'

        # build valid results

        valid_rows = []
        missing_tasks = []

        for _, r in batch_df.iterrows():

            pid = str(
                r['pair_id']
            )

            rid = str(
                r[
                    'representative_id'
                ]
            )

            vote = (
                returned_votes
                .get(
                    pid,
                    {}
                )
                .get(
                    rid
                )
            )

            if vote is None:

                missing_tasks.append(
                    rid + '|' + pid
                )

                continue

            correct_vote = str(
                pair_lookup.loc[
                    pid,
                    'correct_vote'
                ]
            )

            if vote == 'None':

                outcome = (
                    'not_applicable'
                )

            elif vote == correct_vote:

                outcome = (
                    'correct'
                )

            else:

                outcome = (
                    'incorrect'
                )

            valid_rows.append({

                'representative_id':
                    rid,

                'pair_id':
                    pid,

                'vote':
                    vote,

                'correct_vote':
                    correct_vote,

                'outcome':
                    outcome

            })

        # checkpoint immediately

        append_jsonl(

            ROUND3_CKPT,

            [{

                'stage':
                    stage_name,

                'batch_scheme':
                    'round3_350',

                'planned_call_id':
                    int(
                        batch_id
                    ),

                'expected_judgments':
                    int(
                        len(batch_df)
                    ),

                'received_judgments':
                    int(
                        len(valid_rows)
                    ),

                'missing_judgments':
                    int(
                        len(missing_tasks)
                    ),

                'votes':
                    valid_rows,

                'missing_task_ids':
                    missing_tasks

            }]

        )

        run_expected += (
            len(batch_df)
        )

        run_received += (
            len(valid_rows)
        )

        run_missing += (
            len(missing_tasks)
        )

        print(
            f'\nBatch {batch_id}: '
            f'{len(valid_rows)}/'
            f'{len(batch_df)} valid; '
            f'{len(missing_tasks)} missing.'
        )

finally:

    # paid calls off

    RUN_PAID_LLM = False

    client = None

# token usage

call_log = read_jsonl(
    CALL_LOG
)

run_logs = [

    x

    for x in call_log

    if str(
        x.get(
            'stage',
            ''
        )
    ) in set(
        stages_this_run
    )

]

prompt_tokens = sum(
    x.get('prompt_tokens') or 0
    for x in run_logs
)

completion_tokens = sum(
    x.get('completion_tokens') or 0
    for x in run_logs
)

total_tokens = sum(
    x.get('total_tokens') or 0
    for x in run_logs
)

# round-3 completeness

round3_votes = (
    collect_checkpoint_votes(
        ROUND3_CKPT
    )
)

round3_complete = len(
    round3_votes
)

round3_remaining = (
    1500
    -
    round3_complete
)

coverage = (

    run_received
    /
    run_expected

    if run_expected

    else 0
)

print()
print(
    "========== FINAL ROUND-3 PAID RESULT =========="
)

print(
    f'Paid calls completed:       '
    f'{new_calls_sent:,} / 5'
)

print(
    f'Expected judgments:         '
    f'{run_expected:,}'
)

print(
    f'Valid judgments returned:   '
    f'{run_received:,}'
)

print(
    f'Missing judgments:          '
    f'{run_missing:,}'
)

print(
    f'Coverage:                   '
    f'{coverage:.2%}'
)

print()

print(
    'TOKEN USAGE:'
)

print(
    f'  Prompt tokens:     '
    f'{prompt_tokens:,}'
)

print(
    f'  Completion tokens: '
    f'{completion_tokens:,}'
)

print(
    f'  Total tokens:      '
    f'{total_tokens:,}'
)

print()

print(
    'ROUND-3 STATUS:'
)

print(
    f'  Complete judgments: '
    f'{round3_complete:,} / 1,500'
)

print(
    f'  Still missing:      '
    f'{round3_remaining:,}'
)

print()

print(
    'PAID LLM CALLS ARE BLOCKED AGAIN:',
    not RUN_PAID_LLM
)

print(
    "================================================"
)

# if everything is complete:
# build final 25-judgment ranking locally

if round3_remaining == 0:

    top100 = pd.read_csv(
        TOP100_FILE,
        keep_default_na=False
    )

    top100[
        'representative_id'
    ] = top100[
        'representative_id'
    ].astype(str)

    top100_ids = set(
        top100[
            'representative_id'
        ]
    )

    # round 1 votes for top 100

    r1 = collect_checkpoint_votes(
        ROUND1_CKPT
    )

    r1 = (
        r1[
            r1[
                'representative_id'
            ].isin(
                top100_ids
            )
        ]
        .copy()
    )

    # round 2 votes for top 100

    r2 = collect_checkpoint_votes(
        ROUND2_CKPT
    )

    r2 = (
        r2[
            r2[
                'representative_id'
            ].isin(
                top100_ids
            )
        ]
        .copy()
    )

    # combine 3 + 7 + 15 = 25

    all25 = pd.concat(
        [
            r1,
            r2,
            round3_votes
        ],
        ignore_index=True
    )

    if all25[
        'task_id'
    ].duplicated().any():

        raise RuntimeError(
            'Duplicate tasks found in '
            'combined 25-judgment dataset.'
        )

    counts25 = (
        all25
        .groupby(
            'representative_id'
        )
        .size()
    )

    if (
        len(counts25) != 100
        or counts25.min() != 25
        or counts25.max() != 25
    ):

        raise RuntimeError(
            'Not every final principle has '
            'exactly 25 judgments.'
        )

    # score all 2,500 judgments

    final_votes = (

        all25

        .merge(

            pair_order[
                [
                    'pair_id',
                    'correct_vote'
                ]
            ],

            on='pair_id',
            how='left'

        )

    )

    final_votes[
        'applicable'
    ] = final_votes[
        'vote'
    ].isin(
        [
            'A',
            'B'
        ]
    )

    final_votes[
        'correct'
    ] = (

        final_votes[
            'applicable'
        ]

        &

        final_votes[
            'vote'
        ].eq(
            final_votes[
                'correct_vote'
            ]
        )

    )

    final_votes[
        'incorrect'
    ] = (

        final_votes[
            'applicable'
        ]

        &

        ~final_votes[
            'vote'
        ].eq(
            final_votes[
                'correct_vote'
            ]
        )

    )

    # final metrics

    final_stats = (

        final_votes

        .groupby(
            'representative_id'
        )

        .agg(

            tested=(
                'task_id',
                'size'
            ),

            applicable=(
                'applicable',
                'sum'
            ),

            correct=(
                'correct',
                'sum'
            ),

            incorrect=(
                'incorrect',
                'sum'
            )

        )

        .reset_index()

    )

    final_stats[
        'not_applicable'
    ] = (

        final_stats[
            'tested'
        ]

        -
        final_stats[
            'applicable'
        ]

    )

    final_stats[
        'applicability_rate'
    ] = (

        final_stats[
            'applicable'
        ]

        /
        final_stats[
            'tested'
        ]

    )

    final_stats[
        'accuracy_when_applicable'
    ] = (

        final_stats[
            'correct'
        ]

        /
        final_stats[
            'applicable'
        ].replace(
            0,
            np.nan
        )

    )

    final_stats[
        'net_score'
    ] = (

        final_stats[
            'correct'
        ]

        -
        final_stats[
            'incorrect'
        ]

    )

    # remove old metric columns before merging

    old_metric_cols = [
        'tested',
        'applicable',
        'correct',
        'incorrect',
        'not_applicable',
        'applicability_rate',
        'accuracy_when_applicable',
        'net_score'
    ]

    top100_base = top100.drop(

        columns=[
            c
            for c in old_metric_cols
            if c in top100.columns
        ],

        errors='ignore'

    )

    final_ranked = (

        top100_base

        .merge(
            final_stats,
            on='representative_id',
            how='left',
            validate='one_to_one'
        )

    )

    # final ranking

    sort_columns = [
        'net_score',
        'accuracy_when_applicable',
        'applicable'
    ]

    for c in [
        'embedding_accuracy',
        'embedding_mean_margin'
    ]:

        if c in final_ranked.columns:

            sort_columns.append(
                c
            )

    final_ranked = (

        final_ranked

        .sort_values(
            sort_columns,
            ascending=[
                False
            ] * len(
                sort_columns
            )
        )

        .reset_index(drop=True)

    )

    final_ranked.insert(
        0,
        'final_rank',
        np.arange(
            1,
            len(final_ranked) + 1
        )
    )

    # save final testing results

    final_votes.to_csv(

        OUTPUT_DIR /
        'step4_final_25pair_votes.csv',

        index=False

    )

    final_ranked.to_csv(

        OUTPUT_DIR /
        'step4_final_ranked_100_principles.csv',

        index=False

    )

    print()
    print(
        "========== FINAL 25-JUDGMENT RESULTS =========="
    )

    print(
        f'Final principles evaluated: '
        f'{len(final_ranked):,}'
    )

    print(
        'Judgments per principle:    25'
    )

    print()

    print(
        'Top 15 final principles:'
    )

    print(

        final_ranked[
            [
                'final_rank',
                'representative_id',
                'cleaned_principle',
                'applicable',
                'accuracy_when_applicable',
                'net_score'
            ]
        ]

        .head(15)

        .to_string(
            index=False
        )

    )

    print()

    print(
        'Saved:'
    )

    print(
        '  step4_final_25pair_votes.csv'
    )

    print(
        '  step4_final_ranked_100_principles.csv'
    )

    print()

    print(
        "FINAL PAIRWISE TESTING IS COMPLETE."
    )

    print(
        "==============================================="
    )

else:

    print()
    print(
        'Round 3 has a small number of omitted '
        'judgments.'
    )

    print(
        'Do NOT rerun the five batches.'
    )

    print(
        'We will repair only those omissions '
        'in one tiny call.'
    )


In [ ]:

# notebook 1 — finalize top 20 + export audit

# zero paid llm calls

# uses the final 25-judgment ranking.
# original final filter:
# applicability_rate >= .10
# net_score > 0

# saves:
# final_filtered_principles.csv
# principle_metrics.csv
# llm_call_summary.csv
# askscience_icai_principle_audit.docx

from pathlib import Path

import pandas as pd
from docx import Document

RUN_PAID_LLM = False

MIN_APPLICABILITY = 0.10
FINAL_N_PRINCIPLES = 20

# load final 100 if needed

FINAL100_FILE = (
    OUTPUT_DIR /
    'step4_final_ranked_100_principles.csv'
)

if 'final_ranked' not in globals():

    final_ranked = pd.read_csv(
        FINAL100_FILE,
        keep_default_na=False
    )

print(
    f'Final 25-judgment principles available: '
    f'{len(final_ranked):,}'
)

if len(final_ranked) != 100:

    raise RuntimeError(
        f'Expected 100 final-stage principles; '
        f'found {len(final_ranked):,}.'
    )

# original final filter

metrics = final_ranked.copy()

metrics['passes_filter'] = (

    (
        metrics[
            'applicability_rate'
        ]
        >= MIN_APPLICABILITY
    )

    &

    (
        metrics[
            'net_score'
        ]
        > 0
    )

)

metrics = (

    metrics

    .sort_values(

        [
            'passes_filter',
            'net_score',
            'accuracy_when_applicable',
            'applicable'
        ],

        ascending=[
            False,
            False,
            False,
            False
        ]

    )

    .reset_index(drop=True)

)

eligible_final = (

    metrics[
        metrics[
            'passes_filter'
        ]
    ]

    .copy()

)

if len(eligible_final) < FINAL_N_PRINCIPLES:

    raise RuntimeError(
        f'Only {len(eligible_final)} principles pass '
        f'the final filter; need {FINAL_N_PRINCIPLES}.'
    )

final = (

    eligible_final

    .head(
        FINAL_N_PRINCIPLES
    )

    .copy()

)

final.insert(

    0,

    'audit_rank',

    range(
        1,
        len(final) + 1
    )

)

# save csvs

metrics.to_csv(

    OUTPUT_DIR /
    'principle_metrics.csv',

    index=False

)

final.to_csv(

    OUTPUT_DIR /
    'final_filtered_principles.csv',

    index=False

)

# llm call summary

log = pd.DataFrame(
    read_jsonl(
        CALL_LOG
    )
)

if not log.empty:

    call_summary = (

        log

        .groupby(
            'stage',
            as_index=False
        )

        .agg(

            calls=(
                'stage',
                'size'
            ),

            prompt_tokens=(
                'prompt_tokens',
                'sum'
            ),

            completion_tokens=(
                'completion_tokens',
                'sum'
            ),

            total_tokens=(
                'total_tokens',
                'sum'
            )

        )

    )

else:

    call_summary = pd.DataFrame(
        columns=[
            'stage',
            'calls',
            'prompt_tokens',
            'completion_tokens',
            'total_tokens'
        ]
    )

call_summary.to_csv(

    OUTPUT_DIR /
    'llm_call_summary.csv',

    index=False

)

# create audit document

# important:
# the first table contains the top 20.
# notebook 2 reads this first table.

REPORT = (
    OUTPUT_DIR /
    'AskScience_ICAI_principle_audit.docx'
)

doc = Document()

doc.add_heading(
    'ICAI Principle Audit — r/AskScience',
    0
)

doc.add_paragraph(

    'Scaled ICAI-style preference inference run using '
    '10,000 initial r/AskScience preference pairs and '
    '5,000 candidate principles. Candidates were screened '
    'through staged LLM testing at 3, 10, and 25 judgments. '
    'The final 100 principles each received exactly '
    '25 pairwise judgments.'

)

doc.add_paragraph(

    'Final filtering requires applicability of at least '
    '10% and positive net score. The 20 highest-ranked '
    'passing principles are carried into held-out review '
    'and validation.'

)

# table 1 — top 20

doc.add_heading(
    '1. Final filtered principles',
    level=1
)

columns = [
    'audit_rank',
    'representative_id',
    'cleaned_principle',
    'applicability_rate',
    'accuracy_when_applicable',
    'net_score'
]

headers = [
    'Rank',
    'ID',
    'Principle',
    'Applicability',
    'Accuracy',
    'Net'
]

if 'embedding_accuracy' in final.columns:

    columns.append(
        'embedding_accuracy'
    )

    headers.append(
        'Embedding accuracy'
    )

table = doc.add_table(
    rows=1,
    cols=len(columns)
)

table.style = (
    'Light Shading Accent 1'
)

for i, header in enumerate(headers):

    table.rows[
        0
    ].cells[
        i
    ].text = str(
        header
    )

for _, row in final.iterrows():

    cells = (
        table
        .add_row()
        .cells
    )

    for i, col in enumerate(columns):

        value = row[col]

        if pd.isna(value):

            text = ''

        elif col in {
            'applicability_rate',
            'accuracy_when_applicable',
            'embedding_accuracy'
        }:

            text = (
                f'{float(value):.1%}'
            )

        elif col == 'net_score':

            text = str(
                int(value)
            )

        else:

            text = str(
                value
            )

        cells[i].text = text

# table 2 — final 100

doc.add_heading(
    '2. Final-stage 100 principles',
    level=1
)

doc.add_paragraph(

    'These principles all reached the final testing stage '
    'and were evaluated on exactly 25 preference pairs.'

)

cols100 = [
    'final_rank',
    'representative_id',
    'cleaned_principle',
    'applicability_rate',
    'accuracy_when_applicable',
    'net_score'
]

headers100 = [
    'Rank',
    'ID',
    'Principle',
    'Applicability',
    'Accuracy',
    'Net'
]

table100 = doc.add_table(
    rows=1,
    cols=len(cols100)
)

table100.style = (
    'Light Shading Accent 1'
)

for i, header in enumerate(
    headers100
):

    table100.rows[
        0
    ].cells[
        i
    ].text = header

for _, row in metrics.iterrows():

    cells = (
        table100
        .add_row()
        .cells
    )

    values = [

        str(
            row[
                'final_rank'
            ]
        ),

        str(
            row[
                'representative_id'
            ]
        ),

        str(
            row[
                'cleaned_principle'
            ]
        ),

        f"{float(row['applicability_rate']):.1%}",

        f"{float(row['accuracy_when_applicable']):.1%}",

        str(
            int(
                row[
                    'net_score'
                ]
            )
        )

    ]

    for i, value in enumerate(
        values
    ):

        cells[i].text = value

# call accounting

doc.add_heading(
    '3. LLM call accounting',
    level=1
)

if not call_summary.empty:

    total_calls = int(
        call_summary[
            'calls'
        ].sum()
    )

    total_prompt = int(
        call_summary[
            'prompt_tokens'
        ].fillna(0).sum()
    )

    total_completion = int(
        call_summary[
            'completion_tokens'
        ].fillna(0).sum()
    )

    total_tokens = int(
        call_summary[
            'total_tokens'
        ].fillna(0).sum()
    )

    doc.add_paragraph(
        f'Total logged Azure calls: {total_calls:,}. '
        f'Prompt tokens: {total_prompt:,}. '
        f'Completion tokens: {total_completion:,}. '
        f'Total tokens: {total_tokens:,}.'
    )

    ctable = doc.add_table(
        rows=1,
        cols=5
    )

    ctable.style = (
        'Light Shading Accent 1'
    )

    call_headers = [
        'Stage',
        'Calls',
        'Prompt tokens',
        'Completion tokens',
        'Total tokens'
    ]

    for i, header in enumerate(
        call_headers
    ):

        ctable.rows[
            0
        ].cells[
            i
        ].text = header

    for _, row in (
        call_summary
        .iterrows()
    ):

        cells = (
            ctable
            .add_row()
            .cells
        )

        values = [

            str(
                row[
                    'stage'
                ]
            ),

            str(
                int(
                    row[
                        'calls'
                    ]
                )
            ),

            str(
                int(
                    row[
                        'prompt_tokens'
                    ]
                    or 0
                )
            ),

            str(
                int(
                    row[
                        'completion_tokens'
                    ]
                    or 0
                )
            ),

            str(
                int(
                    row[
                        'total_tokens'
                    ]
                    or 0
                )
            )

        ]

        for i, value in enumerate(
            values
        ):

            cells[i].text = value

doc.save(
    REPORT
)

# final output

print()
print(
    "========== NOTEBOOK 1 FINAL RESULT =========="
)

print(
    f'Final-stage principles:       '
    f'{len(metrics):,}'
)

print(
    f'Passing final filter:         '
    f'{len(eligible_final):,}'
)

print(
    f'Principles exported:          '
    f'{len(final):,}'
)

print()

print(
    'TOP 20:'
)

print(

    final[
        [
            'audit_rank',
            'representative_id',
            'cleaned_principle',
            'applicability_rate',
            'accuracy_when_applicable',
            'net_score'
        ]
    ]

    .to_string(
        index=False
    )

)

print()

print(
    'Saved:'
)

print(
    '  final_filtered_principles.csv'
)

print(
    '  principle_metrics.csv'
)

print(
    '  llm_call_summary.csv'
)

print(
    '  AskScience_ICAI_principle_audit.docx'
)

print()

print(
    'PAID LLM CALLS MADE: 0'
)

print(
    'RUN_PAID_LLM remains:',
    RUN_PAID_LLM
)

print()

print(
    'NOTEBOOK 1 IS COMPLETE.'
)

print(
    "============================================="
)


## step 4 — test all 5,000 candidate principles in stages

Every cluster representative starts with 3 LLM judgments. The strongest survivors move to 10 total judgments, and the final group moves to 25. This keeps the experiment much smaller than testing every principle against every Reddit pair.


## step 5 — calculate metrics and filter the final principles


## export the AskScience principle audit and LLM call summary
